In [ ]:
# Elektrotechnik Spickzettel – QV 2026
# Alle Imports auf einen Blick
import numpy as np
import re, math
import matplotlib.pyplot as plt
from scipy.signal import lti, step
import matplotlib.ticker as ticker
import ipywidgets as widgets
from IPython.display import display, Markdown, HTML
import sys
import os

# ============================================================
# SETUP – Bitte zuerst ausführen!
# ============================================================

# 1. Pfad zum Verzeichnis 'spick' ermitteln (eine Ebene über 'notebooks')
# __file__ funktioniert in Notebooks nicht immer, daher nutzen wir os.getcwd()
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))

# 2. Diesen Pfad zu sys.path hinzufügen, falls er noch nicht da ist
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# 3. Import-Versuch
try:
    from calc import *
    from utils import p, fmt, make_input, make_button, make_dropdown, make_toggle, check_inputs
    from utils import EPS0, MU0
except ImportError as e:
    print("Fehler: ", e)
    print("utils.py nicht gefunden – verwende eingebaute Definitionen")
    SI_PREFIXES = {'p':1e-12,'n':1e-9,'u':1e-6,'µ':1e-6,'m':1e-3,'':1.0,'k':1e3,'M':1e6,'G':1e9}

    def p(val_str):
        if val_str is None: return None
        val_str = val_str.strip()
        if val_str == '': return None
        match = re.fullmatch(r'([-+]?\d*\.?\d+)\s*([pnuµmkMG]?)', val_str)
        if not match: raise ValueError(f"Ungültiger Wert: '{val_str}'")
        value, prefix = match.groups()
        return float(value) * SI_PREFIXES[prefix]

    def fmt(val, unit=''):
        if val == 0: return f"0 {unit}".strip()
        abs_val = abs(val)
        for name, factor in [('G',1e9),('M',1e6),('k',1e3),('',1.0),('m',1e-3),('µ',1e-6),('n',1e-9),('p',1e-12)]:
            if abs_val >= factor:
                return f"{val/factor:.4g} {name}{unit}".strip()
        return f"{val:.4e} {unit}".strip()

    def make_input(description, placeholder='leer lassen = berechnen'):
        return widgets.Text(description=description, placeholder=placeholder,
                            layout=widgets.Layout(width='320px'), style={'description_width':'140px'})

    def make_button(label='Berechnen'):
        return widgets.Button(description=label, button_style='primary',
                              layout=widgets.Layout(width='180px', margin='8px 0'))

    def make_dropdown(description, options):
        return widgets.Dropdown(description=description, options=options,
                                style={'description_width':'140px'}, layout=widgets.Layout(width='340px'))

    def make_toggle(description, options, button_width='140px'):
        return widgets.ToggleButtons(description=description, options=options,
                                     style={'description_width':'140px','button_width':button_width})

    def check_inputs(vals):
        miss = [k for k, v in vals.items() if v is None]
        return len(miss), miss

    EPS0 = 8.854187817e-12
    MU0  = 4 * math.pi * 1e-7

THEORY_STYLE = widgets.Layout(width='100%', padding='0 20px 0 0')

BASE_BOX = {
    'min_width': '300px',
}

# Layout für die linke Theorie-Box
LEFT_STYLE = {
    **BASE_BOX, # Kopiert die Basis-Einstellungen hinein
    'width': '70%',
    'padding': '0 20px 0 0',
    'border_right': '2px solid #dde'
}

# Layout für die rechte Rechner-Box
RIGHT_STYLE = {
    **BASE_BOX,
    'width': '25%',
    'padding': '0 0 0 20px'
}

%matplotlib inline

<div style="text-align: center; margin-top: 100px;">

<h1 style="font-size: 48px; color: #003366;">Spickzettel QV 2026</h1>
<h2 style="font-size: 32px; color: #0055a5;">Elektroniker EFZ – Schweiz</h2>

<p style="font-size: 20px; margin-top: 50px;">
<strong>Erstellt am:</strong> 03.01.2026<br>
<strong>Geändert am:</strong> 04.01.2026<br> 
<strong>Geändert am:</strong> 03.03.2026<br> 
<strong>Geändert am:</strong> 07.03.2026<br> 
<strong>Geändert am:</strong> 08.03.2026<br> 
<strong>Version:</strong> v0.1.8<br>
<strong>Autor:</strong> Alvar Bolliger
</p>

<p style="font-size: 16px; margin-top: 30px; color: #555555;">
Hinweis: Dies ist ein persönlicher Spickzettel für die QV 2026. <br>
Interaktive Grafiken und Berechnungen sind im Notebook eingebettet.
</p>

</div>

<div style="border-left:4px solid #003366; padding:16px 24px; border-radius:6px; margin:16px 0; font-family: sans-serif;">
<h2 style="color:#003366; margin-top:0;">Inhaltsverzeichnis</h2>
<ol style="line-height:1.6;">

  <li style="margin-bottom:6px;"><a href="#sec-1-1"><strong>Widerstand (R)</strong></a>
    <ul style="margin-top:4px; color:#444;">
      <li><a href="#sec-1-0">1.0 Kirchhoffsche Gesetze</a></li>
      <li><a href="#sec-1-1">1.1 Ohmsches Gesetz</a></li>
      <li><a href="#sec-1-2">1.2 Leistung</a></li>
      <li><a href="#sec-1-3">1.3 Serie</a></li>
      <li><a href="#sec-1-4">1.4 Parallel</a></li>
      <li><a href="#sec-1-5">1.5 Temperaturkoeffizient</a></li>
      <li><a href="#sec-1-6">1.6 Spez. Widerstand</a></li>
      <li><a href="#sec-1-7">1.7 Farbcode</a></li>
      <li><a href="#sec-1-8">1.8 Spannungsteiler</a></li>
      <li><a href="#sec-1-9">1.9 Leitwert</a></li>
      <li><a href="#sec-1-10">1.10 Stromdichte</a></li>
      <li><a href="#sec-1-11">1.11 Thevenin/Norton</a></li>
      <li><a href="#sec-1-12">1.12 Belasteter Spannungsteiler</a></li>
    </ul>
  </li>

  <li style="margin-bottom:6px;"><a href="#sec-2"><strong>Kondensator (C)</strong></a>
    <ul style="margin-top:4px; color:#444;">
      <li><a href="#sec-2-1">2.1 Grundlagen</a></li>
      <li><a href="#sec-2-2">2.2 Plattenkondensator</a></li>
      <li><a href="#sec-2-3">2.3 Reihe/Parallel</a></li>
      <li><a href="#sec-2-4">2.4 RC-Zeitkonstante</a></li>
      <li><a href="#sec-2-5">2.5 Xc</a></li>
      <li><a href="#sec-2-6">2.6 RC-Impedanz</a></li>
      <li><a href="#sec-2-7">2.7 E-Feld</a></li>
    </ul>
  </li>

  <li style="margin-bottom:6px;"><a href="#sec-3"><strong>Spule (L)</strong></a>
    <ul style="margin-top:4px; color:#444;">
      <li><a href="#sec-3-1">3.1 Selbstinduktion</a></li>
      <li><a href="#sec-3-2">3.2 Zylinderspule</a></li>
      <li><a href="#sec-3-3">3.3 Reihe/Parallel</a></li>
      <li><a href="#sec-3-4">3.4 RL-Zeitkonstante</a></li>
      <li><a href="#sec-3-5">3.5 Xl</a></li>
      <li><a href="#sec-3-6">3.6 RL-Impedanz</a></li>
      <li><a href="#sec-3-7">3.7 Mag. Feld</a></li>
    </ul>
  </li>

  <li style="margin-bottom:6px;"><a href="#sec-4"><strong>Wechselstrom – Leistung</strong></a>
    <ul style="margin-top:4px; color:#444;">
      <li><a href="#sec-4-0">4.0 Effektiv-/Spitzenwert</a></li>
      <li><a href="#sec-4-1">4.1 S / P / Q / cos φ</a></li>
      <li><a href="#sec-4-2">4.2 Blindstromkompensation</a></li>
    </ul>
  </li>

  <li style="margin-bottom:6px;"><a href="#sec-5"><strong>RLC-Schwingkreise</strong></a>
    <ul style="margin-top:4px; color:#444;">
      <li><a href="#sec-5-1">5.1 f₀ Grundlagen</a></li>
      <li><a href="#sec-5-2">5.2 Reihenschwingkreis</a></li>
      <li><a href="#sec-5-3">5.3 Parallelschwingkreis</a></li>
      <li><a href="#sec-5-4">5.4 Güte &amp; Δf</a></li>
    </ul>
  </li>

  <li style="margin-bottom:6px;"><a href="#sec-6"><strong>Dezibel (dB)</strong></a>
    <ul style="margin-top:4px; color:#444;">
      <li><a href="#sec-6-1">6.1 Grundlagen</a></li>
      <li><a href="#sec-6-2">6.2 Wichtige dB-Werte</a></li>
      <li><a href="#sec-6-3">6.3 Kettenrechnung</a></li>
    </ul>
  </li>

  <li style="margin-bottom:6px;"><a href="#sec-7"><strong>Filter</strong></a>
    <ul style="margin-top:4px; color:#444;">
      <li><a href="#sec-7-1">7.1 RC-Tiefpass</a></li>
      <li><a href="#sec-7-2">7.2 RC-Hochpass</a></li>
      <li><a href="#sec-7-3">7.3 RL-Filter</a></li>
      <li><a href="#sec-7-4">7.4 Akt. Tiefpass</a></li>
      <li><a href="#sec-7-5">7.5 Akt. Hochpass</a></li>
      <li><a href="#sec-7-6">7.6 Bandpass</a></li>
      <li><a href="#sec-7-7">7.7 Sallen-Key</a></li>
      <li><a href="#sec-7-8">7.8 Notch</a></li>
    </ul>
  </li>

  <li style="margin-bottom:6px;"><a href="#sec-8"><strong>Transformator</strong></a>
    <ul style="margin-top:4px; color:#444;">
      <li><a href="#sec-8-1">8.1 Grundlagen</a></li>
      <li><a href="#sec-8-2">8.2 Leistung &amp; η</a></li>
      <li><a href="#sec-8-3">8.3 Widerstandstransf.</a></li>
      <li><a href="#sec-8-4">8.4 Leerlauf/Kurzschluss</a></li>
      <li><a href="#sec-8-5">8.5 Spartrafo/Trenntransformator</a></li>
    </ul>
  </li>

  <li style="margin-bottom:6px;"><a href="#sec-9"><strong>Gleichrichter &amp; Siebung</strong></a>
    <ul style="margin-top:4px; color:#444;">
      <li><a href="#sec-9-1">9.1 Grundlagen</a></li>
      <li><a href="#sec-9-2">9.2 Einweggleichrichter</a></li>
      <li><a href="#sec-9-3">9.3 Brückengleichrichter</a></li>
      <li><a href="#sec-9-4">9.4 Siebung</a></li>
      <li><a href="#sec-9-5">9.5 Netzteil-Dimensionierung</a></li>
      <li><a href="#sec-9-6">9.6 Vergleich M1 / B2</a></li>
    </ul>
  </li>

  <li style="margin-bottom:6px;"><a href="#sec-10"><strong>Thermik</strong></a>
    <ul style="margin-top:4px; color:#444;">
      <li><a href="#sec-10-1">10.1 Wärmewiderstand</a></li>
      <li><a href="#sec-10-2">10.2 Kühlkörper</a></li>
      <li><a href="#sec-10-3">10.3 Widerstandsänderung</a></li>
    </ul>
  </li>

  <li style="margin-bottom:6px;"><a href="#anhang-a"><strong>Anhang: Interaktive Diagramme</strong></a>
    <ul style="margin-top:4px; color:#444;">
      <li><a href="#anhang-a">A: RC-Kurve</a></li>
      <li><a href="#anhang-b">B: Bode</a></li>
      <li><a href="#anhang-c">C: RLC</a></li>
      <li><a href="#anhang-d">D: RL-Einschaltstrom</a></li>
      <li><a href="#anhang-e">E: Gleichrichter</a></li>
      <li><a href="#anhang-f">F: Zeigerdiagramm</a></li>
      <li><a href="#anhang-g">G: NTC/PTC</a></li>
    </ul>
  </li>

</ol>
</div>

In [ ]:
display(HTML('<a name="sec-1-0"></a>'))
# ================================================================
# LAYOUT: Theorie (links) | Rechner (rechts)
# ================================================================

# ---- Theorie (links) ----
_theory_out = widgets.Output(layout=THEORY_STYLE)

with _theory_out:
    display(Markdown(
        "---\n"
        "## 1. Widerstand (R)\n"
        "---\n"
        "### 1.0 Kirchhoffsche Gesetze\n"
        "Die Kirchhoffschen Gesetze sind die Grundlage für die Analyse jeder elektrischen Schaltung.\n"
        "\n"
        "#### Knotenregel (KCL – Kirchhoff's Current Law)\n"
        "Die Summe aller Ströme in einem **Knoten** ist null.\n"
        "Zufliessende Ströme positiv, abfliessende negativ.\n"
        "$$\\sum I = 0 \\qquad I_1 + I_2 = I_3 + I_4$$\n"
        "> **Merkhilfe:** Was rein fliesst, muss auch raus fliessen – wie Wasser in einem Rohrknoten.\n"
        "\n"
        "#### Maschenregel (KVL – Kirchhoff's Voltage Law)\n"
        "Die Summe aller Spannungen entlang einer **geschlossenen Masche** ist null.\n"
        "$$\\sum U = 0$$\n"
        "**Vorzeichen-Konvention:** Spannungsquelle in Durchlaufrichtung = positiv, Spannungsabfall = negativ.\n"
        "$$U_{ges} - U_{R1} - U_{R2} - U_{R3} = 0$$\n"
        "$$\\Rightarrow U_{ges} = U_{R1} + U_{R2} + U_{R3}$$\n"
        "> **Merkhilfe:** Einmal rund um die Masche – am Ende landet man wieder beim gleichen Potential.\n"
        "\n"
        "#### Anwendungsbeispiel – Maschenstrom\n"
        "| Schritt | Vorgehen |\n"
        "|---------|----------|\n"
        "| 1 | Knoten und Maschen einzeichnen |\n"
        "| 2 | Ströme mit Pfeilrichtung festlegen |\n"
        "| 3 | KCL für jeden Knoten aufstellen |\n"
        "| 4 | KVL für jede Masche aufstellen |\n"
        "| 5 | Gleichungssystem lösen |\n"
    ))

# ---- Rechner (rechts) ----
# ============================================================
# RECHNER 1.0a: KCL – Knotenregel (bis 5 Ströme)
# ============================================================

kcl_mode = widgets.ToggleButtons(
    options=['KCL – Knoten', 'KVL – Masche'],
    description='Gesetz:', style={'description_width': '130px', 'button_width': '130px'}
)
kcl_fields = [make_input(f'I{i+1} [A] =', '+ = zufl., − = abfl.') for i in range(5)]
kvl_fields = [make_input(f'U{i+1} [V] =', '+ = Quelle, − = Abfall') for i in range(5)]
btn_k  = make_button()
out_k  = widgets.Output()
box_kcl = widgets.VBox(kcl_fields)
box_kvl = widgets.VBox(kvl_fields)
cont_k  = widgets.VBox([])

def update_k_view(_=None):
    if kcl_mode.value == 'KCL – Knoten':
        cont_k.children = [box_kcl, btn_k, out_k]
    else:
        cont_k.children = [box_kvl, btn_k, out_k]
    calc_k()

def calc_k(_=None):
    out_k.clear_output()
    with out_k:
        try:
            if kcl_mode.value == 'KCL – Knoten':
                vals = [p(f.value) for f in kcl_fields]
                vals_v = [v for v in vals if v is not None]
                unknown = [i+1 for i, v in enumerate(vals) if v is None]
                if len(vals_v) < 1:
                    print("Mindestens 1 Strom eingeben"); return
                total = sum(vals_v)
                print(f"  Summe bekannte Ströme = {fmt(total, 'A')}")
                if len(unknown) == 1:
                    missing = -total
                    print(f"  I{unknown[0]} = {fmt(missing, 'A')}  (berechneter Strom)")
                    if missing > 0:
                        print(f"  → I{unknown[0]} fliesst in den Knoten")
                    else:
                        print(f"  → I{unknown[0]} fliesst aus dem Knoten")
                elif len(unknown) == 0:
                    if abs(total) < 1e-12:
                        print("  ✓ KCL erfüllt: Summe = 0")
                    else:
                        print(f"  ✗ KCL verletzt! Summe = {fmt(total, 'A')} ≠ 0")
                else:
                    print(f"  Zu viele Unbekannte ({len(unknown)}), max. 1 leer lassen")
            else:
                vals = [p(f.value) for f in kvl_fields]
                vals_v = [v for v in vals if v is not None]
                unknown = [i+1 for i, v in enumerate(vals) if v is None]
                if len(vals_v) < 1:
                    print("Mindestens 1 Spannung eingeben"); return
                total = sum(vals_v)
                print(f"  Summe bekannte Spannungen = {fmt(total, 'V')}")
                if len(unknown) == 1:
                    missing = -total
                    print(f"  U{unknown[0]} = {fmt(missing, 'V')}  (berechnete Spannung)")
                elif len(unknown) == 0:
                    if abs(total) < 1e-9:
                        print("  ✓ KVL erfüllt: Summe = 0 V")
                    else:
                        print(f"  ✗ KVL verletzt! Summe = {fmt(total, 'V')} ≠ 0")
                else:
                    print(f"  Zu viele Unbekannte ({len(unknown)}), max. 1 leer lassen")
        except Exception as e:
            print("Fehler:", e)

kcl_mode.observe(update_k_view, names='value')
btn_k.on_click(calc_k)
for fw in kcl_fields + kvl_fields:
    fw.observe(calc_k, names='value')
update_k_view()
_calc_widget = widgets.VBox([kcl_mode, cont_k])

# ---- HBox Layout ----
_left_box  = widgets.Box([_theory_out], layout=widgets.Layout(**LEFT_STYLE))
_right_box = widgets.Box([_calc_widget], layout=widgets.Layout(**RIGHT_STYLE))

display(widgets.HBox(
    [_left_box, _right_box],
    layout=widgets.Layout(
        width='100%',
        align_items='flex-start',
        margin='8px 0 28px 0',
    )
))


In [ ]:
display(HTML('<a name="sec-1-1"></a>'))
# ================================================================
# LAYOUT: Theorie (links) | Rechner (rechts)
# ================================================================

# ---- Theorie (links) ----
_theory_out = widgets.Output(layout=THEORY_STYLE)

with _theory_out:
    display(Markdown(
        "---\n"
        "## 1. Widerstand (R)\n"
        "---\n"
        "### 1.1 Die Grundlagen\n"
        "Der elektrische **Widerstand** $R$ gibt an, wie stark ein Bauteil den Stromfluss hemmt.\n"
        "| Gr\u00f6sse | Symbol | Einheit |\n"
        "|--------|--------|---------|\n"
        "| Widerstand | $R$ | Ohm [\u03a9] |\n"
        "| Spannung | $U$ | Volt [V] |\n"
        "| Strom | $I$ | Ampere [A] |\n"
        "| Leistung | $P$ | Watt [W] |\n"
        "### Ohmsches Gesetz\n"
        "$$U = R \\cdot I$$"))

# ---- Rechner (rechts) ----
# ============================================================
# RECHNER 1.1: Ohmsches Gesetz  U = R · I
# ============================================================
U_ohm = make_input('U [V]  =')
R_ohm = make_input('R [Ω]  =')
I_ohm = make_input('I [A]  =')
btn_ohm = make_button()
out_ohm = widgets.Output()

def calc_ohm(_=None):
    # → Logik in calc.py: ohm()
    out_ohm.clear_output()
    with out_ohm:
        try:
            r = ohm(U=p(U_ohm.value), R=p(R_ohm.value), I=p(I_ohm.value))
            _u = {'U': 'V', 'R': 'Ω', 'I': 'A'}
            _skip = ('berechnet', 'ladetabelle', 'stromtabelle', 'widerspruch')
            for k, v in r.items():
                if k in _skip or v is None: continue
                if isinstance(v, bool):
                    if v: print(f'  {k}')
                    continue
                if isinstance(v, list): continue
                u = _u.get(k, '')
                print(f'  {k} = {fmt(v, u) if isinstance(v,(int,float)) else v}')
        except (ValueError, ZeroDivisionError) as e:
            print('i', e)
        except Exception as e:
            print('Fehler:', e)
btn_ohm.on_click(calc_ohm)
for field in [U_ohm, R_ohm, I_ohm]:
    field.observe(calc_ohm, names='value')

_calc_widget = (widgets.VBox([U_ohm, R_ohm, I_ohm, btn_ohm, out_ohm]))

# ---- HBox Layout ----
_left_box = widgets.Box(
    [_theory_out], 
    layout=widgets.Layout(**LEFT_STYLE)
)

_right_box = widgets.Box(
    [_calc_widget], 
    layout=widgets.Layout(**RIGHT_STYLE)
)

display(widgets.HBox(
    [_left_box, _right_box],
    layout=widgets.Layout(
        width='100%',
        align_items='flex-start',
        margin='8px 0 28px 0',
    )
))


In [ ]:
display(HTML('<a name="sec-1-2"></a>'))
# ================================================================
# LAYOUT: Theorie (links) | Rechner (rechts)
# ================================================================

# ---- Theorie (links) ----
_theory_out = widgets.Output(layout=THEORY_STYLE)

with _theory_out:
    display(Markdown(
        "---\n"
        "### 1.2 Leistung\n"
        "$$P = U \\cdot I$$\n"
        "**Energie:** $W = P \\cdot t$ \\[Joule = Watt \u00b7 Sekunden\\]"))

# ---- Rechner (rechts) ----
# ============================================================
# RECHNER 1.2: Leistung  P = U · I
# ============================================================

P_inp = make_input('P [W]  =')
U_inp = make_input('U [V]  =')
I_inp = make_input('I [A]  =')
R_inp = make_input('R [Ω]  =')
btn_P = make_button()
out_P = widgets.Output()

def calc_P(_=None):
    # → Logik in calc.py: leistung()
    out_P.clear_output()
    with out_P:
        try:
            r = leistung(P=p(P_inp.value), U=p(U_inp.value), I=p(I_inp.value), R=p(R_inp.value))
            _u = {'P': 'W', 'U': 'V', 'I': 'A', 'R': 'Ω'}
            _skip = ('berechnet', 'ladetabelle', 'stromtabelle', 'widerspruch')
            for k, v in r.items():
                if k in _skip or v is None: continue
                if isinstance(v, bool):
                    if v: print(f'  {k}')
                    continue
                if isinstance(v, list): continue
                u = _u.get(k, '')
                print(f'  {k} = {fmt(v, u) if isinstance(v,(int,float)) else v}')
        except (ValueError, ZeroDivisionError) as e:
            print('i', e)
        except Exception as e:
            print('Fehler:', e)
btn_P.on_click(calc_P)
for field in [P_inp, U_inp, I_inp, R_inp]:
    field.observe(calc_P, names='value')

_calc_widget = (widgets.VBox([P_inp, U_inp, I_inp, R_inp, btn_P, out_P]))

# ---- HBox Layout ----
_left_box = widgets.Box([_theory_out], layout=widgets.Layout(**LEFT_STYLE))
_right_box = widgets.Box([_calc_widget], layout=widgets.Layout(**RIGHT_STYLE))

display(widgets.HBox(
    [_left_box, _right_box],
    layout=widgets.Layout(
        width='100%',
        align_items='flex-start',
        margin='8px 0 28px 0',
    )
))


In [ ]:
display(HTML('<a name="sec-1-3"></a>'))
# ================================================================
# LAYOUT: Theorie (links) | Rechner (rechts)
# ================================================================

# ---- Theorie (links) ----
_theory_out = widgets.Output(layout=THEORY_STYLE)

with _theory_out:
    display(Markdown(
        "---\n"
        "### 1.3 Serienschaltung\n"
        "$$R_{ges} = R_1 + R_2 + R_3 + \\ldots$$\n"
        "**Eigenschaften:**\n"
        "- \u00dcberall **gleicher Strom**: $I_{ges} = I_1 = I_2 = I_3$\n"
        "- Spannungen addieren sich: $U_{ges} = U_1 + U_2 + U_3$\n"
        "- Gesamtwiderstand **gr\u00f6sser** als jeder Einzelwiderstand\n"
        "**Spannungsteiler:**\n"
        "$$U_x = U_{ges} \\cdot \\frac{R_x}{R_{ges}}$$"))

# ---- Rechner (rechts) ----
# ============================================================
# RECHNER 1.3: Reihenschaltung (bis 5 Widerstände)
# ============================================================

r_fields = [make_input(f'R{i+1} [Ω] =') for i in range(5)]
U_reihe = make_input('U_ges [V] =')
btn_reihe = make_button()
out_reihe = widgets.Output()

def calc_reihe(_=None):
    out_reihe.clear_output()
    with out_reihe:
        try:
            Rs = [p(f.value) for f in r_fields]
            Rs_valid = [r for r in Rs if r is not None]
            U_ges = p(U_reihe.value)
            if len(Rs_valid) < 1:
                print("Mindestens 1 Widerstand eingeben")
                return
            R_ges = sum(Rs_valid)
            print(f"R_ges = {fmt(R_ges, 'Ω')}  ({R_ges:.6e} Ω)")
            if U_ges is not None and R_ges > 0:
                I = U_ges / R_ges
                print(f"I = {fmt(I, 'A')} (bei U_ges = {fmt(U_ges, 'V')})")
                for i, r in enumerate(Rs):
                    if r is not None:
                        U_teil = I * r
                        print(f"U_R{i+1} = {fmt(U_teil, 'V')}")
        except Exception as e:
            print("Fehler:", e)

btn_reihe.on_click(calc_reihe)
for f in r_fields + [U_reihe]:
    f.observe(calc_reihe, names='value')

_calc_widget = (widgets.VBox(r_fields + [U_reihe, btn_reihe, out_reihe]))

# ---- HBox Layout ----
_left_box = widgets.Box([_theory_out], layout=widgets.Layout(**LEFT_STYLE))
_right_box = widgets.Box([_calc_widget], layout=widgets.Layout(**RIGHT_STYLE))

display(widgets.HBox(
    [_left_box, _right_box],
    layout=widgets.Layout(
        width='100%',
        align_items='flex-start',
        margin='8px 0 28px 0',
    )
))


In [ ]:
display(HTML('<a name="sec-1-4"></a>'))
# ================================================================
# LAYOUT: Theorie (links) | Rechner (rechts)
# ================================================================

# ---- Theorie (links) ----
_theory_out = widgets.Output(layout=THEORY_STYLE)

with _theory_out:
    display(Markdown(
        "---\n"
        "### 1.4. Parallelschaltung\n"
        "$$\\frac{1}{R_{ges}} = \\frac{1}{R_1} + \\frac{1}{R_2} + \\frac{1}{R_3} + \\ldots$$\n"
        "**F\u00fcr 2 Widerst\u00e4nde (Produktformel):**\n"
        "$$R_{ges} = \\frac{R_1 \\cdot R_2}{R_1 + R_2}$$\n"
        "**Eigenschaften:**\n"
        "- \u00dcberall **gleiche Spannung**: $U_{ges} = U_1 = U_2 = U_3$\n"
        "- Str\u00f6me addieren sich: $I_{ges} = I_1 + I_2 + I_3$\n"
        "- Gesamtwiderstand **kleiner** als der kleinste Einzelwiderstand\n"
        "**Stromteiler (2 Widerst\u00e4nde):**\n"
        "$$I_1 = I_{ges} \\cdot \\frac{R_2}{R_1 + R_2}$$"))

# ---- Rechner (rechts) ----
# ============================================================
# RECHNER 1.4: Parallelschaltung (bis 5 Widerstände)
# ============================================================

rp_fields = [make_input(f'R{i+1} [Ω] =') for i in range(5)]
U_para = make_input('U_ges [V] =')
btn_para = make_button()
out_para = widgets.Output()

def calc_para(_=None):
    out_para.clear_output()
    with out_para:
        try:
            Rs = [p(f.value) for f in rp_fields]
            Rs_valid = [r for r in Rs if r is not None]
            U_ges = p(U_para.value)
            if len(Rs_valid) < 1:
                print("Mindestens 1 Widerstand eingeben")
                return
            R_ges = 1 / sum(1/r for r in Rs_valid)
            print(f"R_ges = {fmt(R_ges, 'Ω')}  ({R_ges:.6e} Ω)")
            if U_ges is not None:
                I_ges = U_ges / R_ges
                print(f"   I_ges = {fmt(I_ges, 'A')}  (bei U_ges = {fmt(U_ges, 'V')})")
                for i, r in enumerate(Rs):
                    if r is not None:
                        I_teil = U_ges / r
                        print(f"   I_R{i+1}  = {fmt(I_teil, 'A')}")
        except Exception as e:
            print("Fehler:", e)

btn_para.on_click(calc_para)
for f in rp_fields + [U_para]:
    f.observe(calc_para, names='value')

_calc_widget = (widgets.VBox(rp_fields + [U_para, btn_para, out_para]))

# ---- HBox Layout ----
_left_box = widgets.Box([_theory_out], layout=widgets.Layout(**LEFT_STYLE))
_right_box = widgets.Box([_calc_widget], layout=widgets.Layout(**RIGHT_STYLE))

display(widgets.HBox(
    [_left_box, _right_box],
    layout=widgets.Layout(
        width='100%',
        align_items='flex-start',
        margin='8px 0 28px 0',
    )
))


In [ ]:
display(HTML('<a name="sec-1-5"></a>'))
# ================================================================
# LAYOUT: Theorie (links) | Rechner (rechts)
# ================================================================

# ---- Theorie (links) ----
_theory_out = widgets.Output(layout=THEORY_STYLE)

with _theory_out:
    display(Markdown(
        "---\n"
        "### 1.5. Temperaturabh\u00e4ngigkeit\n"
        "$$R_T = R_{20} \\cdot [1 + \\alpha_{20} \\cdot (T - 20\u00b0C)]$$\n"
        "- **$\\alpha_{20}$** = Temperaturkoeffizient (bei 20\u00b0C Referenz)\n"
        "- Metalle: $\\alpha > 0$ \u2192 Widerstand steigt mit Temperatur (PTC)\n"
        "- NTC-Widerst\u00e4nde: $\\alpha < 0$ \u2192 Widerstand sinkt mit Temperatur"))

# ---- Rechner (rechts) ----
# ============================================================
# RECHNER 1.5: Temperaturabhängigkeit  R_T = R_20 · [1 + α·(T-20)]
# ============================================================

R20_inp  = make_input('R₂₀ [Ω]  =')
alpha_inp = make_input('α₂₀ [1/K] =', 'z.B. 0.00393 für Cu')
T_inp    = make_input('T [°C]   =')
RT_inp   = make_input('R_T [Ω]  =')

# Materialvorauswahl
mat_select = widgets.Dropdown(
    options=[('-- Material wählen --', None),
             ('Kupfer Cu', 0.00393), ('Aluminium Al', 0.00403),
             ('Eisen Fe', 0.00650), ('Konstantan', 0.00001), ('Silber Ag', 0.00380)],
    description='Material:', style={'description_width': '130px'},
    layout=widgets.Layout(width='320px')
)

def set_alpha(change):
    if change['new'] is not None:
        alpha_inp.value = str(change['new'])
mat_select.observe(set_alpha, names='value')

btn_temp = make_button()
out_temp = widgets.Output()

def calc_temp(_=None):
    # → Logik in calc.py: temperaturkoeffizient()
    out_temp.clear_output()
    with out_temp:
        try:
            r = temperaturkoeffizient(R20=p(R20_inp.value), alpha=p(alpha_inp.value), T=p(T_inp.value), RT=p(RT_inp.value))
            _u = {'RT': 'Ω', 'R20': 'Ω', 'T': '°C', 'alpha': '1/K'}
            _skip = ('berechnet', 'ladetabelle', 'stromtabelle', 'widerspruch')
            for k, v in r.items():
                if k in _skip or v is None: continue
                if isinstance(v, bool):
                    if v: print(f'  {k}')
                    continue
                if isinstance(v, list): continue
                u = _u.get(k, '')
                print(f'  {k} = {fmt(v, u) if isinstance(v,(int,float)) else v}')
        except (ValueError, ZeroDivisionError) as e:
            print('i', e)
        except Exception as e:
            print('Fehler:', e)
btn_temp.on_click(calc_temp)
for f in [R20_inp, alpha_inp, T_inp, RT_inp]:
    f.observe(calc_temp, names='value')

_calc_widget = (widgets.VBox([mat_select, R20_inp, alpha_inp, T_inp, RT_inp, btn_temp, out_temp]))

# ---- HBox Layout ----
_left_box = widgets.Box([_theory_out], layout=widgets.Layout(**LEFT_STYLE))
_right_box = widgets.Box([_calc_widget], layout=widgets.Layout(**RIGHT_STYLE))

display(widgets.HBox(
    [_left_box, _right_box],
    layout=widgets.Layout(
        width='100%',
        align_items='flex-start',
        margin='8px 0 28px 0',
    )
))


In [ ]:
display(HTML('<a name="sec-1-6"></a>'))
# ================================================================
# LAYOUT: Theorie (links) | Rechner (rechts)
# ================================================================

# ---- Theorie (links) ----
_theory_out = widgets.Output(layout=THEORY_STYLE)

with _theory_out:
    display(Markdown(
        "---\n"
        "### 1.6. Spezifischer Widerstand\n"
        "$$R = \\rho \\cdot \\frac{l}{A}$$\n"
        "| Gr\u00f6sse | Symbol | Einheit |\n"
        "|--------|--------|---------|\n"
        "| Widerstand | $R$ | \u03a9 |\n"
        "| Spez. Widerstand | $\\rho$ (rho) | \u03a9\u00b7mm\u00b2/m |\n"
        "| L\u00e4nge | $l$ | m |\n"
        "| Querschnitt | $A$ | mm\u00b2 |"))

# ---- Rechner (rechts) ----
# ============================================================
# RECHNER 1.6: Leitungswiderstand  R = ρ · l / A
# ============================================================

rho_select = widgets.Dropdown(
    options=[('-- Material wählen --', None),
             ('Kupfer Cu', 0.0178), ('Aluminium Al', 0.0283),
             ('Eisen Fe', 0.100),   ('Konstantan', 0.500)],
    description='Material:', style={'description_width': '130px'},
    layout=widgets.Layout(width='320px')
)

R_leitung  = make_input('R [Ω]        =')
rho_inp    = make_input('ρ [Ω·mm²/m]  =', 'z.B. 0.0178 für Cu')
l_inp      = make_input('l [m]         =')
A_inp      = make_input('A [mm²]       =')

def set_rho(change):
    if change['new'] is not None:
        rho_inp.value = str(change['new'])
rho_select.observe(set_rho, names='value')

btn_leit = make_button()
out_leit = widgets.Output()

def calc_leit(_=None):
    # → Logik in calc.py: leitungswiderstand()
    out_leit.clear_output()
    with out_leit:
        try:
            r = leitungswiderstand(R=p(R_leitung.value), rho=p(rho_inp.value), l=p(l_inp.value), A=p(A_inp.value))
            _u = {'R': 'Ω', 'rho': 'Ω·mm²/m', 'l': 'm', 'A': 'mm²'}
            _skip = ('berechnet', 'ladetabelle', 'stromtabelle', 'widerspruch')
            for k, v in r.items():
                if k in _skip or v is None: continue
                if isinstance(v, bool):
                    if v: print(f'  {k}')
                    continue
                if isinstance(v, list): continue
                u = _u.get(k, '')
                print(f'  {k} = {fmt(v, u) if isinstance(v,(int,float)) else v}')
        except (ValueError, ZeroDivisionError) as e:
            print('i', e)
        except Exception as e:
            print('Fehler:', e)
btn_leit.on_click(calc_leit)
for f in [R_leitung, rho_inp, l_inp, A_inp]:
    f.observe(calc_leit, names='value')

_calc_widget = (widgets.VBox([rho_select, R_leitung, rho_inp, l_inp, A_inp, btn_leit, out_leit]))

# ---- HBox Layout ----
_left_box = widgets.Box([_theory_out], layout=widgets.Layout(**LEFT_STYLE))
_right_box = widgets.Box([_calc_widget], layout=widgets.Layout(**RIGHT_STYLE))

display(widgets.HBox(
    [_left_box, _right_box],
    layout=widgets.Layout(
        width='100%',
        align_items='flex-start',
        margin='8px 0 28px 0',
    )
))


In [ ]:
display(HTML('<a name="sec-1-7"></a>'))
# ================================================================
# LAYOUT: Theorie (links) | Rechner (rechts)
# ================================================================

# ---- Theorie (links) ----
_theory_out = widgets.Output(layout=THEORY_STYLE)

with _theory_out:
    display(Markdown(
        "---\n"
        "### 1.7 Farbcode-Lesung (4-, 5- und 6-Ring Widerst\u00e4nde)\n"
        "**4-Ring** (Standard):\n"
        "```\n"
        "Ring 1+2: Ziffern | Ring 3: Multiplikator | Ring 4: Toleranz\n"
        "```\n"
        "**5-Ring** (Pr\u00e4zisionswiderstand):\n"
        "```\n"
        "Ring 1+2+3: Ziffern | Ring 4: Multiplikator | Ring 5: Toleranz\n"
        "```\n"
        "**6-Ring** (Pr\u00e4zision + Temperaturkoeffizient):\n"
        "```\n"
        "Ring 1+2+3: Ziffern | Ring 4: Multiplikator | Ring 5: Toleranz | Ring 6: TK [ppm/K]\n"
        "```\n"
        "| Farbe | Ziffer | Multiplikator | Toleranz | TK [ppm/K] |\n"
        "|-------|--------|---------------|----------|------------|\n"
        "| Schwarz | 0 | \u00d71 | \u2013 | 250 |\n"
        "| Braun | 1 | \u00d710 | \u00b11% | 100 |\n"
        "| Rot | 2 | \u00d7100 | \u00b12% | 50 |\n"
        "| Orange | 3 | \u00d71k | \u2013 | 15 |\n"
        "| Gelb | 4 | \u00d710k | \u2013 | 25 |\n"
        "| Gr\u00fcn | 5 | \u00d7100k | \u00b10.5% | \u2013 |\n"
        "| Blau | 6 | \u00d71M | \u00b10.25% | 10 |\n"
        "| Violett | 7 | \u00d710M | \u00b10.1% | 5 |\n"
        "| Grau | 8 | \u00d7100M | \u00b10.05% | \u2013 |\n"
        "| Weiss | 9 | \u2013 | \u2013 | \u2013 |\n"
        "| Gold | \u2013 | \u00d70.1 | \u00b15% | \u2013 |\n"
        "| Silber | \u2013 | \u00d70.01 | \u00b110% | \u2013 |"))

# ---- Rechner (rechts) ----
# ============================================================
# RECHNER 1.7: Widerstand-Farbcode Decoder (4 / 5 / 6 Ringe)
# ============================================================

COLORS = {
    'Schwarz': {'digit': 0, 'mult': 1,      'tol': None,     'tk': 250},
    'Braun':   {'digit': 1, 'mult': 10,     'tol': '±1%',    'tk': 100},
    'Rot':     {'digit': 2, 'mult': 100,    'tol': '±2%',    'tk': 50},
    'Orange':  {'digit': 3, 'mult': 1e3,    'tol': None,     'tk': 15},
    'Gelb':    {'digit': 4, 'mult': 1e4,    'tol': None,     'tk': 25},
    'Grün':    {'digit': 5, 'mult': 1e5,    'tol': '±0.5%',  'tk': None},
    'Blau':    {'digit': 6, 'mult': 1e6,    'tol': '±0.25%', 'tk': 10},
    'Violett': {'digit': 7, 'mult': 1e7,    'tol': '±0.1%',  'tk': 5},
    'Grau':    {'digit': 8, 'mult': 1e8,    'tol': '±0.05%', 'tk': None},
    'Weiss':   {'digit': 9, 'mult': None,   'tol': None,     'tk': None},
    'Gold':    {'digit': None, 'mult': 0.1, 'tol': '±5%',    'tk': None},
    'Silber':  {'digit': None, 'mult': 0.01,'tol': '±10%',   'tk': None},
}

digit_colors = [c for c, v in COLORS.items() if v['digit'] is not None]
mult_colors  = [c for c, v in COLORS.items() if v['mult']  is not None]
tol_colors   = [c for c, v in COLORS.items() if v['tol']   is not None]
tk_colors    = [c for c, v in COLORS.items() if v['tk']    is not None]

dd_style  = {'description_width': '160px'}
dd_layout = widgets.Layout(width='340px')

# Ringanzahl wählen
ring_count = widgets.ToggleButtons(
    options=['4 Ringe', '5 Ringe', '6 Ringe'],
    description='Ausführung:',
    style={'description_width': '160px', 'button_width': '90px'}
)

# Alle Ringe vorbereiten
r1 = widgets.Dropdown(options=digit_colors, description='Ring 1 (Ziffer 1):', style=dd_style, layout=dd_layout)
r2 = widgets.Dropdown(options=digit_colors, description='Ring 2 (Ziffer 2):', style=dd_style, layout=dd_layout)
r3 = widgets.Dropdown(options=digit_colors, description='Ring 3 (Ziffer 3):', style=dd_style, layout=dd_layout)
r4 = widgets.Dropdown(options=mult_colors,  description='Ring 4 (Multiplik.):', style=dd_style, layout=dd_layout)
r5 = widgets.Dropdown(options=tol_colors,   description='Ring 5 (Toleranz):', style=dd_style, layout=dd_layout, value='Gold')
r6 = widgets.Dropdown(options=tk_colors,    description='Ring 6 (TK ppm/K):', style=dd_style, layout=dd_layout)

# Ring-Boxen für 4 / 5 / 6
box_4 = widgets.VBox([r1, r2,
                      widgets.Dropdown(options=mult_colors, description='Ring 3 (Multiplik.):', style=dd_style, layout=dd_layout),
                      widgets.Dropdown(options=tol_colors,  description='Ring 4 (Toleranz):',  style=dd_style, layout=dd_layout, value='Gold')])
box_5 = widgets.VBox([r1, r2, r3, r4, r5])
box_6 = widgets.VBox([r1, r2, r3, r4, r5, r6])

# eigene refs für 4-Ring (separate Dropdowns im box_4)
r3_4 = box_4.children[2]
r4_4 = box_4.children[3]

out_farb = widgets.Output()
container = widgets.VBox([])

def update_view(_=None):
    if ring_count.value == '4 Ringe':
        container.children = [box_4, out_farb]
    elif ring_count.value == '5 Ringe':
        container.children = [box_5, out_farb]
    else:
        container.children = [box_6, out_farb]
    calc_farb()

def calc_farb(_=None):
    out_farb.clear_output()
    with out_farb:
        try:
            mode = ring_count.value
            if mode == '4 Ringe':
                d1   = COLORS[r1.value]['digit']
                d2   = COLORS[r2.value]['digit']
                mult = COLORS[r3_4.value]['mult']
                tol  = COLORS[r4_4.value]['tol']
                R    = (d1 * 10 + d2) * mult
                tk_str = ''
            elif mode == '5 Ringe':
                d1   = COLORS[r1.value]['digit']
                d2   = COLORS[r2.value]['digit']
                d3   = COLORS[r3.value]['digit']
                mult = COLORS[r4.value]['mult']
                tol  = COLORS[r5.value]['tol']
                R    = (d1 * 100 + d2 * 10 + d3) * mult
                tk_str = ''
            else:  # 6 Ringe
                d1   = COLORS[r1.value]['digit']
                d2   = COLORS[r2.value]['digit']
                d3   = COLORS[r3.value]['digit']
                mult = COLORS[r4.value]['mult']
                tol  = COLORS[r5.value]['tol']
                tk   = COLORS[r6.value]['tk']
                R    = (d1 * 100 + d2 * 10 + d3) * mult
                tk_str = f"   TK    = {tk} ppm/K" if tk else ''

            tol_num = float(tol.strip('±%'))
            R_min = R * (1 - tol_num / 100)
            R_max = R * (1 + tol_num / 100)
            print(f"R      = {fmt(R, 'Ω')}  {tol}")
            print(f"Bereich: {fmt(R_min, 'Ω')} … {fmt(R_max, 'Ω')}")
            if tk_str:
                print(tk_str)
        except Exception as e:
            print("Fehler:", e)

ring_count.observe(update_view, names='value')
for w in [r1, r2, r3, r4, r5, r6, r3_4, r4_4]:
    w.observe(calc_farb, names='value')

update_view()
_calc_widget = (widgets.VBox([ring_count, container]))

# ---- HBox Layout ----
_left_box = widgets.Box([_theory_out], layout=widgets.Layout(**LEFT_STYLE))
_right_box = widgets.Box([_calc_widget], layout=widgets.Layout(**RIGHT_STYLE))

display(widgets.HBox(
    [_left_box, _right_box],
    layout=widgets.Layout(
        width='100%',
        align_items='flex-start',
        margin='8px 0 28px 0',
    )
))


In [ ]:
display(HTML('<a name="sec-1-8"></a>'))
# ================================================================
# LAYOUT: Theorie (links) | Rechner (rechts)
# ================================================================

# ---- Theorie (links) ----
_theory_out = widgets.Output(layout=THEORY_STYLE)

with _theory_out:
    display(Markdown(
        "---\n"
        "### 1.8. Spannungsteiler (unbelastet)\n"
        "$$U_2 = U_{ges} \\cdot \\frac{R_2}{R_1 + R_2}$$\n"
        "Oder allgemein:\n"
        "$$U_x = U_{ges} \\cdot \\frac{R_x}{R_{ges}}$$\n"
        "> **Wichtig:** Diese Formel gilt nur f\u00fcr den **unbelasteten** Spannungsteiler (kein Strom wird abgenommen)!"))

# ---- Rechner (rechts) ----
# ============================================================
# RECHNER 1.8: Spannungsteiler (unbelastet)
# ============================================================

Uges_st = make_input('U_ges [V] =')
R1_st   = make_input('R1 [Ω]   =')
R2_st   = make_input('R2 [Ω]   =')
U2_st   = make_input('U2 [V]   =')
btn_st  = make_button()
out_st  = widgets.Output()

def calc_st(_=None):
    # → Logik in calc.py: spannungsteiler()
    out_st.clear_output()
    with out_st:
        try:
            r = spannungsteiler(U_ges=p(Uges_st.value), R1=p(R1_st.value), R2=p(R2_st.value), U2=p(U2_st.value))
            _u = {'U_ges': 'V', 'U2': 'V', 'U1': 'V', 'R1': 'Ω', 'R2': 'Ω', 'I': 'A'}
            _skip = ('berechnet', 'ladetabelle', 'stromtabelle', 'widerspruch')
            for k, v in r.items():
                if k in _skip or v is None: continue
                if isinstance(v, bool):
                    if v: print(f'  {k}')
                    continue
                if isinstance(v, list): continue
                u = _u.get(k, '')
                print(f'  {k} = {fmt(v, u) if isinstance(v,(int,float)) else v}')
        except (ValueError, ZeroDivisionError) as e:
            print('i', e)
        except Exception as e:
            print('Fehler:', e)
btn_st.on_click(calc_st)
for f in [Uges_st, R1_st, R2_st, U2_st]:
    f.observe(calc_st, names='value')

_calc_widget = (widgets.VBox([Uges_st, R1_st, R2_st, U2_st, btn_st, out_st]))

# ---- HBox Layout ----
_left_box = widgets.Box([_theory_out], layout=widgets.Layout(**LEFT_STYLE))
_right_box = widgets.Box([_calc_widget], layout=widgets.Layout(**RIGHT_STYLE))

display(widgets.HBox(
    [_left_box, _right_box],
    layout=widgets.Layout(
        width='100%',
        align_items='flex-start',
        margin='8px 0 28px 0',
    )
))


In [ ]:
display(HTML('<a name="sec-1-9"></a>'))
# ================================================================
# LAYOUT: Theorie (links) | Rechner (rechts)
# ================================================================

# ---- Theorie (links) ----
_theory_out = widgets.Output(layout=THEORY_STYLE)

with _theory_out:
    display(Markdown(
        "---\n"
        "### 1.9 Leitwert und Siemens\n"
        "Der **Leitwert** $G$ ist der Kehrwert des Widerstands.\n"
        "Er gibt an wie gut ein Bauteil den Strom leitet.\n"
        "$$G = \\frac{1}{R} \\qquad [S] = \\left[\\frac{1}{\\Omega}\\right]$$\n"
        "$$I = G \\cdot U \\qquad G_{ges,parallel} = G_1 + G_2 + G_3 + \\ldots$$\n"
        "> Parallelschaltung mit Leitwerten ist einfacher als mit Widerst\u00e4nden \u2013\n"
        "> Leitwerte addieren sich direkt!"))

# ---- Rechner (rechts) ----
# ============================================================
# RECHNER 1.9: Leitwert G = 1/R
# ============================================================

lw_R   = make_input('R [Ω]  =')
lw_G   = make_input('G [S]  =')
lw_U   = make_input('U [V]  =', 'optional → I berechnen')
btn_lw = make_button()
out_lw = widgets.Output()

def calc_lw(_=None):
    # → Logik in calc.py: leitwert()
    out_lw.clear_output()
    with out_lw:
        try:
            r = leitwert(R=p(lw_R.value), G=p(lw_G.value))
            _u = {'G': 'S', 'R': 'Ω'}
            _skip = ('berechnet', 'ladetabelle', 'stromtabelle', 'widerspruch')
            for k, v in r.items():
                if k in _skip or v is None: continue
                if isinstance(v, bool):
                    if v: print(f'  {k}')
                    continue
                if isinstance(v, list): continue
                u = _u.get(k, '')
                print(f'  {k} = {fmt(v, u) if isinstance(v,(int,float)) else v}')
        except (ValueError, ZeroDivisionError) as e:
            print('i', e)
        except Exception as e:
            print('Fehler:', e)
btn_lw.on_click(calc_lw)
for fw in [lw_R, lw_G, lw_U]:
    fw.observe(calc_lw, names='value')
_calc_widget = (widgets.VBox([lw_R, lw_G, lw_U, btn_lw, out_lw]))

# ---- HBox Layout ----
_left_box = widgets.Box([_theory_out], layout=widgets.Layout(**LEFT_STYLE))
_right_box = widgets.Box([_calc_widget], layout=widgets.Layout(**RIGHT_STYLE))

display(widgets.HBox(
    [_left_box, _right_box],
    layout=widgets.Layout(
        width='100%',
        align_items='flex-start',
        margin='8px 0 28px 0',
    )
))


In [ ]:
display(HTML('<a name="sec-1-10"></a>'))
# ================================================================
# LAYOUT: Theorie (links) | Rechner (rechts)
# ================================================================

# ---- Theorie (links) ----
_theory_out = widgets.Output(layout=THEORY_STYLE)

with _theory_out:
    display(Markdown(
        "---\n"
        "### 1.10 Stromdichte\n"
        "Die **Stromdichte** $J$ beschreibt wie viel Strom pro Querschnittsfl\u00e4che fliesst.\n"
        "Zu hohe Stromdichte \u2192 \u00dcberhitzung der Leitung!\n"
        "$$J = \\frac{I}{A} \\qquad \\left[\\frac{A}{mm^2}\\right]$$\n"
        "| Leitungstyp | Zul. Stromdichte |\n"
        "|-------------|-----------------|\n"
        "| Festverlegung (NYM) | bis $6\\,A/mm^2$ |\n"
        "| Flexible Leitung | bis $4\\,A/mm^2$ |\n"
        "| Kurzzeit / Anlauf | bis $10\\,A/mm^2$ |\n"
        "| Leiterplatte (PCB) | bis $20\\,A/mm^2$ |\n"
        "### Querschnitt w\u00e4hlen\n"
        "$$A_{min} = \\frac{I}{J_{max}}$$"))

# ---- Rechner (rechts) ----
# ============================================================
# RECHNER 1.10: Stromdichte J = I/A
# ============================================================

jmax_select = widgets.Dropdown(
    options=[('-- Leitungstyp wählen --', None),
             ('Festverlegung NYM', 6), ('Flexible Leitung', 4),
             ('Kurzzeit / Anlauf', 10), ('Leiterplatte PCB', 20)],
    description='Leitungstyp:', style={'description_width': '140px'},
    layout=widgets.Layout(width='340px')
)

sd_I   = make_input('I [A]      =')
sd_A   = make_input('A [mm²]    =', 'Querschnitt')
sd_J   = make_input('J [A/mm²]  =', 'leer = berechnen')
btn_sd = make_button()
out_sd = widgets.Output()

def set_jmax(change):
    if change['new'] is not None:
        sd_J.value = str(change['new'])
jmax_select.observe(set_jmax, names='value')

def calc_sd(_=None):
    # → Logik in calc.py: stromdichte()
    out_sd.clear_output()
    with out_sd:
        try:
            r = stromdichte(I=p(sd_I.value), A=p(sd_A.value), J=p(sd_J.value))
            _u = {'J': 'A/mm²', 'A': 'mm²', 'I': 'A'}
            _skip = ('berechnet', 'ladetabelle', 'stromtabelle', 'widerspruch')
            for k, v in r.items():
                if k in _skip or v is None: continue
                if isinstance(v, bool):
                    if v: print(f'  {k}')
                    continue
                if isinstance(v, list): continue
                u = _u.get(k, '')
                print(f'  {k} = {fmt(v, u) if isinstance(v,(int,float)) else v}')
        except (ValueError, ZeroDivisionError) as e:
            print('i', e)
        except Exception as e:
            print('Fehler:', e)
btn_sd.on_click(calc_sd)
for fw in [sd_I, sd_A, sd_J]:
    fw.observe(calc_sd, names='value')
_calc_widget = (widgets.VBox([jmax_select, sd_I, sd_A, sd_J, btn_sd, out_sd]))

# ---- HBox Layout ----
_left_box = widgets.Box([_theory_out], layout=widgets.Layout(**LEFT_STYLE))
_right_box = widgets.Box([_calc_widget], layout=widgets.Layout(**RIGHT_STYLE))

display(widgets.HBox(
    [_left_box, _right_box],
    layout=widgets.Layout(
        width='100%',
        align_items='flex-start',
        margin='8px 0 28px 0',
    )
))


In [ ]:
display(HTML('<a name="sec-1-11"></a>'))
# ================================================================
# LAYOUT: Theorie (links) | Rechner (rechts)
# ================================================================

# ---- Theorie (links) ----
_theory_out = widgets.Output(layout=THEORY_STYLE)

with _theory_out:
    display(Markdown(
        "---\n"
        "### 1.11 Thevenin-Quelle und Norton-\u00c4quivalent\n"
        "Jede lineare Schaltung (egal wie komplex) kann vereinfacht werden zu:\n"
        "**Thevenin-\u00c4quivalent:** Spannungsquelle $U_{th}$ in Reihe mit $R_{th}$\n"
        "$$U_{th} = U_{Leerlauf} \\qquad R_{th} = \\frac{U_{Leerlauf}}{I_{Kurzschluss}}$$\n"
        "**Norton-\u00c4quivalent:** Stromquelle $I_N$ parallel zu $R_N$\n"
        "$$I_N = I_{Kurzschluss} \\qquad R_N = R_{th}$$\n"
        "#### Umrechnung Thevenin \u2194 Norton\n"
        "$$U_{th} = I_N \\cdot R_N \\qquad I_N = \\frac{U_{th}}{R_{th}} \\qquad R_{th} = R_N$$\n"
        "#### Spannungsteiler-Methode (einfache Schaltungen)\n"
        "$$U_{th} = U_{ges} \\cdot \\frac{R_2}{R_1 + R_2} \\qquad R_{th} = R_1 \\| R_2 = \\frac{R_1 \\cdot R_2}{R_1 + R_2}$$\n"
        "> **Anwendung:** Maximale Leistungs\u00fcbertragung wenn $R_{Last} = R_{th}$ (Leistungsanpassung)\n"
        "> $$P_{max} = \\frac{U_{th}^2}{4 \\cdot R_{th}}$$"))

# ---- Rechner (rechts) ----
# ============================================================
# RECHNER 1.11: Thevenin / Norton Äquivalent
# ============================================================

th_mode = make_toggle('Methode:', ['Ull + Ik → Thevenin', 'Uth + Rth → Norton', 'Spannungsteiler'], '200px')
th_Ull  = make_input('U_Leerlauf [V]  =', 'Spannung ohne Last')
th_Ik   = make_input('I_Kurzschluss [A]=','Strom bei Kurzschluss')
th_Uth  = make_input('U_th [V]         =', 'Thevenin-Spannung')
th_Rth  = make_input('R_th [Ω]         =', 'Thevenin-Widerstand')
th_Rl   = make_input('R_Last [Ω]       =', 'optional: Lastspannung')
# Spannungsteiler Felder
th_Uges = make_input('U_ges [V]  =', 'Versorgungsspannung')
th_R1   = make_input('R1 [Ω]     =', 'oberer Widerstand')
th_R2   = make_input('R2 [Ω]     =', 'unterer Widerstand')
btn_th  = make_button()
out_th  = widgets.Output()
box_ull = widgets.VBox([th_Ull, th_Ik])
box_nr  = widgets.VBox([th_Uth, th_Rth])
box_st  = widgets.VBox([th_Uges, th_R1, th_R2])
container_th = widgets.VBox([])

def update_th_view(_=None):
    m = th_mode.value
    if m == 'Ull + Ik → Thevenin':
        container_th.children = [box_ull, th_Rl, btn_th, out_th]
    elif m == 'Uth + Rth → Norton':
        container_th.children = [box_nr, th_Rl, btn_th, out_th]
    else:
        container_th.children = [box_st, th_Rl, btn_th, out_th]
    calc_th()

def calc_th(_=None):
    # → Logik in calc.py: thevenin_aus_ull_ik / thevenin_aus_spannungsteiler()
    out_th.clear_output()
    with out_th:
        try:
            mode = th_mode.value
            RL = p(th_Rl.value)
            if mode == 'Ull + Ik → Thevenin':
                r = thevenin_aus_ull_ik(Ull=p(th_Ull.value), Ik=p(th_Ik.value), R_last=RL)
            elif mode == 'Spannungsteiler':
                r = thevenin_aus_spannungsteiler(U_ges=p(th_Uges.value),
                                                  R1=p(th_R1.value),
                                                  R2=p(th_R2.value),
                                                  R_last=RL)
            else:  # Uth + Rth → Norton
                Uth = p(th_Uth.value); Rth = p(th_Rth.value)
                if not Uth or not Rth: print("U_th und R_th eingeben"); return
                IN = Uth / Rth
                r = {'Uth': Uth, 'Rth': Rth, 'IN': IN, 'P_max': Uth**2/(4*Rth)}
                if RL: r.update({'U_last': Uth*RL/(Rth+RL), 'I_last': Uth/(Rth+RL)})
            _u = {'Uth':'V','Rth':'Ω','IN':'A','P_max':'W','U_last':'V','I_last':'A','P_last':'W'}
            for k, v in r.items():
                if v is None or isinstance(v,(bool,list)): continue
                u = _u.get(k,'')
                print(f'  {k} = {fmt(v, u) if isinstance(v,(int,float)) else v}')
        except (ValueError, ZeroDivisionError) as e:
            print('i', e)
        except Exception as e:
            print('Fehler:', e)
th_mode.observe(update_th_view, names='value')
btn_th.on_click(calc_th)
for fw in [th_Ull, th_Ik, th_Uth, th_Rth, th_Rl, th_Uges, th_R1, th_R2]:
    fw.observe(calc_th, names='value')
update_th_view()
_calc_widget = (widgets.VBox([th_mode, container_th]))

# ---- HBox Layout ----
_left_box = widgets.Box([_theory_out], layout=widgets.Layout(**LEFT_STYLE))
_right_box = widgets.Box([_calc_widget], layout=widgets.Layout(**RIGHT_STYLE))

display(widgets.HBox(
    [_left_box, _right_box],
    layout=widgets.Layout(
        width='100%',
        align_items='flex-start',
        margin='8px 0 28px 0',
    )
))


In [ ]:
display(HTML('<a name="sec-1-12"></a>'))
# ================================================================
# LAYOUT: Theorie (links) | Rechner (rechts)
# ================================================================

# ---- Theorie (links) ----
_theory_out = widgets.Output(layout=THEORY_STYLE)

with _theory_out:
    display(Markdown(
        "---\n"
        "### 1.12 Belasteter Spannungsteiler\n"
        "Wenn ein **Verbraucher (Last)** $R_L$ angeschlossen wird, verändert sich die Ausgangsspannung –\n"
        "weil $R_L$ parallel zu $R_2$ liegt und so den effektiven unteren Widerstand verkleinert.\n"
        "\n"
        "#### Ersatzwiderstand $R_2$ parallel $R_L$\n"
        "$$R_{2L} = \\frac{R_2 \\cdot R_L}{R_2 + R_L}$$\n"
        "\n"
        "#### Belastete Ausgangsspannung\n"
        "$$U_2 = U_{ges} \\cdot \\frac{R_{2L}}{R_1 + R_{2L}}$$\n"
        "\n"
        "#### Vergleich unbelastet / belastet\n"
        "| | Unbelastet | Belastet |\n"
        "|--|--|--|\n"
        "| Formel | $U_2 = U \\cdot \\frac{R_2}{R_1+R_2}$ | $U_2 = U \\cdot \\frac{R_{2L}}{R_1+R_{2L}}$ |\n"
        "| $R_L$ | $\\infty$ | endlich |\n"
        "| $U_2$ | maximal | kleiner |\n"
        "\n"
        "> **Faustregel:** Damit der Spannungsteiler \u201estabil\u201c arbeitet,\n"
        "> sollte $R_L \\geq 10 \\cdot R_2$ sein.\n"
        "> Sonst bricht die Spannung stark ein.\n"
    ))

# ---- Rechner (rechts) ----
# ============================================================
# RECHNER 1.12: Belasteter Spannungsteiler
# ============================================================

bst_Uges = make_input('U_ges [V] =')
bst_R1   = make_input('R1 [Ω]   =', 'oberer Widerstand')
bst_R2   = make_input('R2 [Ω]   =', 'unterer Widerstand')
bst_RL   = make_input('R_L [Ω]  =', 'Last (leer = unbelastet)')
btn_bst  = make_button()
out_bst  = widgets.Output()

def calc_bst(_=None):
    out_bst.clear_output()
    with out_bst:
        try:
            U  = p(bst_Uges.value)
            R1 = p(bst_R1.value)
            R2 = p(bst_R2.value)
            RL = p(bst_RL.value)
            if any(x is None for x in [U, R1, R2]):
                print("U_ges, R1 und R2 eingeben"); return

            # Unbelastet
            U2_unb = U * R2 / (R1 + R2)
            I_unb  = U / (R1 + R2)
            print(f"  --- Unbelastet ---")
            print(f"  U2       = {fmt(U2_unb, 'V')}")
            print(f"  I        = {fmt(I_unb, 'A')}")

            if RL is not None:
                # Belastet
                R2L  = R2 * RL / (R2 + RL)
                U2_b = U * R2L / (R1 + R2L)
                I_ges = U / (R1 + R2L)
                I_R2  = U2_b / R2
                I_RL  = U2_b / RL
                abw   = (U2_unb - U2_b) / U2_unb * 100

                print(f"\n  --- Belastet (R_L = {fmt(RL, 'Ω')}) ---")
                print(f"  R2 ‖ R_L = {fmt(R2L, 'Ω')}")
                print(f"  U2       = {fmt(U2_b, 'V')}  (−{abw:.2f}% gegenüber unbelastet)")
                print(f"  I_ges    = {fmt(I_ges, 'A')}")
                print(f"  I_R2     = {fmt(I_R2, 'A')}")
                print(f"  I_RL     = {fmt(I_RL, 'A')}")
                if RL >= 10 * R2:
                    print(f"\n  ✓ R_L ≥ 10·R2 → stabile Belastung")
                else:
                    print(f"\n  ⚠ R_L < 10·R2 → starke Spannungsabweichung!")
        except Exception as e:
            print("Fehler:", e)

btn_bst.on_click(calc_bst)
for fw in [bst_Uges, bst_R1, bst_R2, bst_RL]:
    fw.observe(calc_bst, names='value')
_calc_widget = widgets.VBox([bst_Uges, bst_R1, bst_R2, bst_RL, btn_bst, out_bst])

# ---- HBox Layout ----
_left_box  = widgets.Box([_theory_out], layout=widgets.Layout(**LEFT_STYLE))
_right_box = widgets.Box([_calc_widget], layout=widgets.Layout(**RIGHT_STYLE))

display(widgets.HBox(
    [_left_box, _right_box],
    layout=widgets.Layout(
        width='100%',
        align_items='flex-start',
        margin='8px 0 28px 0',
    )
))


In [ ]:
display(HTML('<a name="sec-2-1"></a>'))
# ================================================================
# LAYOUT: Theorie (links) | Rechner (rechts)
# ================================================================

# ---- Theorie (links) ----
_theory_out = widgets.Output(layout=THEORY_STYLE)

with _theory_out:
    display(Markdown(
        "---\n"
        "## Kapitel 2: Kondensatoren\n"
        "---\n"
        "## 2.1  Grundlagen des Kondensators\n"
        "Ein **Kondensator** speichert elektrische Energie im elektrischen Feld zwischen zwei leitenden Platten.\n"
        "| Gr\u00f6sse | Symbol | Einheit |\n"
        "|--------|--------|---------|\n"
        "| Kapazit\u00e4t | $C$ | Farad [F] |\n"
        "| Ladung | $Q$ | Coulomb [C] |\n"
        "| Spannung | $U$ | Volt [V] |\n"
        "### Grundformel\n"
        "$$Q = C \\cdot U$$\n"
        "> **Energie im Kondensator:**\n"
        "> $$W = \\frac{1}{2} \\cdot C \\cdot U^2$$"))

# ---- Rechner (rechts) ----
# ============================================================
# RECHNER 2.1: Grundgrössen Q = C · U
# ============================================================

Q_inp = make_input('Q [C]  =', 'z.B. 100u, 4.7m')
C_inp = make_input('C [F]  =', 'z.B. 100n, 4.7u')
U_inp = make_input('U [V]  =')
btn_qcu = make_button()
out_qcu = widgets.Output()

def calc_qcu(_=None):
    # → Logik in calc.py: kondensator_qcu()
    out_qcu.clear_output()
    with out_qcu:
        try:
            r = kondensator_qcu(Q=p(Q_inp.value), C=p(C_inp.value), U=p(U_inp.value))
            _u = {'Q': 'C', 'C': 'F', 'U': 'V', 'W': 'J'}
            _skip = ('berechnet', 'ladetabelle', 'stromtabelle', 'widerspruch')
            for k, v in r.items():
                if k in _skip or v is None: continue
                if isinstance(v, bool):
                    if v: print(f'  {k}')
                    continue
                if isinstance(v, list): continue
                u = _u.get(k, '')
                print(f'  {k} = {fmt(v, u) if isinstance(v,(int,float)) else v}')
        except (ValueError, ZeroDivisionError) as e:
            print('i', e)
        except Exception as e:
            print('Fehler:', e)
btn_qcu.on_click(calc_qcu)
for f in [Q_inp, C_inp, U_inp]:
    f.observe(calc_qcu, names='value')
_calc_widget = (widgets.VBox([Q_inp, C_inp, U_inp, btn_qcu, out_qcu]))

# ---- HBox Layout ----
_left_box = widgets.Box([_theory_out], layout=widgets.Layout(**LEFT_STYLE))
_right_box = widgets.Box([_calc_widget], layout=widgets.Layout(**RIGHT_STYLE))

display(widgets.HBox(
    [_left_box, _right_box],
    layout=widgets.Layout(
        width='100%',
        align_items='flex-start',
        margin='8px 0 28px 0',
    )
))


In [ ]:
display(HTML('<a name="sec-2-2"></a>'))
# ================================================================
# LAYOUT: Theorie (links) | Rechner (rechts)
# ================================================================

# ---- Theorie (links) ----
_theory_out = widgets.Output(layout=THEORY_STYLE)

with _theory_out:
    display(Markdown(
        "---\n"
        "## 2.2 Plattenkondensator\n"
        "$$C = \\varepsilon_0 \\cdot \\varepsilon_r \\cdot \\frac{A}{d}$$\n"
        "| Gr\u00f6sse | Symbol | Wert / Einheit |\n"
        "|--------|--------|----------------|\n"
        "| Permittivit\u00e4t Vakuum | $\\varepsilon_0$ | $8.854 \\times 10^{-12}$ F/m |\n"
        "| Relative Permittivit\u00e4t | $\\varepsilon_r$ | dimensionslos |\n"
        "| Plattenfl\u00e4che | $A$ | m\u00b2 |\n"
        "| Plattenabstand | $d$ | m |\n"
        "\n"
        "| Material | $\\varepsilon_r$ |\n"
        "|----------|-----------------|\n"
        "| Luft / Vakuum | 1.0 |\n"
        "| Papier | 2.0\u20132.5 |\n"
        "| Glas | 4\u201310 |\n"
        "| Keramik | 10\u201310'000 |\n"
        "| Polyester | 3.2 |\n"))

# ---- Rechner (rechts) ----
# ============================================================
# RECHNER 2.2: Plattenkondensator  C = ε0·εr·A/d
# ============================================================

eps_mat = widgets.Dropdown(
    options=[('-- Material wählen --', None), ('Luft/Vakuum', 1.0),
             ('Papier', 2.2), ('Polyester', 3.2), ('Glas', 6.0), ('Keramik', 100.0)],
    description='Material:', style={'description_width': '130px'},
    layout=widgets.Layout(width='320px')
)

C_plat  = make_input('C [F]   =', 'leer = berechnen')
A_plat  = make_input('A [m²]  =', 'z.B. 0.01')
d_plat  = make_input('d [m]   =', 'z.B. 1m, 0.5m')
er_plat = make_input('ε_r     =', 'z.B. 1 für Luft')
btn_plat = make_button()
out_plat = widgets.Output()

def set_er(change):
    if change['new'] is not None:
        er_plat.value = str(change['new'])
eps_mat.observe(set_er, names='value')

def calc_plat(_=None):
    # → Logik in calc.py: plattenkondensator()
    out_plat.clear_output()
    with out_plat:
        try:
            r = plattenkondensator(C=p(C_plat.value), A=p(A_plat.value), d=p(d_plat.value), er=p(er_plat.value))
            _u = {'C': 'F', 'A': 'm²', 'd': 'm', 'er': ''}
            _skip = ('berechnet', 'ladetabelle', 'stromtabelle', 'widerspruch')
            for k, v in r.items():
                if k in _skip or v is None: continue
                if isinstance(v, bool):
                    if v: print(f'  {k}')
                    continue
                if isinstance(v, list): continue
                u = _u.get(k, '')
                print(f'  {k} = {fmt(v, u) if isinstance(v,(int,float)) else v}')
        except (ValueError, ZeroDivisionError) as e:
            print('i', e)
        except Exception as e:
            print('Fehler:', e)
btn_plat.on_click(calc_plat)
for f in [C_plat, A_plat, d_plat, er_plat]:
    f.observe(calc_plat, names='value')
_calc_widget = (widgets.VBox([eps_mat, C_plat, A_plat, d_plat, er_plat, btn_plat, out_plat]))

# ---- HBox Layout ----
_left_box = widgets.Box([_theory_out], layout=widgets.Layout(**LEFT_STYLE))
_right_box = widgets.Box([_calc_widget], layout=widgets.Layout(**RIGHT_STYLE))

display(widgets.HBox(
    [_left_box, _right_box],
    layout=widgets.Layout(
        width='100%',
        align_items='flex-start',
        margin='8px 0 28px 0',
    )
))


In [ ]:
display(HTML('<a name="sec-2-3"></a>'))
# ================================================================
# LAYOUT: Theorie (links) | Rechner (rechts)
# ================================================================

# ---- Theorie (links) ----
_theory_out = widgets.Output(layout=THEORY_STYLE)

with _theory_out:
    display(Markdown(
        "---\n"
        "## 2.3 Kondensatoren in Reihe und Parallel\n"
        "### Parallelschaltung\n"
        "$$C_{ges} = C_1 + C_2 + C_3 + \\ldots$$\n"
        "- Gleiche **Spannung** an allen Kondensatoren\n"
        "- Kapazit\u00e4ten addieren sich direkt\n"
        "### Reihenschaltung\n"
        "$$\\frac{1}{C_{ges}} = \\frac{1}{C_1} + \\frac{1}{C_2} + \\frac{1}{C_3} + \\ldots$$\n"
        "- Gleiche **Ladung** auf allen Kondensatoren\n"
        "- Gesamtkapazit\u00e4t **kleiner** als kleinster Einzelwert"))

# ---- Rechner (rechts) ----
# ============================================================
# RECHNER 2.3: Kondensatoren Reihe / Parallel
# ============================================================

schalt_toggle = widgets.ToggleButtons(
    options=['Parallelschaltung', 'Reihenschaltung'],
    description='Schaltung:', style={'description_width': '130px', 'button_width': '160px'}
)
c_fields = [make_input(f'C{i+1} [F] =', 'z.B. 100n, 4.7u') for i in range(5)]
U_kond   = make_input('U_ges [V] =', 'optional')
btn_kond = make_button()
out_kond = widgets.Output()

def calc_kond(_=None):
    out_kond.clear_output()
    with out_kond:
        try:
            Cs = [p(f.value) for f in c_fields]
            Cs_v = [c for c in Cs if c is not None]
            Uv   = p(U_kond.value)
            if len(Cs_v) < 1:
                print("Mindestens 1 Kondensator eingeben"); return

            if schalt_toggle.value == 'Parallelschaltung':
                C_ges = sum(Cs_v)
                print(f"C_ges (parallel) = {fmt(C_ges, 'F')}")
                if Uv:
                    print(f"   Q_ges = {fmt(C_ges * Uv, 'C')}")
                    for i, c in enumerate(Cs):
                        if c: print(f"   Q_C{i+1} = {fmt(c * Uv, 'C')}  (U = {fmt(Uv, 'V')})")
            else:
                C_ges = 1 / sum(1/c for c in Cs_v)
                print(f"C_ges (Reihe) = {fmt(C_ges, 'F')}")
                if Uv:
                    Q = C_ges * Uv
                    print(f"   Q     = {fmt(Q, 'C')}  (gleich auf allen)")
                    for i, c in enumerate(Cs):
                        if c: print(f"   U_C{i+1} = {fmt(Q / c, 'V')}")
        except Exception as e:
            print("Fehler:", e)

btn_kond.on_click(calc_kond)
schalt_toggle.observe(calc_kond, names='value')
for f in c_fields + [U_kond]:
    f.observe(calc_kond, names='value')
_calc_widget = (widgets.VBox([schalt_toggle] + c_fields + [U_kond, btn_kond, out_kond]))

# ---- HBox Layout ----
_left_box = widgets.Box([_theory_out], layout=widgets.Layout(**LEFT_STYLE))
_right_box = widgets.Box([_calc_widget], layout=widgets.Layout(**RIGHT_STYLE))

display(widgets.HBox(
    [_left_box, _right_box],
    layout=widgets.Layout(
        width='100%',
        align_items='flex-start',
        margin='8px 0 28px 0',
    )
))


In [ ]:
display(HTML('<a name="sec-2-4"></a>'))
# ================================================================
# LAYOUT: Theorie (links) | Rechner (rechts)
# ================================================================

# ---- Theorie (links) ----
_theory_out = widgets.Output(layout=THEORY_STYLE)

with _theory_out:
    display(Markdown(
        "---\n"
        "## 2.4 RC-Glied \u2013 Zeitkonstante und Lade-/Entladekurve\n"
        "$$\\tau = R \\cdot C$$\n"
        "| Zeit | Ladespannung $U_C$ | Restspannung (Entladung) |\n"
        "|------|--------------------|--------------------------|\n"
        "| $1\\tau$ | 63.2% von $U_0$ | 36.8% |\n"
        "| $2\\tau$ | 86.5% | 13.5% |\n"
        "| $3\\tau$ | 95.0% | 5.0% |\n"
        "| $4\\tau$ | 98.2% | 1.8% |\n"
        "| $5\\tau$ | 99.3% | 0.7% \u2248 vollst\u00e4ndig |\n"
        "**Ladespannung:**\n"
        "$$U_C(t) = U_0 \\cdot \\left(1 - e^{-t/\\tau}\\right)$$\n"
        "**Entladespannung:**\n"
        "$$U_C(t) = U_0 \\cdot e^{-t/\\tau}$$\n"
        "**Ladestrom:**\n"
        "$$i(t) = \\frac{U_0}{R} \\cdot e^{-t/\\tau}$$"))

# ---- Rechner (rechts) ----
# ============================================================
# RECHNER 2.4a: Zeitkonstante τ = R·C
# ============================================================

tau_R = make_input('R [Ω]  =')
tau_C = make_input('C [F]  =', 'z.B. 100n, 4.7u')
tau_T = make_input('τ [s]  =', 'z.B. 10m = 10ms')
btn_tau = make_button()
out_tau = widgets.Output()

def calc_tau(_=None):
    # → Logik in calc.py: rc_zeitkonstante()
    out_tau.clear_output()
    with out_tau:
        try:
            r = rc_zeitkonstante(R=p(tau_R.value), C=p(tau_C.value), tau=p(tau_T.value))
            _u = {'tau': 's', 'R': 'Ω', 'C': 'F'}
            _skip = ('berechnet', 'ladetabelle', 'stromtabelle', 'widerspruch')
            for k, v in r.items():
                if k in _skip or v is None: continue
                if isinstance(v, bool):
                    if v: print(f'  {k}')
                    continue
                if isinstance(v, list): continue
                u = _u.get(k, '')
                print(f'  {k} = {fmt(v, u) if isinstance(v,(int,float)) else v}')
        except (ValueError, ZeroDivisionError) as e:
            print('i', e)
        except Exception as e:
            print('Fehler:', e)
btn_tau.on_click(calc_tau)
for f in [tau_R, tau_C, tau_T]:
    f.observe(calc_tau, names='value')
# _calc_widget = (widgets.VBox([tau_R, tau_C, tau_T, btn_tau, out_tau]))

# ============================================================
# RECHNER 2.4b: Spannung / Strom zu bestimmtem Zeitpunkt
# ============================================================

rc2_mode = widgets.ToggleButtons(
    options=['Laden', 'Entladen'],
    description='Vorgang:', style={'description_width': '130px', 'button_width': '100px'}
)
rc2_R  = make_input('R [Ω]  =')
rc2_C  = make_input('C [F]  =', 'z.B. 100n')
rc2_U0 = make_input('U₀ [V] =', 'Quellspannung')
rc2_t  = make_input('t [s]  =', 'z.B. 5m = 5ms')
btn_rc2 = make_button()
out_rc2 = widgets.Output()

def calc_rc2(_=None):
    out_rc2.clear_output()
    with out_rc2:
        try:
            R  = p(rc2_R.value);  C  = p(rc2_C.value)
            U0 = p(rc2_U0.value); t  = p(rc2_t.value)
            if any(x is None for x in [R, C, U0, t]):
                print("Alle 4 Felder ausfüllen"); return
            tau = R * C
            if rc2_mode.value == 'Laden':
                Uc = U0 * (1 - math.exp(-t / tau))
                Ur = U0 - Uc
                I  = (U0 / R) * math.exp(-t / tau)
            else:
                Uc = U0 * math.exp(-t / tau)
                Ur = U0 - Uc
                I  = (U0 / R) * math.exp(-t / tau)
            print(f"τ     = {fmt(tau, 's')}")
            print(f"t/τ   = {t/tau:.4g}")
            print(f"U_C   = {fmt(Uc, 'V')}  ({Uc/U0*100:.2f}% von U₀)")
            print(f"U_R   = {fmt(Ur, 'V')}")
            print(f"I     = {fmt(I, 'A')}")
        except Exception as e:
            print("Fehler:", e)

btn_rc2.on_click(calc_rc2)
rc2_mode.observe(calc_rc2, names='value')
for f in [rc2_R, rc2_C, rc2_U0, rc2_t]:
    f.observe(calc_rc2, names='value')

_calc_widget = (widgets.VBox([tau_R, tau_C, tau_T, btn_tau, out_tau, rc2_mode, rc2_R, rc2_C, rc2_U0, rc2_t, btn_rc2, out_rc2]))

# ---- HBox Layout ----
_left_box = widgets.Box([_theory_out], layout=widgets.Layout(**LEFT_STYLE))
_right_box = widgets.Box([_calc_widget], layout=widgets.Layout(**RIGHT_STYLE))

display(widgets.HBox(
    [_left_box, _right_box],
    layout=widgets.Layout(
        width='100%',
        align_items='flex-start',
        margin='8px 0 28px 0',
    )
))


In [ ]:
display(HTML('<a name="sec-2-5"></a>'))
# ================================================================
# LAYOUT: Theorie (links) | Rechner (rechts)
# ================================================================

# ---- Theorie (links) ----
_theory_out = widgets.Output(layout=THEORY_STYLE)

with _theory_out:
    display(Markdown(
        "---\n"
        "## 2.5 Kondensator an Wechselspannung \u2013 Kapazitiver Widerstand\n"
        "An Wechselspannung wirkt der Kondensator wie ein **frequenzabh\u00e4ngiger Widerstand**:\n"
        "$$X_C = \\frac{1}{2\\pi \\cdot f \\cdot C} = \\frac{1}{\\omega \\cdot C}$$\n"
        "| Gr\u00f6sse | Symbol | Einheit |\n"
        "|--------|--------|---------|\n"
        "| Kapazitiver Widerstand | $X_C$ | \u03a9 |\n"
        "| Frequenz | $f$ | Hz |\n"
        "| Kreisfrequenz | $\\omega = 2\\pi f$ | rad/s |\n"
        "| Kapazit\u00e4t | $C$ | F |\n"
        "> **Wichtig:** Bei h\u00f6herer Frequenz \u2192 kleiner $X_C$ \u2192 Kondensator \\\"leitet besser\\\"\n"
        "> Bei Gleichspannung (f = 0) \u2192 $X_C = \\infty$ \u2192 Kondensator sperrt!\n"
        "**Phasenverschiebung:** Der Strom **eilt** der Spannung um **90\u00b0** voraus.\n"
        "$$I = \\frac{U}{X_C}$$"))

# ---- Rechner (rechts) ----
# ============================================================
# RECHNER 2.5: Kapazitiver Widerstand X_C = 1 / (2π·f·C)
# ============================================================

xc_Xc = make_input('X_C [Ω] =')
xc_f  = make_input('f [Hz]  =', 'z.B. 50, 1k, 10k')
xc_C  = make_input('C [F]   =', 'z.B. 100n, 4.7u')
xc_U  = make_input('U [V]   =', 'optional → I berechnen')
btn_xc = make_button()
out_xc = widgets.Output()

def calc_xc(_=None):
    # → Logik in calc.py: kapazitiver_widerstand()
    out_xc.clear_output()
    with out_xc:
        try:
            r = kapazitiver_widerstand(Xc=p(xc_Xc.value), f=p(xc_f.value), C=p(xc_C.value))
            _u = {'Xc': 'Ω', 'f': 'Hz', 'C': 'F', 'omega': 'rad/s'}
            _skip = ('berechnet', 'ladetabelle', 'stromtabelle', 'widerspruch')
            for k, v in r.items():
                if k in _skip or v is None: continue
                if isinstance(v, bool):
                    if v: print(f'  {k}')
                    continue
                if isinstance(v, list): continue
                u = _u.get(k, '')
                print(f'  {k} = {fmt(v, u) if isinstance(v,(int,float)) else v}')
        except (ValueError, ZeroDivisionError) as e:
            print('i', e)
        except Exception as e:
            print('Fehler:', e)
btn_xc.on_click(calc_xc)
for f_w in [xc_Xc, xc_f, xc_C, xc_U]:
    f_w.observe(calc_xc, names='value')
_calc_widget = (widgets.VBox([xc_Xc, xc_f, xc_C, xc_U, btn_xc, out_xc]))

# ---- HBox Layout ----
_left_box = widgets.Box([_theory_out], layout=widgets.Layout(**LEFT_STYLE))
_right_box = widgets.Box([_calc_widget], layout=widgets.Layout(**RIGHT_STYLE))

display(widgets.HBox(
    [_left_box, _right_box],
    layout=widgets.Layout(
        width='100%',
        align_items='flex-start',
        margin='8px 0 28px 0',
    )
))


In [ ]:
display(HTML('<a name="sec-2-6"></a>'))
# ================================================================
# LAYOUT: Theorie (links) | Rechner (rechts)
# ================================================================

# ---- Theorie (links) ----
_theory_out = widgets.Output(layout=THEORY_STYLE)

with _theory_out:
    display(Markdown(
        "---\n"
        "## 2.6 RC-Reihenschaltung an Wechselspannung \u2013 Impedanz\n"
        "$$Z = \\sqrt{R^2 + X_C^2}$$\n"
        "$$\\varphi = -\\arctan\\!\\left(\\frac{X_C}{R}\\right)$$\n"
        "| Gr\u00f6sse | Symbol | Einheit |\n"
        "|--------|--------|---------|\n"
        "| Impedanz (Scheinwiderstand) | $Z$ | \u03a9 |\n"
        "| Wirkwiderstand | $R$ | \u03a9 |\n"
        "| Kapazitiver Widerstand | $X_C$ | \u03a9 |\n"
        "| Phasenwinkel | $\\varphi$ | \u00b0 |\n"
        "**Spannungen:**\n"
        "$$U_{ges} = \\sqrt{U_R^2 + U_C^2}$$\n"
        "> Spannungen d\u00fcrfen **nicht** direkt addiert werden \u2013 sie sind phasenverschoben!"))

# ---- Rechner (rechts) ----
# ============================================================
# RECHNER 2.6: RC-Reihenschaltung Impedanz Z = √(R² + Xc²)
# ============================================================

rc_wR  = make_input('R [Ω]  =')
rc_wf  = make_input('f [Hz]  =', 'z.B. 50, 1k')
rc_wC  = make_input('C [F]   =', 'z.B. 100n')
rc_wU  = make_input('U_ges [V]=', 'optional')
btn_rcw = make_button()
out_rcw = widgets.Output()

def calc_rcw(_=None):
    out_rcw.clear_output()
    with out_rcw:
        try:
            R = p(rc_wR.value); f = p(rc_wf.value)
            C = p(rc_wC.value); U = p(rc_wU.value)
            if any(x is None for x in [R, f, C]):
                print("R, f und C eingeben"); return
            Xc = 1 / (2 * math.pi * f * C)
            Z  = math.sqrt(R**2 + Xc**2)
            phi = -math.degrees(math.atan(Xc / R))
            print(f"X_C  = {fmt(Xc, 'Ω')}")
            print(f"Z    = {fmt(Z, 'Ω')}")
            print(f"φ    = {phi:.2f}°  (Strom eilt Spannung vor)")
            if U:
                I = U / Z
                Ur = I * R
                Uc = I * Xc
                print(f"   I    = {fmt(I, 'A')}")
                print(f"   U_R  = {fmt(Ur, 'V')}")
                print(f"   U_C  = {fmt(Uc, 'V')}")
                print(f"   Probe: √(U_R²+U_C²) = {fmt(math.sqrt(Ur**2+Uc**2), 'V')}")
        except Exception as e:
            print("Fehler:", e)

btn_rcw.on_click(calc_rcw)
for f_w in [rc_wR, rc_wf, rc_wC, rc_wU]:
    f_w.observe(calc_rcw, names='value')
_calc_widget = (widgets.VBox([rc_wR, rc_wf, rc_wC, rc_wU, btn_rcw, out_rcw]))

# ---- HBox Layout ----
_left_box = widgets.Box([_theory_out], layout=widgets.Layout(**LEFT_STYLE))
_right_box = widgets.Box([_calc_widget], layout=widgets.Layout(**RIGHT_STYLE))

display(widgets.HBox(
    [_left_box, _right_box],
    layout=widgets.Layout(
        width='100%',
        align_items='flex-start',
        margin='8px 0 28px 0',
    )
))


In [ ]:
display(HTML('<a name="sec-2-7"></a>'))
# ================================================================
# LAYOUT: Theorie (links) | Rechner (rechts)
# ================================================================

# ---- Theorie (links) ----
_theory_out = widgets.Output(layout=THEORY_STYLE)

with _theory_out:
    display(Markdown(
        "---\n"
        "## 2.7 Elektrisches Feld\n"
        "Das **elektrische Feld** beschreibt den Einfluss einer Ladung auf den umgebenden Raum.\n"
        "| Gr\u00f6sse | Symbol | Einheit |\n"
        "|--------|--------|---------|\n"
        "| Elektrische Feldst\u00e4rke | $E$ | V/m |\n"
        "| Elektrische Flussdichte | $D$ | C/m\u00b2 |\n"
        "| Spannung | $U$ | V |\n"
        "| Plattenabstand | $d$ | m |\n"
        "| Permittivit\u00e4t | $\\varepsilon = \\varepsilon_0 \\cdot \\varepsilon_r$ | F/m |\n"
        "### Feldst\u00e4rke im Plattenkondensator (homogenes Feld)\n"
        "$$E = \\frac{U}{d} \\qquad [V/m]$$\n"
        "### Elektrische Flussdichte\n"
        "$$D = \\varepsilon_0 \\cdot \\varepsilon_r \\cdot E = \\frac{Q}{A} \\qquad [C/m^2]$$\n"
        "### Zusammenhang\n"
        "$$D = \\varepsilon \\cdot E \\qquad C = \\varepsilon \\cdot \\frac{A}{d}$$\n"
        "> Die Feldst\u00e4rke $E$ gibt an wie stark das Feld ist (unabh\u00e4ngig vom Material).\n"
        "> Die Flussdichte $D$ ber\u00fccksichtigt das Dielektrikum."))

# ---- Rechner (rechts) ----
# ============================================================
# RECHNER 2.7: Elektrisches Feld E und D
# ============================================================

ef_mat = widgets.Dropdown(
    options=[('-- Dielektrikum --', None), ('Luft/Vakuum', 1.0),
             ('Papier', 2.2), ('Polyester', 3.2), ('Glas', 6.0), ('Keramik', 100.0)],
    description='Material:', style={'description_width': '130px'},
    layout=widgets.Layout(width='320px')
)

ef_E   = make_input('E [V/m]  =')
ef_U   = make_input('U [V]    =')
ef_d   = make_input('d [m]    =', 'Plattenabstand')
ef_D   = make_input('D [C/m²] =', 'optional')
ef_er  = make_input('ε_r      =', 'für D-Berechnung')
btn_ef = make_button()
out_ef = widgets.Output()

def set_er_ef(change):
    if change['new'] is not None:
        ef_er.value = str(change['new'])
ef_mat.observe(set_er_ef, names='value')

def calc_ef(_=None):
    # → Logik in calc.py: elektrisches_feld()
    out_ef.clear_output()
    with out_ef:
        try:
            r = elektrisches_feld(E=p(ef_E.value), U=p(ef_U.value), d=p(ef_d.value), er=p(ef_er.value))
            _u = {'E': 'V/m', 'U': 'V', 'd': 'm', 'D': 'C/m²'}
            _skip = ('berechnet', 'ladetabelle', 'stromtabelle', 'widerspruch')
            for k, v in r.items():
                if k in _skip or v is None: continue
                if isinstance(v, bool):
                    if v: print(f'  {k}')
                    continue
                if isinstance(v, list): continue
                u = _u.get(k, '')
                print(f'  {k} = {fmt(v, u) if isinstance(v,(int,float)) else v}')
        except (ValueError, ZeroDivisionError) as e:
            print('i', e)
        except Exception as e:
            print('Fehler:', e)
btn_ef.on_click(calc_ef)
for fw in [ef_E, ef_U, ef_d, ef_er]:
    fw.observe(calc_ef, names='value')
_calc_widget = (widgets.VBox([ef_mat, ef_E, ef_U, ef_d, ef_er, btn_ef, out_ef]))

# ---- HBox Layout ----
_left_box = widgets.Box([_theory_out], layout=widgets.Layout(**LEFT_STYLE))
_right_box = widgets.Box([_calc_widget], layout=widgets.Layout(**RIGHT_STYLE))

display(widgets.HBox(
    [_left_box, _right_box],
    layout=widgets.Layout(
        width='100%',
        align_items='flex-start',
        margin='8px 0 28px 0',
    )
))


In [ ]:
display(HTML('<a name="sec-3-1"></a>'))
# ================================================================
# LAYOUT: Theorie (links) | Rechner (rechts)
# ================================================================

# ---- Theorie (links) ----
_theory_out = widgets.Output(layout=THEORY_STYLE)

with _theory_out:
    display(Markdown(
        "---\n"
        "# Kapitel 3: Spulen (Induktivit\u00e4ten)\n"
        "---\n"
        "## 3.1 Grundlagen der Spule\n"
        "Eine **Spule** speichert elektrische Energie im **magnetischen Feld**.\n"
        "| Gr\u00f6sse | Symbol | Einheit |\n"
        "|--------|--------|---------|\n"
        "| Induktivit\u00e4t | $L$ | Henry [H] |\n"
        "| Spannung | $U_L$ | Volt [V] |\n"
        "| Strom | $I$ | Ampere [A] |\n"
        "| Magnetischer Fluss | $\\Phi$ | Weber [Wb] |\n"
        "### Grundformel \u2013 Selbstinduktion\n"
        "$$U_L = L \\cdot \\frac{\\Delta I}{\\Delta t}$$\n"
        "Die Spule **widersetzt sich jeder Strom\u00e4nderung**.\n"
        "### Energie in der Spule\n"
        "$$W = \\frac{1}{2} \\cdot L \\cdot I^2$$"))

# ---- Rechner (rechts) ----
# ============================================================
# RECHNER 3.1: Selbstinduktionsspannung  U_L = L · ΔI/Δt
# ============================================================

ul_UL  = make_input('U_L [V]    =')
ul_L   = make_input('L [H]      =', 'z.B. 10m, 4.7u')
ul_dI  = make_input('ΔI [A]     =', 'Stromänderung')
ul_dt  = make_input('Δt [s]     =', 'z.B. 1m = 1ms')
btn_ul = make_button()
out_ul = widgets.Output()

def calc_ul(_=None):
    # → Logik in calc.py: selbstinduktion()
    out_ul.clear_output()
    with out_ul:
        try:
            r = selbstinduktion(UL=p(ul_UL.value), L=p(ul_L.value), dI=p(ul_dI.value), dt=p(ul_dt.value))
            _u = {'UL': 'V', 'L': 'H', 'dI': 'A', 'dt': 's', 'W': 'J'}
            _skip = ('berechnet', 'ladetabelle', 'stromtabelle', 'widerspruch')
            for k, v in r.items():
                if k in _skip or v is None: continue
                if isinstance(v, bool):
                    if v: print(f'  {k}')
                    continue
                if isinstance(v, list): continue
                u = _u.get(k, '')
                print(f'  {k} = {fmt(v, u) if isinstance(v,(int,float)) else v}')
        except (ValueError, ZeroDivisionError) as e:
            print('i', e)
        except Exception as e:
            print('Fehler:', e)
btn_ul.on_click(calc_ul)
for f in [ul_UL, ul_L, ul_dI, ul_dt]:
    f.observe(calc_ul, names='value')
_calc_widget = (widgets.VBox([ul_UL, ul_L, ul_dI, ul_dt, btn_ul, out_ul]))

# ---- HBox Layout ----
_left_box = widgets.Box([_theory_out], layout=widgets.Layout(**LEFT_STYLE))
_right_box = widgets.Box([_calc_widget], layout=widgets.Layout(**RIGHT_STYLE))

display(widgets.HBox(
    [_left_box, _right_box],
    layout=widgets.Layout(
        width='100%',
        align_items='flex-start',
        margin='8px 0 28px 0',
    )
))


In [ ]:
display(HTML('<a name="sec-3-2"></a>'))
# ================================================================
# LAYOUT: Theorie (links) | Rechner (rechts)
# ================================================================

# ---- Theorie (links) ----
_theory_out = widgets.Output(layout=THEORY_STYLE)

with _theory_out:
    display(Markdown(
        "---\n"
        "## 3.2 Induktivit\u00e4t einer Spule (Zylinderspule)\n"
        "$$L = \\mu_0 \\cdot \\mu_r \\cdot \\frac{N^2 \\cdot A}{l}$$\n"
        "| Gr\u00f6sse | Symbol | Wert / Einheit |\n"
        "|--------|--------|----------------|\n"
        "| Permeabilit\u00e4t Vakuum | $\\mu_0$ | $4\\pi \\times 10^{-7}$ H/m |\n"
        "| Relative Permeabilit\u00e4t | $\\mu_r$ | dimensionslos |\n"
        "| Windungszahl | $N$ | \u2013 |\n"
        "| Querschnittsfl\u00e4che | $A$ | m\u00b2 |\n"
        "| Spulenl\u00e4nge | $l$ | m |\n"
        " \n"
        "| Material | $\\mu_r$ |\n"
        "|----------|---------|\n"
        "| Luft / Vakuum | 1 |\n"
        "| Ferrit | 10\u201310'000 |\n"
        "| Siliziumstahl | 1'000\u201310'000 |\n"
        "| Mu-Metall | bis 100'000 |"))

# ---- Rechner (rechts) ----
# ============================================================
# RECHNER 3.2: Induktivität Zylinderspule  L = µ0·µr·N²·A/l
# ============================================================

kern_mat = widgets.Dropdown(
    options=[('-- Kernmaterial --', None), ('Luft/Vakuum', 1),
             ('Ferrit (mittel)', 1000), ('Siliziumstahl', 5000), ('Mu-Metall', 50000)],
    description='Kernmaterial:', style={'description_width': '130px'},
    layout=widgets.Layout(width='320px')
)

sp_L   = make_input('L [H]    =', 'leer = berechnen')
sp_mur = make_input('µ_r      =', 'z.B. 1 für Luft')
sp_N   = make_input('N (Windungen) =', 'Anzahl')
sp_A   = make_input('A [m²]   =', 'z.B. 0.001')
sp_l   = make_input('l [m]    =', 'Spulenlänge')
btn_sp = make_button()
out_sp = widgets.Output()

def set_mur(change):
    if change['new'] is not None:
        sp_mur.value = str(change['new'])
kern_mat.observe(set_mur, names='value')

def calc_sp(_=None):
    out_sp.clear_output()
    with out_sp:
        try:
            L   = p(sp_L.value)
            mur = p(sp_mur.value)
            N   = p(sp_N.value)
            A   = p(sp_A.value)
            l   = p(sp_l.value)
            vals = {'L': L, 'µ_r': mur, 'N': N, 'A': A, 'l': l}
            miss = [k for k, v in vals.items() if v is None]
            if len(miss) != 1:
                print("Genau ein Feld leer lassen!" if len(miss) > 1 else "Ein Feld leer lassen!")
                return
            m = miss[0]
            if   m == 'L':   res = MU0 * mur * N**2 * A / l;           print(f"L   = {fmt(res, 'H')}")
            elif m == 'µ_r': res = L * l / (MU0 * N**2 * A);           print(f"µ_r = {res:.4g}")
            elif m == 'N':   res = math.sqrt(L * l / (MU0 * mur * A)); print(f"N   = {res:.4g} Windungen")
            elif m == 'A':   res = L * l / (MU0 * mur * N**2);         print(f"A   = {fmt(res, 'm²')}")
            else:            res = MU0 * mur * N**2 * A / L;           print(f"l   = {fmt(res, 'm')}")
        except Exception as e:
            print("Fehler:", e)

btn_sp.on_click(calc_sp)
for f in [sp_L, sp_mur, sp_N, sp_A, sp_l]:
    f.observe(calc_sp, names='value')
_calc_widget = (widgets.VBox([kern_mat, sp_L, sp_mur, sp_N, sp_A, sp_l, btn_sp, out_sp]))

# ---- HBox Layout ----
_left_box = widgets.Box([_theory_out], layout=widgets.Layout(**LEFT_STYLE))
_right_box = widgets.Box([_calc_widget], layout=widgets.Layout(**RIGHT_STYLE))

display(widgets.HBox(
    [_left_box, _right_box],
    layout=widgets.Layout(
        width='100%',
        align_items='flex-start',
        margin='8px 0 28px 0',
    )
))


In [ ]:
display(HTML('<a name="sec-3-3"></a>'))
# ================================================================
# LAYOUT: Theorie (links) | Rechner (rechts)
# ================================================================

# ---- Theorie (links) ----
_theory_out = widgets.Output(layout=THEORY_STYLE)

with _theory_out:
    display(Markdown(
        "---\n"
        "## 3.3 Spulen in Reihe und Parallel\n"
        "### Reihenschaltung (ohne gegenseitige Beeinflussung)\n"
        "$$L_{ges} = L_1 + L_2 + L_3 + \\ldots$$\n"
        "### Parallelschaltung\n"
        "$$\\frac{1}{L_{ges}} = \\frac{1}{L_1} + \\frac{1}{L_2} + \\frac{1}{L_3} + \\ldots$$\n"
        "> Gleiche Regeln wie bei Widerst\u00e4nden \u2013 aber umgekehrt zu Kondensatoren!"))

# ---- Rechner (rechts) ----
# ============================================================
# RECHNER 3.3: Spulen Reihe / Parallel
# ============================================================

l_toggle = widgets.ToggleButtons(
    options=['Reihenschaltung', 'Parallelschaltung'],
    description='Schaltung:', style={'description_width': '130px', 'button_width': '160px'}
)
l_fields = [make_input(f'L{i+1} [H] =', 'z.B. 10m, 4.7u') for i in range(5)]
btn_lp = make_button()
out_lp = widgets.Output()

def calc_lp(_=None):
    out_lp.clear_output()
    with out_lp:
        try:
            Ls = [p(f.value) for f in l_fields]
            Ls_v = [l for l in Ls if l is not None]
            if len(Ls_v) < 1:
                print("Mindestens 1 Spule eingeben"); return
            if l_toggle.value == 'Reihenschaltung':
                L_ges = sum(Ls_v)
                print(f"L_ges (Reihe)    = {fmt(L_ges, 'H')}")
            else:
                L_ges = 1 / sum(1/l for l in Ls_v)
                print(f"L_ges (Parallel) = {fmt(L_ges, 'H')}")
        except Exception as e:
            print("Fehler:", e)

btn_lp.on_click(calc_lp)
l_toggle.observe(calc_lp, names='value')
for f in l_fields:
    f.observe(calc_lp, names='value')
_calc_widget = (widgets.VBox([l_toggle] + l_fields + [btn_lp, out_lp]))

# ---- HBox Layout ----
_left_box = widgets.Box([_theory_out], layout=widgets.Layout(**LEFT_STYLE))
_right_box = widgets.Box([_calc_widget], layout=widgets.Layout(**RIGHT_STYLE))

display(widgets.HBox(
    [_left_box, _right_box],
    layout=widgets.Layout(
        width='100%',
        align_items='flex-start',
        margin='8px 0 28px 0',
    )
))


In [ ]:
display(HTML('<a name="sec-3-4"></a>'))
# ================================================================
# LAYOUT: Theorie (links) | Rechner (rechts)
# ================================================================

# ---- Theorie (links) ----
_theory_out = widgets.Output(layout=THEORY_STYLE)

with _theory_out:
    display(Markdown(
        "---\n"
        "## 3.4 RL-Glied \u2013 Zeitkonstante und Ein-/Ausschalten\n"
        "$$\\tau = \\frac{L}{R}$$\n"
        "| Zeit | Strom beim Einschalten | Strom beim Ausschalten |\n"
        "|------|------------------------|------------------------|\n"
        "| $1\\tau$ | 63.2% von $I_{max}$ | 36.8% |\n"
        "| $2\\tau$ | 86.5% | 13.5% |\n"
        "| $3\\tau$ | 95.0% | 5.0% |\n"
        "| $5\\tau$ | 99.3% | 0.7% \u2248 vollst\u00e4ndig |\n"
        "**Einschalten (Strom steigt):**\n"
        "$$i(t) = I_{max} \\cdot \\left(1 - e^{-t/\\tau}\\right) \\qquad I_{max} = \\frac{U_0}{R}$$\n"
        "**Ausschalten (Strom f\u00e4llt):**\n"
        "$$i(t) = I_{max} \\cdot e^{-t/\\tau}$$\n"
        "**Spannung an der Spule:**\n"
        "$$u_L(t) = U_0 \\cdot e^{-t/\\tau}$$\n"
        "> Beim **Ausschalten** entsteht eine hohe Induktionsspannung! Deshalb werden Freilaufdioden verwendet."))

# ---- Rechner (rechts) ----
# ============================================================
# RECHNER 3.4a: Zeitkonstante τ = L/R
# ============================================================

rl_L   = make_input('L [H]  =', 'z.B. 10m, 4.7u')
rl_R   = make_input('R [Ω]  =')
rl_tau = make_input('τ [s]  =')
btn_rlt = make_button()
out_rlt = widgets.Output()

def calc_rlt(_=None):
    # → Logik in calc.py: rl_zeitkonstante()
    out_rlt.clear_output()
    with out_rlt:
        try:
            r = rl_zeitkonstante(L=p(rl_L.value), R=p(rl_R.value), tau=p(rl_tau.value))
            _u = {'tau': 's', 'L': 'H', 'R': 'Ω'}
            _skip = ('berechnet', 'ladetabelle', 'stromtabelle', 'widerspruch')
            for k, v in r.items():
                if k in _skip or v is None: continue
                if isinstance(v, bool):
                    if v: print(f'  {k}')
                    continue
                if isinstance(v, list): continue
                u = _u.get(k, '')
                print(f'  {k} = {fmt(v, u) if isinstance(v,(int,float)) else v}')
        except (ValueError, ZeroDivisionError) as e:
            print('i', e)
        except Exception as e:
            print('Fehler:', e)
btn_rlt.on_click(calc_rlt)
for f in [rl_L, rl_R, rl_tau]:
    f.observe(calc_rlt, names='value')

# ============================================================
# RECHNER 3.4b: Strom / Spannung zu Zeitpunkt t im RL-Glied
# ============================================================

rl2_mode = widgets.ToggleButtons(
    options=['Einschalten', 'Ausschalten'],
    description='Vorgang:', style={'description_width': '130px', 'button_width': '120px'}
)
rl2_L  = make_input('L [H]   =', 'z.B. 10m')
rl2_R  = make_input('R [Ω]   =')
rl2_U0 = make_input('U₀ [V]  =', 'Quellspannung')
rl2_t  = make_input('t [s]   =', 'z.B. 5m = 5ms')
btn_rl2 = make_button()
out_rl2 = widgets.Output()

def calc_rl2(_=None):
    out_rl2.clear_output()
    with out_rl2:
        try:
            L  = p(rl2_L.value);  R  = p(rl2_R.value)
            U0 = p(rl2_U0.value); t  = p(rl2_t.value)
            if any(x is None for x in [L, R, U0, t]):
                print("Alle 4 Felder ausfüllen"); return
            tau   = L / R
            I_max = U0 / R
            if rl2_mode.value == 'Einschalten':
                I  = I_max * (1 - math.exp(-t / tau))
                UL = U0 * math.exp(-t / tau)
                UR = I * R
            else:
                I  = I_max * math.exp(-t / tau)
                UL = -U0 * math.exp(-t / tau)
                UR = I * R
            print(f"τ      = {fmt(tau, 's')}")
            print(f"I_max  = {fmt(I_max, 'A')}")
            print(f"t/τ    = {t/tau:.4g}")
            print(f"I      = {fmt(I, 'A')}  ({I/I_max*100:.2f}% von I_max)")
            print(f"U_L    = {fmt(UL, 'V')}")
            print(f"U_R    = {fmt(UR, 'V')}")
        except Exception as e:
            print("Fehler:", e)

btn_rl2.on_click(calc_rl2)
rl2_mode.observe(calc_rl2, names='value')
for f in [rl2_L, rl2_R, rl2_U0, rl2_t]:
    f.observe(calc_rl2, names='value')

_calc_widget = (widgets.VBox([rl_L, rl_R, rl_tau, btn_rlt, out_rlt, rl2_mode, rl2_L, rl2_R, rl2_U0, rl2_t, btn_rl2, out_rl2]))

# ---- HBox Layout ----
_left_box = widgets.Box([_theory_out], layout=widgets.Layout(**LEFT_STYLE))
_right_box = widgets.Box([_calc_widget], layout=widgets.Layout(**RIGHT_STYLE))

display(widgets.HBox(
    [_left_box, _right_box],
    layout=widgets.Layout(
        width='100%',
        align_items='flex-start',
        margin='8px 0 28px 0',
    )
))


In [ ]:
display(HTML('<a name="sec-3-5"></a>'))
# ================================================================
# LAYOUT: Theorie (links) | Rechner (rechts)
# ================================================================

# ---- Theorie (links) ----
_theory_out = widgets.Output(layout=THEORY_STYLE)

with _theory_out:
    display(Markdown(
        "---\n"
        "## 3.5 Spule an Wechselspannung \u2013 Induktiver Widerstand\n"
        "$$X_L = 2\\pi \\cdot f \\cdot L = \\omega \\cdot L$$\n"
        "| Gr\u00f6sse | Symbol | Einheit |\n"
        "|--------|--------|---------|\n"
        "| Induktiver Widerstand | $X_L$ | \u03a9 |\n"
        "| Frequenz | $f$ | Hz |\n"
        "| Kreisfrequenz | $\\omega = 2\\pi f$ | rad/s |\n"
        "| Induktivit\u00e4t | $L$ | H |\n"
        "> **Wichtig:** Bei h\u00f6herer Frequenz \u2192 gr\u00f6sser $X_L$ \u2192 Spule sperrt besser\n"
        "> Bei Gleichspannung (f = 0) \u2192 $X_L = 0$ \u2192 Spule ist nur ihr Wicklungswiderstand\n"
        "**Phasenverschiebung:** Der Strom **eilt** der Spannung um **90\u00b0 nach**.\n"
        "$$I = \\frac{U}{X_L}$$"))

# ---- Rechner (rechts) ----
# ============================================================
# RECHNER 3.5: Induktiver Widerstand  X_L = 2π·f·L
# ============================================================

xl_XL = make_input('X_L [Ω] =')
xl_f  = make_input('f [Hz]  =', 'z.B. 50, 1k, 10k')
xl_L  = make_input('L [H]   =', 'z.B. 10m, 4.7u')
xl_U  = make_input('U [V]   =', 'optional → I berechnen')
btn_xl = make_button()
out_xl = widgets.Output()

def calc_xl(_=None):
    # → Logik in calc.py: induktiver_widerstand()
    out_xl.clear_output()
    with out_xl:
        try:
            r = induktiver_widerstand(XL=p(xl_XL.value), f=p(xl_f.value), L=p(xl_L.value))
            _u = {'XL': 'Ω', 'f': 'Hz', 'L': 'H', 'omega': 'rad/s'}
            _skip = ('berechnet', 'ladetabelle', 'stromtabelle', 'widerspruch')
            for k, v in r.items():
                if k in _skip or v is None: continue
                if isinstance(v, bool):
                    if v: print(f'  {k}')
                    continue
                if isinstance(v, list): continue
                u = _u.get(k, '')
                print(f'  {k} = {fmt(v, u) if isinstance(v,(int,float)) else v}')
        except (ValueError, ZeroDivisionError) as e:
            print('i', e)
        except Exception as e:
            print('Fehler:', e)
btn_xl.on_click(calc_xl)
for fw in [xl_XL, xl_f, xl_L, xl_U]:
    fw.observe(calc_xl, names='value')
_calc_widget = (widgets.VBox([xl_XL, xl_f, xl_L, xl_U, btn_xl, out_xl]))

# ---- HBox Layout ----
_left_box = widgets.Box([_theory_out], layout=widgets.Layout(**LEFT_STYLE))
_right_box = widgets.Box([_calc_widget], layout=widgets.Layout(**RIGHT_STYLE))

display(widgets.HBox(
    [_left_box, _right_box],
    layout=widgets.Layout(
        width='100%',
        align_items='flex-start',
        margin='8px 0 28px 0',
    )
))


In [ ]:
display(HTML('<a name="sec-3-6"></a>'))
# ================================================================
# LAYOUT: Theorie (links) | Rechner (rechts)
# ================================================================

# ---- Theorie (links) ----
_theory_out = widgets.Output(layout=THEORY_STYLE)

with _theory_out:
    display(Markdown(
        "---\n"
        "## 3.6 RL-Reihenschaltung an Wechselspannung \u2013 Impedanz\n"
        "$$Z = \\sqrt{R^2 + X_L^2}$$\n"
        "$$\\varphi = +\\arctan\\!\\left(\\frac{X_L}{R}\\right)$$\n"
        "**Spannungen (nicht direkt addierbar!):**\n"
        "$$U_{ges} = \\sqrt{U_R^2 + U_L^2}$$\n"
        "| Vergleich | RC-Glied | RL-Glied |\n"
        "|-----------|----------|----------|\n"
        "| Phasenverschiebung | $\\varphi < 0\u00b0$ (I eilt vor) | $\\varphi > 0\u00b0$ (I eilt nach) |\n"
        "| Widerstand steigt mit f | Nein | Ja |\n"
        "| Widerstand f\u00e4llt mit f | Ja | Nein |"))

# ---- Rechner (rechts) ----
# ============================================================
# RECHNER 3.6: RL-Reihenschaltung Impedanz Z = √(R² + XL²)
# ============================================================

rl_wR  = make_input('R [Ω]   =')
rl_wf  = make_input('f [Hz]  =', 'z.B. 50, 1k')
rl_wL  = make_input('L [H]   =', 'z.B. 10m')
rl_wU  = make_input('U_ges [V]=', 'optional')
btn_rlw = make_button()
out_rlw = widgets.Output()

def calc_rlw(_=None):
    out_rlw.clear_output()
    with out_rlw:
        try:
            R = p(rl_wR.value); f = p(rl_wf.value)
            L = p(rl_wL.value); U = p(rl_wU.value)
            if any(x is None for x in [R, f, L]):
                print("R, f und L eingeben"); return
            XL  = 2 * math.pi * f * L
            Z   = math.sqrt(R**2 + XL**2)
            phi = math.degrees(math.atan(XL / R))
            print(f"X_L  = {fmt(XL, 'Ω')}")
            print(f"Z    = {fmt(Z, 'Ω')}")
            print(f"φ    = +{phi:.2f}°  (Strom eilt Spannung nach)")
            if U:
                I  = U / Z
                Ur = I * R
                Ul = I * XL
                print(f"   I    = {fmt(I, 'A')}")
                print(f"   U_R  = {fmt(Ur, 'V')}")
                print(f"   U_L  = {fmt(Ul, 'V')}")
                print(f"   Probe: √(U_R²+U_L²) = {fmt(math.sqrt(Ur**2+Ul**2), 'V')}")
        except Exception as e:
            print("Fehler:", e)

btn_rlw.on_click(calc_rlw)
for fw in [rl_wR, rl_wf, rl_wL, rl_wU]:
    fw.observe(calc_rlw, names='value')
_calc_widget = (widgets.VBox([rl_wR, rl_wf, rl_wL, rl_wU, btn_rlw, out_rlw]))

# ---- HBox Layout ----
_left_box = widgets.Box([_theory_out], layout=widgets.Layout(**LEFT_STYLE))
_right_box = widgets.Box([_calc_widget], layout=widgets.Layout(**RIGHT_STYLE))

display(widgets.HBox(
    [_left_box, _right_box],
    layout=widgets.Layout(
        width='100%',
        align_items='flex-start',
        margin='8px 0 28px 0',
    )
))


In [ ]:
display(HTML('<a name="sec-3-7"></a>'))
# ================================================================
# LAYOUT: Theorie (links) | Rechner (rechts)
# ================================================================

# ---- Theorie (links) ----
_theory_out = widgets.Output(layout=THEORY_STYLE)

with _theory_out:
    display(Markdown(
        "---\n"
        "## 3.7 Magnetisches Feld\n"
        "| Gr\u00f6sse | Symbol | Einheit |\n"
        "|--------|--------|---------|\n"
        "| Magnetische Feldst\u00e4rke | $H$ | A/m |\n"
        "| Magnetische Flussdichte | $B$ | Tesla [T] |\n"
        "| Magnetischer Fluss | $\\Phi$ | Weber [Wb] |\n"
        "| Durchflutung | $\\Theta$ | A (Amperewindungen) |\n"
        "| Permeabilit\u00e4t | $\\mu = \\mu_0 \\cdot \\mu_r$ | H/m |\n"
        "### Durchflutung\n"
        "$$\\Theta = N \\cdot I \\qquad [A]$$\n"
        "### Feldst\u00e4rke in einer Spule\n"
        "$$H = \\frac{\\Theta}{l} = \\frac{N \\cdot I}{l} \\qquad [A/m]$$\n"
        "### Flussdichte\n"
        "$$B = \\mu_0 \\cdot \\mu_r \\cdot H \\qquad [T]$$\n"
        "### Magnetischer Fluss\n"
        "$$\\Phi = B \\cdot A \\qquad [Wb] = [V \\cdot s]$$\n"
        "### Induktionsgesetz\n"
        "$$U_{ind} = N \\cdot \\frac{\\Delta \\Phi}{\\Delta t} \\qquad [V]$$"))

# ---- Rechner (rechts) ----
# ============================================================
# RECHNER 3.7: Magnetisches Feld – H, B, Φ, Θ
# ============================================================

mf_kern = widgets.Dropdown(
    options=[('-- Kernmaterial --', None), ('Luft/Vakuum', 1),
             ('Ferrit (mittel)', 1000), ('Siliziumstahl', 5000), ('Mu-Metall', 50000)],
    description='Kernmaterial:', style={'description_width': '140px'},
    layout=widgets.Layout(width='340px')
)

mf_N   = make_input('N (Windungen) =')
mf_I   = make_input('I [A]         =')
mf_l   = make_input('l [m]         =', 'Magnetkreislänge')
mf_A   = make_input('A [m²]        =', 'Kernquerschnitt')
mf_mur = make_input('µ_r           =', 'rel. Permeabilität')
btn_mf = make_button()
out_mf = widgets.Output()

def set_mur_mf(change):
    if change['new'] is not None:
        mf_mur.value = str(change['new'])
mf_kern.observe(set_mur_mf, names='value')

def calc_mf(_=None):
    out_mf.clear_output()
    with out_mf:
        try:
            N   = p(mf_N.value);   I   = p(mf_I.value)
            l   = p(mf_l.value);   A   = p(mf_A.value)
            mur = p(mf_mur.value)

            if any(x is None for x in [N, I, l]):
                print("N, I und l eingeben"); return

            Theta = N * I
            H     = Theta / l
            print(f"  Θ = N·I  = {fmt(Theta, 'A')}  (Amperewindungen)")
            print(f"  H = Θ/l  = {fmt(H, 'A/m')}")

            if mur:
                B   = MU0 * mur * H
                print(f"  B = µ₀·µᵣ·H = {B:.4e} T")
                if A:
                    Phi = B * A
                    print(f"  Φ = B·A  = {Phi:.4e} Wb")
                # Sättigung prüfen
                if mur > 1 and B > 1.5:
                    print(f"  B > 1.5 T – Kernsättigung möglich!")
        except Exception as e:
            print("Fehler:", e)

btn_mf.on_click(calc_mf)
for fw in [mf_N, mf_I, mf_l, mf_A, mf_mur]:
    fw.observe(calc_mf, names='value')
_calc_widget = (widgets.VBox([mf_kern, mf_N, mf_I, mf_l, mf_A, mf_mur, btn_mf, out_mf]))

# ---- HBox Layout ----
_left_box = widgets.Box([_theory_out], layout=widgets.Layout(**LEFT_STYLE))
_right_box = widgets.Box([_calc_widget], layout=widgets.Layout(**RIGHT_STYLE))

display(widgets.HBox(
    [_left_box, _right_box],
    layout=widgets.Layout(
        width='100%',
        align_items='flex-start',
        margin='8px 0 28px 0',
    )
))


In [ ]:
display(HTML('<a name="sec-4-0"></a>'))
# ================================================================
# LAYOUT: Theorie (links) | Rechner (rechts)
# ================================================================

# ---- Theorie (links) ----
_theory_out = widgets.Output(layout=THEORY_STYLE)

with _theory_out:
    display(Markdown(
        "---\n"
        "# Kapitel 4: Wechselstrom – Leistung\n"
        "---\n"
        "### 4.0 Effektiv-, Spitzen- und Mittelwert\n"
        "\n"
        "#### Sinusförmige Wechselspannung\n"
        "$$u(t) = \\hat{U} \\cdot \\sin(\\omega t + \\varphi_0)$$\n"
        "\n"
        "| Grösse | Symbol | Formel | Bedeutung |\n"
        "|--------|--------|--------|-----------|\n"
        "| Spitzenwert | $\\hat{U}$ | – | Maximale Amplitude |\n"
        "| Effektivwert | $U_{eff}$ | $\\hat{U}/\\sqrt{2}$ | Äquivalente DC-Leistungswirkung |\n"
        "| Gleichrichtwert | $\\bar{U}$ | $2\\hat{U}/\\pi$ | Mittelwert der gleichgericht. Kurve |\n"
        "| Scheitelfaktor | $k_s$ | $\\hat{U}/U_{eff} = \\sqrt{2}$ | Verhältnis Peak / Effektiv |\n"
        "| Formfaktor | $k_f$ | $U_{eff}/\\bar{U} = \\pi/(2\\sqrt{2})$ | Kurvenform-Kennwert |\n"
        "\n"
        "$$U_{eff} = \\frac{\\hat{U}}{\\sqrt{2}} \\approx 0.707 \\cdot \\hat{U}$$\n"
        "$$\\hat{U} = U_{eff} \\cdot \\sqrt{2} \\approx 1.414 \\cdot U_{eff}$$\n"
        "\n"
        "#### Nicht-sinusförmige und unsymmetrische Signale\n"
        "Für beliebige periodische Signale gilt die **allgemeine Definition**:\n"
        "$$U_{eff} = \\sqrt{\\frac{1}{T} \\int_0^T u^2(t)\\, dt}$$\n"
        "\n"
        "Diese Formel gilt immer – unabhängig von der Kurvenform!\n"
        "\n"
        "| Signalform | Effektivwert | Merkhilfe |\n"
        "|------------|-------------|-----------|\n"
        "| Sinus symmetrisch | $\\hat{U}/\\sqrt{2}$ | Standardfall |\n"
        "| Rechteck symmetrisch | $\\hat{U}$ | Immer bei Scheitelwert |\n"
        "| Rechteck Tastverhältnis D | $\\hat{U}\\cdot\\sqrt{D}$ | $D = t_{on}/T$ |\n"
        "| Dreieck symmetrisch | $\\hat{U}/\\sqrt{3}$ | |\n"
        "| Sinus + DC-Offset | $\\sqrt{U_{DC}^2 + (\\hat{U}/\\sqrt{2})^2}$ | Quadratische Addition |\n"
        "\n"
        "> **Wichtig:** Bei Mischspannung (AC + DC) addieren sich die\n"
        "> quadratischen Anteile: $U_{eff,ges} = \\sqrt{U_{DC}^2 + U_{AC,eff}^2}$\n"
        "\n"
        "> **Netz-Beispiel:** 230 V Netz = **Effektivwert**.\n"
        "> Spitzenwert: $\\hat{U} = 230 \\cdot \\sqrt{2} \\approx 325\\,V$\n"
    ))

# ---- Rechner (rechts) ----
# ============================================================
# RECHNER 4.0: Effektivwert / Spitzenwert (Sinus)
# ============================================================

ew_mode = widgets.ToggleButtons(
    options=['Sinus', 'Rechteck (D)', 'Dreieck', 'Sinus + DC-Offset'],
    description='Signal:',
    style={'description_width': '120px', 'button_width': '130px'}
)

ew_Up   = make_input('Û (Spitzenwert) [V] =', 'Amplitude')
ew_Ue   = make_input('U_eff [V]           =', 'leer = berechnen')
ew_D    = make_input('D (Tastverhältnis)  =', '0.0 – 1.0 z.B. 0.5')
ew_Udc  = make_input('U_DC [V]            =', 'Gleichanteil')
btn_ew  = make_button()
out_ew  = widgets.Output()

box_sin = widgets.VBox([ew_Up, ew_Ue])
box_rect = widgets.VBox([ew_Up, ew_Ue, ew_D])
box_tri  = widgets.VBox([ew_Up, ew_Ue])
box_dc   = widgets.VBox([ew_Up, ew_Udc, ew_Ue])
cont_ew  = widgets.VBox([])

def update_ew_view(_=None):
    m = ew_mode.value
    if m == 'Sinus':
        cont_ew.children = [box_sin, btn_ew, out_ew]
    elif m == 'Rechteck (D)':
        cont_ew.children = [box_rect, btn_ew, out_ew]
    elif m == 'Dreieck':
        cont_ew.children = [box_tri, btn_ew, out_ew]
    else:
        cont_ew.children = [box_dc, btn_ew, out_ew]
    calc_ew()

def calc_ew(_=None):
    out_ew.clear_output()
    with out_ew:
        try:
            Up  = p(ew_Up.value)
            Ue  = p(ew_Ue.value)
            D   = p(ew_D.value)
            Udc = p(ew_Udc.value)
            m   = ew_mode.value

            if m == 'Sinus':
                if Up is not None:
                    Ue_calc = Up / math.sqrt(2)
                    Ub_calc = 2 * Up / math.pi
                    ks = math.sqrt(2)
                    kf = math.pi / (2 * math.sqrt(2))
                    print(f"  Û        = {fmt(Up, 'V')}")
                    print(f"  U_eff    = {fmt(Ue_calc, 'V')}  (= Û/√2)")
                    print(f"  Ū        = {fmt(Ub_calc, 'V')}  (Gleichrichtwert)")
                    print(f"  k_s      = {ks:.4g}  (Scheitelfaktor)")
                    print(f"  k_f      = {kf:.4g}  (Formfaktor)")
                elif Ue is not None:
                    Up_calc = Ue * math.sqrt(2)
                    Ub_calc = 2 * Up_calc / math.pi
                    print(f"  U_eff    = {fmt(Ue, 'V')}")
                    print(f"  Û        = {fmt(Up_calc, 'V')}  (= U_eff·√2)")
                    print(f"  Ū        = {fmt(Ub_calc, 'V')}  (Gleichrichtwert)")
                else:
                    print("Û oder U_eff eingeben")

            elif m == 'Rechteck (D)':
                if D is None:
                    print("Tastverhältnis D eingeben (0.0 – 1.0)"); return
                if not (0 < D <= 1):
                    print("D muss zwischen 0 und 1 liegen"); return
                if Up is not None:
                    Ue_calc = Up * math.sqrt(D)
                    print(f"  Û        = {fmt(Up, 'V')}")
                    print(f"  D        = {D:.4g}")
                    print(f"  U_eff    = Û·√D = {fmt(Ue_calc, 'V')}")
                    if D == 0.5:
                        print(f"  (50% Tastverhältnis = symmetrisches Rechteck)")
                elif Ue is not None:
                    Up_calc = Ue / math.sqrt(D)
                    print(f"  U_eff    = {fmt(Ue, 'V')}")
                    print(f"  Û        = {fmt(Up_calc, 'V')}")
                else:
                    print("Û oder U_eff eingeben")

            elif m == 'Dreieck':
                if Up is not None:
                    Ue_calc = Up / math.sqrt(3)
                    print(f"  Û        = {fmt(Up, 'V')}")
                    print(f"  U_eff    = Û/√3 = {fmt(Ue_calc, 'V')}")
                    print(f"  k_s      = {math.sqrt(3):.4g}")
                elif Ue is not None:
                    Up_calc = Ue * math.sqrt(3)
                    print(f"  U_eff    = {fmt(Ue, 'V')}")
                    print(f"  Û        = {fmt(Up_calc, 'V')}")
                else:
                    print("Û oder U_eff eingeben")

            else:  # Sinus + DC
                if Up is None and Ue is None:
                    print("Û oder U_eff (AC-Anteil) eingeben"); return
                if Udc is None:
                    print("U_DC eingeben"); return
                if Up is not None:
                    Uac_eff = Up / math.sqrt(2)
                else:
                    Uac_eff = Ue
                    Up = Ue * math.sqrt(2)
                Uges_eff = math.sqrt(Udc**2 + Uac_eff**2)
                print(f"  U_DC     = {fmt(Udc, 'V')}")
                print(f"  Û (AC)   = {fmt(Up, 'V')}")
                print(f"  U_eff(AC)= {fmt(Uac_eff, 'V')}")
                print(f"  U_eff,ges= √(U_DC²+U_AC²) = {fmt(Uges_eff, 'V')}")
        except Exception as e:
            print("Fehler:", e)

ew_mode.observe(update_ew_view, names='value')
btn_ew.on_click(calc_ew)
for fw in [ew_Up, ew_Ue, ew_D, ew_Udc]:
    fw.observe(calc_ew, names='value')
update_ew_view()
_calc_widget = widgets.VBox([ew_mode, cont_ew])

# ---- HBox Layout ----
_left_box  = widgets.Box([_theory_out], layout=widgets.Layout(**LEFT_STYLE))
_right_box = widgets.Box([_calc_widget], layout=widgets.Layout(**RIGHT_STYLE))

display(widgets.HBox(
    [_left_box, _right_box],
    layout=widgets.Layout(
        width='100%',
        align_items='flex-start',
        margin='8px 0 28px 0',
    )
))


In [ ]:
display(HTML('<a name="sec-4-1"></a>'))
# ================================================================
# LAYOUT: Theorie (links) | Rechner (rechts)
# ================================================================

# ---- Theorie (links) ----
_theory_out = widgets.Output(layout=THEORY_STYLE)

with _theory_out:
    display(Markdown(
        "---\n"
        "## 4.1 Wirkleistung, Blindleistung, Scheinleistung\n"
        "Im Wechselstromkreis unterscheidet man drei Leistungsarten:\n"
        "| Gr\u00f6sse | Symbol | Einheit | Bedeutung |\n"
        "|--------|--------|---------|-----------|\n"
        "| Scheinleistung | $S$ | VA (Voltampere) | Gesamtleistung aus Spannung \u00d7 Strom |\n"
        "| Wirkleistung | $P$ | W (Watt) | Nutzbare Leistung (wird in W\u00e4rme/Arbeit umgewandelt) |\n"
        "| Blindleistung | $Q$ | var (Voltampere reaktiv) | Pendelt zwischen Quelle und Bauteil |\n"
        "### Formeln\n"
        "$$S = U \\cdot I \\qquad [VA]$$\n"
        "$$P = U \\cdot I \\cdot \\cos \\varphi \\qquad [W]$$\n"
        "$$Q = U \\cdot I \\cdot \\sin \\varphi \\qquad [var]$$\n"
        "### Leistungsdreieck\n"
        "$$S^2 = P^2 + Q^2 \\qquad S = \\sqrt{P^2 + Q^2}$$\n"
        "### Leistungsfaktor\n"
        "$$\\cos \\varphi = \\frac{P}{S} \\qquad \\varphi = \\arccos\\!\\left(\\frac{P}{S}\\right)$$\n"
        "| Schaltung | \u03c6 | cos \u03c6 | Charakter |\n"
        "|-----------|---|-------|-----------|\n"
        "| Rein ohmsch (R) | 0\u00b0 | 1.0 | Nur Wirkleistung |\n"
        "| Rein induktiv (L) | +90\u00b0 | 0 | Nur Blindleistung (induktiv) |\n"
        "| Rein kapazitiv (C) | \u221290\u00b0 | 0 | Nur Blindleistung (kapazitiv) |\n"
        "> **Wichtig:** Scheinleistung ist **nicht** gleich Wirkleistung!\n"
        "> Ein Motor mit $S = 1000\\,VA$ und $\\cos\\varphi = 0.8$ leistet nur $P = 800\\,W$ Wirkarbeit."))

# ---- Rechner (rechts) ----
# ============================================================
# RECHNER 4.1: Wirkleistung / Blindleistung / Scheinleistung
# ============================================================

ac_U    = make_input('U [V]     =', 'Effektivwert')
ac_I    = make_input('I [A]     =', 'Effektivwert')
ac_phi  = make_input('φ [°]     =', 'Phasenwinkel (+ induktiv, − kapazitiv)')
ac_cosP = make_input('cos φ     =', 'Leistungsfaktor z.B. 0.8')
ac_S    = make_input('S [VA]    =', 'Scheinleistung')
ac_P    = make_input('P [W]     =', 'Wirkleistung')
ac_Q    = make_input('Q [var]   =', 'Blindleistung')
btn_ac  = make_button()
out_ac  = widgets.Output()

def calc_ac(_=None):
    # → Logik in calc.py: ac_leistung()
    out_ac.clear_output()
    with out_ac:
        try:
            r = ac_leistung(U=p(ac_U.value), I=p(ac_I.value), phi_grad=p(ac_phi.value), cos_phi=p(ac_cosP.value), S=p(ac_S.value), P=p(ac_P.value), Q=p(ac_Q.value))
            _u = {'S': 'VA', 'P': 'W', 'Q': 'var', 'cos_phi': '', 'phi_grad': '°', 'U': 'V', 'I': 'A'}
            _skip = ('berechnet', 'ladetabelle', 'stromtabelle', 'widerspruch')
            for k, v in r.items():
                if k in _skip or v is None: continue
                if isinstance(v, bool):
                    if v: print(f'  {k}')
                    continue
                if isinstance(v, list): continue
                u = _u.get(k, '')
                print(f'  {k} = {fmt(v, u) if isinstance(v,(int,float)) else v}')
        except (ValueError, ZeroDivisionError) as e:
            print('i', e)
        except Exception as e:
            print('Fehler:', e)
btn_ac.on_click(calc_ac)
for fw in [ac_U, ac_I, ac_phi, ac_cosP, ac_S, ac_P, ac_Q]:
    fw.observe(calc_ac, names='value')
_calc_widget = (widgets.VBox([ac_U, ac_I, ac_phi, ac_cosP, ac_S, ac_P, ac_Q, btn_ac, out_ac]))

# ---- HBox Layout ----
_left_box = widgets.Box([_theory_out], layout=widgets.Layout(**LEFT_STYLE))
_right_box = widgets.Box([_calc_widget], layout=widgets.Layout(**RIGHT_STYLE))

display(widgets.HBox(
    [_left_box, _right_box],
    layout=widgets.Layout(
        width='100%',
        align_items='flex-start',
        margin='8px 0 28px 0',
    )
))


In [ ]:
display(HTML('<a name="sec-4-2"></a>'))
# ================================================================
# LAYOUT: Theorie (links) | Rechner (rechts)
# ================================================================

# ---- Theorie (links) ----
_theory_out = widgets.Output(layout=THEORY_STYLE)

with _theory_out:
    display(Markdown(
        "---\n"
        "## 4.2 Leistungsfaktor und Blindstromkompensation\n"
        "Ein schlechter Leistungsfaktor ($\\cos\\varphi < 1$) bedeutet unn\u00f6tiger Blindstrom im Netz.\n"
        "Durch einen **Kompensationskondensator** kann der Leistungsfaktor verbessert werden.\n"
        "### Ben\u00f6tigte Kapazit\u00e4t zur Kompensation\n"
        "$$C = \\frac{Q_C}{\\omega \\cdot U^2} = \\frac{P \\cdot (\\tan\\varphi_1 - \\tan\\varphi_2)}{\\omega \\cdot U^2}$$\n"
        "- $\\varphi_1$ = urspr\u00fcnglicher Phasenwinkel (schlechter cos \u03c6)\n"
        "- $\\varphi_2$ = Ziel-Phasenwinkel (besserer cos \u03c6)\n"
        "> **Typisches Ziel in der Praxis:** $\\cos\\varphi \\geq 0.9$"))

# ---- Rechner (rechts) ----
# ============================================================
# RECHNER 4.2: Blindstromkompensation – Kondensator berechnen
# ============================================================

bk_P     = make_input('P [W]        =', 'Wirkleistung der Last')
bk_U     = make_input('U [V]        =', 'Netzspannung (Effektivwert)')
bk_f     = make_input('f [Hz]       =', 'z.B. 50')
bk_cos1  = make_input('cos φ₁      =', 'Ist-Leistungsfaktor z.B. 0.65')
bk_cos2  = make_input('cos φ₂ Ziel =', 'Soll z.B. 0.95')
btn_bk   = make_button()
out_bk   = widgets.Output()

def calc_bk(_=None):
    # → Logik in calc.py: blindstromkompensation()
    out_bk.clear_output()
    with out_bk:
        try:
            r = blindstromkompensation(P=p(bk_P.value), U=p(bk_U.value), f=p(bk_f.value), cos1=p(bk_cos1.value), cos2=p(bk_cos2.value))
            _u = {'C': 'F', 'Q_komp': 'var', 'Q1': 'var', 'Q2': 'var', 'S1': 'VA', 'S2': 'VA', 'I1': 'A', 'I2': 'A', 'strom_reduktion_pct': '%'}
            _skip = ('berechnet', 'ladetabelle', 'stromtabelle', 'widerspruch')
            for k, v in r.items():
                if k in _skip or v is None: continue
                if isinstance(v, bool):
                    if v: print(f'  {k}')
                    continue
                if isinstance(v, list): continue
                u = _u.get(k, '')
                print(f'  {k} = {fmt(v, u) if isinstance(v,(int,float)) else v}')
        except (ValueError, ZeroDivisionError) as e:
            print('i', e)
        except Exception as e:
            print('Fehler:', e)
btn_bk.on_click(calc_bk)
for fw in [bk_P, bk_U, bk_f, bk_cos1, bk_cos2]:
    fw.observe(calc_bk, names='value')
_calc_widget = (widgets.VBox([bk_P, bk_U, bk_f, bk_cos1, bk_cos2, btn_bk, out_bk]))

# ---- HBox Layout ----
_left_box = widgets.Box([_theory_out], layout=widgets.Layout(**LEFT_STYLE))
_right_box = widgets.Box([_calc_widget], layout=widgets.Layout(**RIGHT_STYLE))

display(widgets.HBox(
    [_left_box, _right_box],
    layout=widgets.Layout(
        width='100%',
        align_items='flex-start',
        margin='8px 0 28px 0',
    )
))


In [ ]:
display(HTML('<a name="sec-5-1"></a>'))
# ================================================================
# LAYOUT: Theorie (links) | Rechner (rechts)
# ================================================================

# ---- Theorie (links) ----
_theory_out = widgets.Output(layout=THEORY_STYLE)

with _theory_out:
    display(Markdown(
        "---\n"
        "# Kapitel 5: RLC-Schwingkreise\n"
        "---\n"
        "## 5.1 Grundlagen des Schwingkreises\n"
        "Ein **Schwingkreis** besteht aus einer Spule $L$ und einem Kondensator $C$.\n"
        "Energie pendelt zwischen magnetischem Feld (Spule) und elektrischem Feld (Kondensator).\n"
        "| Gr\u00f6sse | Symbol | Einheit |\n"
        "|--------|--------|---------|\n"
        "| Resonanzfrequenz | $f_0$ | Hz |\n"
        "| Kreisfrequenz | $\\omega_0$ | rad/s |\n"
        "| Induktivit\u00e4t | $L$ | H |\n"
        "| Kapazit\u00e4t | $C$ | F |\n"
        "| Widerstand | $R$ | \u03a9 |\n"
        "### Resonanzfrequenz\n"
        "$$f_0 = \\frac{1}{2\\pi \\cdot \\sqrt{L \\cdot C}} \\qquad \\omega_0 = \\frac{1}{\\sqrt{L \\cdot C}}$$\n"
        "Bei Resonanz gilt: $X_L = X_C$ \u2192 die Blindwiderst\u00e4nde heben sich auf!"))

# ---- Rechner (rechts) ----
# ============================================================
# RECHNER 5.1: Resonanzfrequenz  f0 = 1 / (2π·√(L·C))
# ============================================================

res_f  = make_input('f₀ [Hz] =')
res_L  = make_input('L [H]   =', 'z.B. 10m, 4.7u')
res_C  = make_input('C [F]   =', 'z.B. 100n, 4.7u')
btn_res = make_button()
out_res = widgets.Output()

def calc_res(_=None):
    # → Logik in calc.py: resonanzfrequenz()
    out_res.clear_output()
    with out_res:
        try:
            r = resonanzfrequenz(f0=p(res_f.value), L=p(res_L.value), C=p(res_C.value))
            _u = {'f0': 'Hz', 'L': 'H', 'C': 'F', 'omega0': 'rad/s', 'XL_res': 'Ω'}
            _skip = ('berechnet', 'ladetabelle', 'stromtabelle', 'widerspruch')
            for k, v in r.items():
                if k in _skip or v is None: continue
                if isinstance(v, bool):
                    if v: print(f'  {k}')
                    continue
                if isinstance(v, list): continue
                u = _u.get(k, '')
                print(f'  {k} = {fmt(v, u) if isinstance(v,(int,float)) else v}')
        except (ValueError, ZeroDivisionError) as e:
            print('i', e)
        except Exception as e:
            print('Fehler:', e)
btn_res.on_click(calc_res)
for fw in [res_f, res_L, res_C]:
    fw.observe(calc_res, names='value')
_calc_widget = (widgets.VBox([res_f, res_L, res_C, btn_res, out_res]))

# ---- HBox Layout ----
_left_box = widgets.Box([_theory_out], layout=widgets.Layout(**LEFT_STYLE))
_right_box = widgets.Box([_calc_widget], layout=widgets.Layout(**RIGHT_STYLE))

display(widgets.HBox(
    [_left_box, _right_box],
    layout=widgets.Layout(
        width='100%',
        align_items='flex-start',
        margin='8px 0 28px 0',
    )
))


In [ ]:
display(HTML('<a name="sec-5-2"></a>'))
# ================================================================
# LAYOUT: Theorie (links) | Rechner (rechts)
# ================================================================

# ---- Theorie (links) ----
_theory_out = widgets.Output(layout=THEORY_STYLE)

with _theory_out:
    display(Markdown(
        "---\n"
        "## 5.2 Reihenschwingkreis\n"
        "Im **Reihenschwingkreis** liegen R, L und C in Reihe.\n"
        "$$Z = \\sqrt{R^2 + (X_L - X_C)^2}$$\n"
        "Bei Resonanz: $X_L = X_C$ \u2192 $Z = R$ (minimaler Widerstand \u2192 maximaler Strom!)\n"
        "### G\u00fcte (Qualit\u00e4tsfaktor)\n"
        "$$Q = \\frac{1}{R} \\cdot \\sqrt{\\frac{L}{C}} = \\frac{f_0}{\\Delta f} = \\frac{X_L}{R} = \\frac{X_C}{R}$$\n"
        "- Hohe G\u00fcte $Q$ \u2192 schmales, scharfes Resonanzverhalten\n"
        "- Typische Werte: $Q = 10 \\ldots 200$\n"
        "### Bandbreite\n"
        "Frequenzbereich zwischen den beiden \u22123dB Punkten (halbe Leistung).\n"
        "$$\\Delta f = \\frac{f_0}{Q} \\qquad [Hz]$$\n"
        "### Grenzfrequenzen\n"
        "$$f_u = f_0 - \\frac{\\Delta f}{2} \\qquad f_o = f_0 + \\frac{\\Delta f}{2}$$\n"
        "### Resonanz\u00fcberh\u00f6hung\n"
        "Bei Resonanz sind $U_L$ und $U_C$ jeweils $Q$-mal gr\u00f6sser als $U_{ges}$!\n"
        "$$U_L = U_C = Q \\cdot U_{ges}$$"))

# ---- Rechner (rechts) ----
# ============================================================
# RECHNER 5.2: Reihenschwingkreis – Z, Q, Δf
# ============================================================

sr_R  = make_input('R [Ω]   =')
sr_L  = make_input('L [H]   =', 'z.B. 10m')
sr_C  = make_input('C [F]   =', 'z.B. 100n')
sr_f  = make_input('f [Hz]  =', 'Betriebsfreq. (leer = f₀)')
sr_U  = make_input('U_ges [V]=', 'optional')
btn_sr = make_button()
out_sr = widgets.Output()

def calc_sr(_=None):
    # → Logik in calc.py: reihenschwingkreis()
    out_sr.clear_output()
    with out_sr:
        try:
            r = reihenschwingkreis(R=p(sr_R.value), L=p(sr_L.value), C=p(sr_C.value), U=p(sr_U.value), f_ein=p(sr_f.value))
            _u = {'f0': 'Hz', 'Q': '', 'df': 'Hz', 'Z': 'Ω', 'XL': 'Ω', 'XC': 'Ω', 'f_u': 'Hz', 'f_o': 'Hz', 'I': 'A', 'U_R': 'V', 'U_L': 'V', 'U_C': 'V'}
            _skip = ('berechnet', 'ladetabelle', 'stromtabelle', 'widerspruch')
            for k, v in r.items():
                if k in _skip or v is None: continue
                if isinstance(v, bool):
                    if v: print(f'  {k}')
                    continue
                if isinstance(v, list): continue
                u = _u.get(k, '')
                print(f'  {k} = {fmt(v, u) if isinstance(v,(int,float)) else v}')
        except (ValueError, ZeroDivisionError) as e:
            print('i', e)
        except Exception as e:
            print('Fehler:', e)
btn_sr.on_click(calc_sr)
for fw in [sr_R, sr_L, sr_C, sr_f, sr_U]:
    fw.observe(calc_sr, names='value')
_calc_widget = (widgets.VBox([sr_R, sr_L, sr_C, sr_f, sr_U, btn_sr, out_sr]))

# ---- HBox Layout ----
_left_box = widgets.Box([_theory_out], layout=widgets.Layout(**LEFT_STYLE))
_right_box = widgets.Box([_calc_widget], layout=widgets.Layout(**RIGHT_STYLE))

display(widgets.HBox(
    [_left_box, _right_box],
    layout=widgets.Layout(
        width='100%',
        align_items='flex-start',
        margin='8px 0 28px 0',
    )
))


In [ ]:
display(HTML('<a name="sec-5-3"></a>'))
# ================================================================
# LAYOUT: Theorie (links) | Rechner (rechts)
# ================================================================

# ---- Theorie (links) ----
_theory_out = widgets.Output(layout=THEORY_STYLE)

with _theory_out:
    display(Markdown(
        "---\n"
        "## 5.3 Parallelschwingkreis\n"
        "Im **Parallelschwingkreis** liegen L und C parallel (R ist der Verlustwiderstand der Spule).\n"
        "Bei Resonanz: $Z = Z_{max}$ (maximaler Widerstand \u2192 minimaler Strom aus der Quelle!)\n"
        "### Resonanzfrequenz (mit Verlustwiderstand)\n"
        "$$f_0 = \\frac{1}{2\\pi} \\cdot \\sqrt{\\frac{1}{LC} - \\frac{R^2}{L^2}}$$\n"
        "F\u00fcr $Q \\gg 1$ (verlustarm) vereinfacht sich dies zu:\n"
        "$$f_0 \\approx \\frac{1}{2\\pi\\sqrt{LC}}$$\n"
        "### G\u00fcte\n"
        "$$Q = \\frac{\\omega_0 \\cdot L}{R} = \\frac{X_L}{R}$$\n"
        "### Impedanz bei Resonanz (Parallelwiderstand)\n"
        "$$R_p = Q^2 \\cdot R = \\frac{L}{R \\cdot C}$$\n"
        "### Bandbreite\n"
        "Frequenzbereich zwischen den beiden \u22123dB Punkten (halbe Leistung).\n"
        "$$\\Delta f = \\frac{f_0}{Q}$$"))

# ---- Rechner (rechts) ----
# ============================================================
# RECHNER 5.3: Parallelschwingkreis – f0, Q, Rp, Δf
# ============================================================

par_R  = make_input('R [Ω]   =', 'Verlustwiderstand Spule')
par_L  = make_input('L [H]   =', 'z.B. 10m')
par_C  = make_input('C [F]   =', 'z.B. 100n')
par_U  = make_input('U [V]   =', 'optional')
btn_par = make_button()
out_par = widgets.Output()

def calc_par(_=None):
    out_par.clear_output()
    with out_par:
        try:
            R = p(par_R.value); L = p(par_L.value)
            C = p(par_C.value); U = p(par_U.value)
            if any(x is None for x in [R, L, C]):
                print("R, L und C eingeben"); return

            omega0_ideal = 1 / math.sqrt(L*C)
            # Exakte Formel mit Verlust
            radicand = 1/(L*C) - (R/L)**2
            if radicand <= 0:
                print("R zu gross – kein Schwingen möglich (R ≥ √(L/C))")
                return
            omega0 = math.sqrt(radicand)
            f0     = omega0 / (2*math.pi)
            f0_id  = omega0_ideal / (2*math.pi)
            XL     = omega0_ideal * L
            Q      = XL / R
            Rp     = Q**2 * R
            df     = f0 / Q
            f_u    = f0 - df/2
            f_o    = f0 + df/2

            print(f"--- Parallelschwingkreis ---")
            print(f"f₀ (exakt)  = {fmt(f0, 'Hz')}")
            print(f"f₀ (ideal)  = {fmt(f0_id, 'Hz')}")
            print(f"Q           = {Q:.4g}")
            print(f"R_p (Z_max) = {fmt(Rp, 'Ω')}")
            print(f"Δf          = {fmt(df, 'Hz')}")
            print(f"f_u         = {fmt(f_u, 'Hz')}")
            print(f"f_o         = {fmt(f_o, 'Hz')}")
            if Q < 10:
                print(f"\n   Q = {Q:.2f} < 10 → Näherung f₀ ≈ 1/(2π√LC) ungenau!")
            if U:
                I_ges = U / Rp
                print(f"\n   Bei U = {fmt(U, 'V')}:")
                print(f"   I_ges (Quelle) = {fmt(I_ges, 'A')}  (minimal bei Resonanz)")
                print(f"   I_L = I_C      = {fmt(Q * I_ges, 'A')}  (Kreisström!)")
        except Exception as e:
            print("Fehler:", e)

btn_par.on_click(calc_par)
for fw in [par_R, par_L, par_C, par_U]:
    fw.observe(calc_par, names='value')
_calc_widget = (widgets.VBox([par_R, par_L, par_C, par_U, btn_par, out_par]))

# ---- HBox Layout ----
_left_box = widgets.Box([_theory_out], layout=widgets.Layout(**LEFT_STYLE))
_right_box = widgets.Box([_calc_widget], layout=widgets.Layout(**RIGHT_STYLE))

display(widgets.HBox(
    [_left_box, _right_box],
    layout=widgets.Layout(
        width='100%',
        align_items='flex-start',
        margin='8px 0 28px 0',
    )
))


In [ ]:
display(HTML('<a name="sec-5-4"></a>'))
# ================================================================
# LAYOUT: Theorie (links) | Rechner (rechts)
# ================================================================

# ---- Theorie (links) ----
_theory_out = widgets.Output(layout=THEORY_STYLE)

with _theory_out:
    display(Markdown(
        "---\n"
        "## 5.4 G\u00fcte und Bandbreite \u2013 Vergleich\n"
        "$$Q = \\frac{f_0}{\\Delta f} \\qquad \\Delta f = f_o - f_u$$\n"
        "| G\u00fcte $Q$ | Charakter | Typische Anwendung |\n"
        "|----------|-----------|--------------------|\n"
        "| $Q < 1$ | \u00dcberd\u00e4mpft \u2013 kein Schwingen | D\u00e4mpfungsglieder |\n"
        "| $Q = 1 \\ldots 5$ | Breitbandig | Audio-Filter |\n"
        "| $Q = 10 \\ldots 50$ | Mittel | ZF-Filter, Empf\u00e4nger |\n"
        "| $Q > 100$ | Schmalbandig | Quarzoszillatoren |\n"
        "### Zusammenhang G\u00fcte \u2013 D\u00e4mpfung\n"
        "$$d = \\frac{1}{Q} \\qquad \\text{(D\u00e4mpfungsgrad)}$$\n"
        "| $d$ | $Q$ | Verhalten |\n"
        "|-----|-----|-----------|\n"
        "| $d > 1$ | $Q < 0.5$ | \u00dcberd\u00e4mpft |\n"
        "| $d = 1$ | $Q = 0.5$ | Aperiodischer Grenzfall |\n"
        "| $d < 1$ | $Q > 0.5$ | Unterd\u00e4mpft (schwingt) |"))

# ---- Rechner (rechts) ----
# ============================================================
# RECHNER 5.4: Güte / Bandbreite – Umrechner
# ============================================================

qb_f0 = make_input('f₀ [Hz] =', 'Resonanzfrequenz')
qb_Q  = make_input('Q       =', 'Güte')
qb_df = make_input('Δf [Hz] =', 'Bandbreite')
btn_qb = make_button()
out_qb = widgets.Output()

def calc_qb(_=None):
    # → Logik in calc.py: guete_bandbreite()
    out_qb.clear_output()
    with out_qb:
        try:
            r = guete_bandbreite(f0=p(qb_f0.value), Q=p(qb_Q.value), df=p(qb_df.value))
            _u = {'f0': 'Hz', 'Q': '', 'df': 'Hz', 'f_u': 'Hz', 'f_o': 'Hz'}
            _skip = ('berechnet', 'ladetabelle', 'stromtabelle', 'widerspruch')
            for k, v in r.items():
                if k in _skip or v is None: continue
                if isinstance(v, bool):
                    if v: print(f'  {k}')
                    continue
                if isinstance(v, list): continue
                u = _u.get(k, '')
                print(f'  {k} = {fmt(v, u) if isinstance(v,(int,float)) else v}')
        except (ValueError, ZeroDivisionError) as e:
            print('i', e)
        except Exception as e:
            print('Fehler:', e)
btn_qb.on_click(calc_qb)
for fw in [qb_f0, qb_Q, qb_df]:
    fw.observe(calc_qb, names='value')
_calc_widget = (widgets.VBox([qb_f0, qb_Q, qb_df, btn_qb, out_qb]))

# ---- HBox Layout ----
_left_box = widgets.Box([_theory_out], layout=widgets.Layout(**LEFT_STYLE))
_right_box = widgets.Box([_calc_widget], layout=widgets.Layout(**RIGHT_STYLE))

display(widgets.HBox(
    [_left_box, _right_box],
    layout=widgets.Layout(
        width='100%',
        align_items='flex-start',
        margin='8px 0 28px 0',
    )
))


In [ ]:
display(HTML('<a name="sec-6-1"></a>'))
# ================================================================
# LAYOUT: Theorie (links) | Rechner (rechts)
# ================================================================

# ---- Theorie (links) ----
_theory_out = widgets.Output(layout=THEORY_STYLE)

with _theory_out:
    display(Markdown(
        "---\n"
        "# Kapitel 6: Dezibel (dB)\n"
        "---\n"
        "## 6.1 Grundlagen\n"
        "Das **Dezibel** ist ein logarithmisches Mass f\u00fcr Verh\u00e4ltnisse.\n"
        "Es wird verwendet um grosse Wertebereiche kompakt darzustellen.\n"
        "### Leistungs-Dezibel\n"
        "$$L_P = 10 \\cdot \\lg\\left(\\frac{P_2}{P_1}\\right) \\quad [dB]$$\n"
        "### Spannungs- und Strom-Dezibel\n"
        "$$L_U = 20 \\cdot \\lg\\left(\\frac{U_2}{U_1}\\right) \\quad [dB]$$\n"
        "$$L_I = 20 \\cdot \\lg\\left(\\frac{I_2}{I_1}\\right) \\quad [dB]$$\n"
        "> Der Faktor 20 (statt 10) kommt daher, dass $P \\propto U^2$ bzw. $P \\propto I^2$.\n"
        "### Vorzeichen\n"
        "| Ergebnis | Bedeutung |\n"
        "|----------|-----------|\n"
        "| $> 0\\,dB$ | Verst\u00e4rkung |\n"
        "| $= 0\\,dB$ | Keine \u00c4nderung ($U_2 = U_1$) |\n"
        "| $< 0\\,dB$ | D\u00e4mpfung |"))

# ---- Rechner (rechts) ----
# ============================================================
# RECHNER 6.1: Dezibel Grundrechner  (Spannung / Strom / Leistung)
# ============================================================

db_mode = widgets.ToggleButtons(
    options=['Spannung (20·lg)', 'Strom (20·lg)', 'Leistung (10·lg)'],
    description='Grösse:', style={'description_width': '130px', 'button_width': '140px'}
)
db_x1  = make_input('Eingang (x₁) =', 'z.B. 1, 100m')
db_x2  = make_input('Ausgang (x₂) =', 'leer = aus dB berechnen')
db_dB  = make_input('dB           =', 'leer = berechnen')
btn_db = make_button()
out_db = widgets.Output()

def calc_db(_=None):
    out_db.clear_output()
    with out_db:
        try:
            x1 = p(db_x1.value); x2 = p(db_x2.value); dB = p(db_dB.value)
            factor = 10 if db_mode.value == 'Leistung (10·lg)' else 20

            if x1 is not None and x2 is not None:
                res = factor * math.log10(x2 / x1)
                unit = 'V' if 'Spannung' in db_mode.value else ('A' if 'Strom' in db_mode.value else 'W')
                print(f"  dB  = {res:.4g} dB")
                if res > 0:   print(f"  → Verstärkung um {res:.4g} dB")
                elif res < 0: print(f"  → Dämpfung um {abs(res):.4g} dB")
                else:         print(f"  → Kein Unterschied (x₁ = x₂)")

            elif x1 is not None and dB is not None:
                res = x1 * 10**(dB / factor)
                unit = 'V' if 'Spannung' in db_mode.value else ('A' if 'Strom' in db_mode.value else 'W')
                print(f"  x₂  = {fmt(res, unit)}")

            elif x2 is not None and dB is not None:
                res = x2 / 10**(dB / factor)
                unit = 'V' if 'Spannung' in db_mode.value else ('A' if 'Strom' in db_mode.value else 'W')
                print(f"  x₁  = {fmt(res, unit)}")

            else:
                print("Zwei Felder ausfüllen (x₁+x₂, x₁+dB, oder x₂+dB)")
        except Exception as e:
            print("Fehler:", e)

btn_db.on_click(calc_db)
db_mode.observe(calc_db, names='value')
for fw in [db_x1, db_x2, db_dB]:
    fw.observe(calc_db, names='value')
_calc_widget = (widgets.VBox([db_mode, db_x1, db_x2, db_dB, btn_db, out_db]))

# ---- HBox Layout ----
_left_box = widgets.Box([_theory_out], layout=widgets.Layout(**LEFT_STYLE))
_right_box = widgets.Box([_calc_widget], layout=widgets.Layout(**RIGHT_STYLE))

display(widgets.HBox(
    [_left_box, _right_box],
    layout=widgets.Layout(
        width='100%',
        align_items='flex-start',
        margin='8px 0 28px 0',
    )
))


In [ ]:
display(HTML('<a name="sec-6-2"></a>'))
# ================================================================
# LAYOUT: Theorie (links) | Rechner (rechts)
# ================================================================

# ---- Theorie (links) ----
_theory_out = widgets.Output(layout=THEORY_STYLE)

with _theory_out:
    display(Markdown(
        "---\n"
        "## 6.2 Wichtige dB-Werte auswendig lernen\n"
        "Diese Werte sollte man im Schlaf kennen:\n"
        "### Spannung / Strom (20\u00b7lg)\n"
        "| dB | Verh\u00e4ltnis $U_2/U_1$ | Merkhilfe |\n"
        "|----|----------------------|-----------|\n"
        "| +20 dB | 10 | Zehnfache Spannung |\n"
        "| +6 dB | $\\approx 2$ | Doppelte Spannung |\n"
        "| +3 dB | $\\approx 1.414 = \\sqrt{2}$ | Faktor $\\sqrt{2}$ |\n"
        "| 0 dB | 1 | Keine \u00c4nderung |\n"
        "| \u22123 dB | $\\approx 0.707 = 1/\\sqrt{2}$ | Grenzfrequenz! |\n"
        "| \u22126 dB | $\\approx 0.5$ | Halbe Spannung |\n"
        "| \u221220 dB | 0.1 | Zehntel Spannung |\n"
        "| \u221240 dB | 0.01 | Hundertstel Spannung |\n"
        "### Leistung (10\u00b7lg)\n"
        "| dB | Verh\u00e4ltnis $P_2/P_1$ | Merkhilfe |\n"
        "|----|----------------------|-----------|\n"
        "| +10 dB | 10 | Zehnfache Leistung |\n"
        "| +3 dB | $\\approx 2$ | Doppelte Leistung |\n"
        "| 0 dB | 1 | Keine \u00c4nderung |\n"
        "| \u22123 dB | $\\approx 0.5$ | Halbe Leistung |\n"
        "| \u221210 dB | 0.1 | Zehntel Leistung |\n"
        "> Der **\u22123 dB Punkt** ist die **Grenzfrequenz** bei Filtern!\n"
        "> Dort ist $U = U_0 / \\sqrt{2}$ und $P = P_0 / 2$."))

# ---- Rechner (rechts) ----
# ============================================================
# RECHNER 6.2: dB Merktabelle – Schnellreferenz
# ============================================================

out_tab = widgets.Output()
with out_tab:
    print("Spannung / Strom (20·lg):")
    print(f"  {'dB':>7}  {'Verhältnis':>12}  {'Kehrwert':>12}")
    print("  " + "-"*36)
    for db in [40, 20, 14, 6, 3, 0, -3, -6, -14, -20, -40]:
        ratio = 10**(db/20)
        print(f"  {db:>+7.0f} dB  {ratio:>12.5g}  {1/ratio:>12.5g}")
    print()
    print("Leistung (10·lg):")
    print(f"  {'dB':>7}  {'Verhältnis':>12}  {'Kehrwert':>12}")
    print("  " + "-"*36)
    for db in [20, 10, 3, 0, -3, -10, -20]:
        ratio = 10**(db/10)
        print(f"  {db:>+7.0f} dB  {ratio:>12.5g}  {1/ratio:>12.5g}")
_calc_widget = (out_tab)

# ---- HBox Layout ----
_left_box = widgets.Box([_theory_out], layout=widgets.Layout(**LEFT_STYLE))
_right_box = widgets.Box([_calc_widget], layout=widgets.Layout(**RIGHT_STYLE))

display(widgets.HBox(
    [_left_box, _right_box],
    layout=widgets.Layout(
        width='100%',
        align_items='flex-start',
        margin='8px 0 28px 0',
    )
))


In [ ]:
display(HTML('<a name="sec-6-3"></a>'))
# ================================================================
# LAYOUT: Theorie (links) | Rechner (rechts)
# ================================================================

# ---- Theorie (links) ----
_theory_out = widgets.Output(layout=THEORY_STYLE)

with _theory_out:
    display(Markdown(
        "---\n"
        "## 6.3 Kettenrechnung \u2013 dB addieren\n"
        "Der grosse Vorteil von dB: **Verst\u00e4rkungen und D\u00e4mpfungen werden addiert statt multipliziert.**\n"
        "$$L_{ges} = L_1 + L_2 + L_3 + \\ldots \\quad [dB]$$\n"
        "**Beispiel:** Verst\u00e4rker +26 dB \u2192 Kabel \u22123 dB \u2192 Filter \u22126 dB\n"
        "$$L_{ges} = +26 - 3 - 6 = +17 \\text{ dB}$$\n"
        "### Absoluter Bezugspegel\n"
        "| Einheit | Bezug | Verwendung |\n"
        "|---------|-------|------------|\n"
        "| dBm | 1 mW an 50 \u03a9 | HF-Technik |\n"
        "| dBV | 1 V | Audiotechnik |\n"
        "| dB\u00b5V | 1 \u00b5V | Antennentechnik |"))

# ---- Rechner (rechts) ----
# ============================================================
# RECHNER 6.3: Kettendämpfung – bis 6 Stufen
# ============================================================

kette_fields = [make_input(f'Stufe {i+1} [dB] =', 'z.B. +20, -3, -6') for i in range(6)]
kette_U_ein  = make_input('U_ein [V]    =', 'optional → U_aus berechnen')
btn_kette    = make_button()
out_kette    = widgets.Output()

def calc_kette(_=None):
    out_kette.clear_output()
    with out_kette:
        try:
            dbs   = [p(f.value) for f in kette_fields]
            dbs_v = [d for d in dbs if d is not None]
            U_ein = p(kette_U_ein.value)
            if len(dbs_v) < 1:
                print("Mindestens eine Stufe eingeben"); return

            total = sum(dbs_v)
            print(f"  Stufen:   {' + '.join(f'{d:+.4g}' for d in dbs_v)} dB")
            print(f"  Gesamt:   {total:+.4g} dB")
            ratio_U = 10**(total/20)
            ratio_P = 10**(total/10)
            print(f"  U-Faktor: {ratio_U:.5g}  (20·lg)")
            print(f"  P-Faktor: {ratio_P:.5g}  (10·lg)")
            if total > 0:   print(f"  → Gesamtverstärkung")
            elif total < 0: print(f"  → Gesamtdämpfung")
            else:           print(f"  → Keine Gesamtänderung")
            if U_ein:
                U_aus = U_ein * ratio_U
                print(f"\n  U_ein = {fmt(U_ein, 'V')}")
                print(f"  U_aus = {fmt(U_aus, 'V')}")
        except Exception as e:
            print("Fehler:", e)

btn_kette.on_click(calc_kette)
for fw in kette_fields + [kette_U_ein]:
    fw.observe(calc_kette, names='value')
_calc_widget = (widgets.VBox(kette_fields + [kette_U_ein, btn_kette, out_kette]))

# ---- HBox Layout ----
_left_box = widgets.Box([_theory_out], layout=widgets.Layout(**LEFT_STYLE))
_right_box = widgets.Box([_calc_widget], layout=widgets.Layout(**RIGHT_STYLE))

display(widgets.HBox(
    [_left_box, _right_box],
    layout=widgets.Layout(
        width='100%',
        align_items='flex-start',
        margin='8px 0 28px 0',
    )
))


In [ ]:
display(HTML('<a name="sec-7-1"></a>'))
# ================================================================
# LAYOUT: Theorie (links) | Rechner (rechts)
# ================================================================

# ---- Theorie (links) ----
_theory_out = widgets.Output(layout=THEORY_STYLE)

with _theory_out:
    display(Markdown(
        "---\n"
        "# Kapitel 7: Filter\n"
        "---\n\n"
        "Ein Filter l\u00e4sst bestimmte Frequenzen durch und d\u00e4mpft andere.\n"
        "Die Grenzfrequenz $f_g$ liegt dort wo die Spannung auf $1/\\sqrt{2} \\approx 0.707$ gefallen ist (\u22123 dB Punkt).\n\n"
        "---\n"
        "## 7.1 Passiver RC-Tiefpass (1. Ordnung)\n"
        "Ein einfacher Tiefpass l\u00e4sst tiefe Frequenzen durch und d\u00e4mpft hohe.\n"
        "$$f_g = \\frac{1}{2\\pi \\cdot R \\cdot C}$$\n"
        "**\u00dcbertragungsfunktion (Betrag):**\n"
        "$$\\left|\\underline{H}\\right| = \\frac{1}{\\sqrt{1 + \\left(\\frac{f}{f_g}\\right)^2}}$$\n"
        "**Phasengang:**\n"
        "$$\\varphi = -\\arctan\\!\\left(\\frac{f}{f_g}\\right)$$\n"
        "**Flankensteilheit:** \u221220 dB/Dekade (1. Ordnung)\n"
        "<img src=\"Bilder/rc_tiefpass_schaltung.svg\" width=\"300\">\n"
))

# ---- Rechner (rechts) ----
# ============================================================
# RECHNER 7.1: Passiver RC-Tiefpass
# ============================================================

tp_R  = make_input('R [Ω]   =')
tp_C  = make_input('C [F]   =', 'z.B. 100n')
tp_f  = make_input('f [Hz]  =', 'optional: Verhalten bei f')
tp_U0 = make_input('U_ein [V]=', 'optional')
btn_tp = make_button()
out_tp = widgets.Output()

def calc_tp(_=None):
    out_tp.clear_output()
    with out_tp:
        try:
            R = p(tp_R.value); C = p(tp_C.value)
            f = p(tp_f.value); U0 = p(tp_U0.value)
            if any(x is None for x in [R, C]):
                print("R und C eingeben"); return
            fg = 1 / (2 * math.pi * R * C)
            print(f"  f_g   = {fmt(fg, 'Hz')}")
            print(f"  τ     = R·C = {fmt(R*C, 's')}")
            if f:
                H     = 1 / math.sqrt(1 + (f/fg)**2)
                phi   = -math.degrees(math.atan(f/fg))
                dB_val = 20 * math.log10(H)
                print(f"\n  Bei f = {fmt(f, 'Hz')}:")
                print(f"  |H|   = {H:.5g}  ({dB_val:.3g} dB)")
                print(f"  φ     = {phi:.3g}°")
                if U0:
                    print(f"  U_aus = {fmt(H * U0, 'V')}")
                if f < fg:   print(f"  → Durchlassbereich (f < f_g)")
                elif f == fg: print(f"  → Genau bei Grenzfrequenz (−3 dB)")
                else:         print(f"  → Sperrbereich (f > f_g)")
        except Exception as e:
            print("Fehler:", e)

btn_tp.on_click(calc_tp)
for fw in [tp_R, tp_C, tp_f, tp_U0]:
    fw.observe(calc_tp, names='value')
_calc_widget = (widgets.VBox([tp_R, tp_C, tp_f, tp_U0, btn_tp, out_tp]))

# ---- HBox Layout ----
_left_box = widgets.Box([_theory_out], layout=widgets.Layout(**LEFT_STYLE))
_right_box = widgets.Box([_calc_widget], layout=widgets.Layout(**RIGHT_STYLE))

display(widgets.HBox(
    [_left_box, _right_box],
    layout=widgets.Layout(
        width='100%',
        align_items='flex-start',
        margin='8px 0 28px 0',
    )
))


In [ ]:
display(HTML('<a name="sec-7-2"></a>'))
# ================================================================
# LAYOUT: Theorie (links) | Rechner (rechts)
# ================================================================

# ---- Theorie (links) ----
_theory_out = widgets.Output(layout=THEORY_STYLE)

with _theory_out:
    display(Markdown(
        "---\n"
        "## 7.2 Passiver RC-Hochpass (1. Ordnung)\n"
        "Ein Hochpass l\u00e4sst hohe Frequenzen durch und d\u00e4mpft tiefe.\n"
        "$$f_g = \\frac{1}{2\\pi \\cdot R \\cdot C}$$\n"
        "**\u00dcbertragungsfunktion:**\n"
        "$$\\left|\\underline{H}\\right| = \\frac{f/f_g}{\\sqrt{1 + \\left(\\frac{f}{f_g}\\right)^2}}$$\n"
        "**Phasengang:**\n"
        "$$\\varphi = +\\arctan\\!\\left(\\frac{f_g}{f}\\right)$$\n"
        "**Flankensteilheit:** +20 dB/Dekade unterhalb $f_g$\n"
        "<img src=\"Bilder/rc_hochpass_schaltung.svg\" width=\"300\">\n"
))

# ---- Rechner (rechts) ----
# ============================================================
# RECHNER 7.2: Passiver RC-Hochpass
# ============================================================

hp_R  = make_input('R [Ω]   =')
hp_C  = make_input('C [F]   =', 'z.B. 100n')
hp_f  = make_input('f [Hz]  =', 'optional: Verhalten bei f')
hp_U0 = make_input('U_ein [V]=', 'optional')
btn_hp = make_button()
out_hp = widgets.Output()

def calc_hp(_=None):
    out_hp.clear_output()
    with out_hp:
        try:
            R = p(hp_R.value); C = p(hp_C.value)
            f = p(hp_f.value); U0 = p(hp_U0.value)
            if any(x is None for x in [R, C]):
                print("R und C eingeben"); return
            fg = 1 / (2 * math.pi * R * C)
            print(f"  f_g   = {fmt(fg, 'Hz')}")
            print(f"  τ     = R·C = {fmt(R*C, 's')}")
            if f:
                ratio = f / fg
                H     = ratio / math.sqrt(1 + ratio**2)
                phi   = math.degrees(math.atan(fg / f))
                dB_val = 20 * math.log10(H)
                print(f"\n  Bei f = {fmt(f, 'Hz')}:")
                print(f"  |H|   = {H:.5g}  ({dB_val:.3g} dB)")
                print(f"  φ     = +{phi:.3g}°")
                if U0:
                    print(f"  U_aus = {fmt(H * U0, 'V')}")
                if f > fg:   print(f"  → Durchlassbereich (f > f_g)")
                elif f == fg: print(f"  → Genau bei Grenzfrequenz (−3 dB)")
                else:         print(f"  → Sperrbereich (f < f_g)")
        except Exception as e:
            print("Fehler:", e)

btn_hp.on_click(calc_hp)
for fw in [hp_R, hp_C, hp_f, hp_U0]:
    fw.observe(calc_hp, names='value')
_calc_widget = (widgets.VBox([hp_R, hp_C, hp_f, hp_U0, btn_hp, out_hp]))

# ---- HBox Layout ----
_left_box = widgets.Box([_theory_out], layout=widgets.Layout(**LEFT_STYLE))
_right_box = widgets.Box([_calc_widget], layout=widgets.Layout(**RIGHT_STYLE))

display(widgets.HBox(
    [_left_box, _right_box],
    layout=widgets.Layout(
        width='100%',
        align_items='flex-start',
        margin='8px 0 28px 0',
    )
))


In [ ]:
display(HTML('<a name="sec-7-3"></a>'))
# ================================================================
# LAYOUT: Theorie (links) | Rechner (rechts)
# ================================================================

# ---- Theorie (links) ----
_theory_out = widgets.Output(layout=THEORY_STYLE)

with _theory_out:
    display(Markdown(
        "---\n"
        "## 7.3 Passiver RL-Tiefpass und Hochpass\n"
        "Gleiche Logik wie RC \u2013 aber mit Spule statt Kondensator.\n"
        "| | RL-Tiefpass | RL-Hochpass |\n"
        "|---|---|---|\n"
        "| Ausgang an | R | L |\n"
        "| Durchlass | Tiefe f | Hohe f |\n"
        "| Grenzfrequenz | $f_g = R / (2\\pi L)$ | $f_g = R / (2\\pi L)$ |\n"
        "$$f_g = \\frac{R}{2\\pi \\cdot L}$$\n"
        "<!-- Bild: RL Tiefpass und Hochpass -->\n"
        "<!-- Dateiname: rl_filter_schaltung.png  \u2192 Google: \"RL lowpass highpass filter circuit diagram\" -->"))

# ---- Rechner (rechts) ----
# ============================================================
# RECHNER 7.3: Passiver RL-Filter (Tief- und Hochpass)
# ============================================================

rl_filt_mode = widgets.ToggleButtons(
    options=['Tiefpass (Ausgang an R)', 'Hochpass (Ausgang an L)'],
    description='Typ:', style={'description_width': '130px', 'button_width': '200px'}
)
rlf_R  = make_input('R [Ω]   =')
rlf_L  = make_input('L [H]   =', 'z.B. 10m')
rlf_f  = make_input('f [Hz]  =', 'optional: Verhalten bei f')
rlf_U0 = make_input('U_ein [V]=', 'optional')
btn_rlf = make_button()
out_rlf = widgets.Output()

def calc_rlf(_=None):
    out_rlf.clear_output()
    with out_rlf:
        try:
            R = p(rlf_R.value); L = p(rlf_L.value)
            f = p(rlf_f.value); U0 = p(rlf_U0.value)
            if any(x is None for x in [R, L]):
                print("R und L eingeben"); return
            fg = R / (2 * math.pi * L)
            print(f"  f_g   = {fmt(fg, 'Hz')}")
            if f:
                ratio = f / fg
                if 'Tiefpass' in rl_filt_mode.value:
                    H   = 1 / math.sqrt(1 + ratio**2)
                    phi = -math.degrees(math.atan(ratio))
                else:
                    H   = ratio / math.sqrt(1 + ratio**2)
                    phi = math.degrees(math.atan(fg / f))
                dB_val = 20 * math.log10(H)
                print(f"\n  Bei f = {fmt(f, 'Hz')}:")
                print(f"  |H|   = {H:.5g}  ({dB_val:.3g} dB)")
                print(f"  φ     = {phi:.3g}°")
                if U0:
                    print(f"  U_aus = {fmt(H * U0, 'V')}")
        except Exception as e:
            print("Fehler:", e)

btn_rlf.on_click(calc_rlf)
rl_filt_mode.observe(calc_rlf, names='value')
for fw in [rlf_R, rlf_L, rlf_f, rlf_U0]:
    fw.observe(calc_rlf, names='value')
_calc_widget = (widgets.VBox([rl_filt_mode, rlf_R, rlf_L, rlf_f, rlf_U0, btn_rlf, out_rlf]))

# ---- HBox Layout ----
_left_box = widgets.Box([_theory_out], layout=widgets.Layout(**LEFT_STYLE))
_right_box = widgets.Box([_calc_widget], layout=widgets.Layout(**RIGHT_STYLE))

display(widgets.HBox(
    [_left_box, _right_box],
    layout=widgets.Layout(
        width='100%',
        align_items='flex-start',
        margin='8px 0 28px 0',
    )
))


In [ ]:
display(HTML('<a name="sec-7-4"></a>'))
# ================================================================
# LAYOUT: Theorie (links) | Rechner (rechts)
# ================================================================

# ---- Theorie (links) ----
_theory_out = widgets.Output(layout=THEORY_STYLE)

with _theory_out:
    display(Markdown(
        "---\n"
        "## 7.4 Aktiver Tiefpass mit OPV (Sallen-Key, 1. Ordnung)\n"
        "Der OPV puffert den Ausgang \u2013 die Last beeinflusst den Filter nicht mehr.\n"
        "Mit Verst\u00e4rkung $A_0 = 1 + R_2/R_1$.\n"
        "$$f_g = \\frac{1}{2\\pi \\cdot R \\cdot C}$$\n"
        "$$A_0 = 1 + \\frac{R_f}{R_1}$$\n"
        "**Gesamt\u00fcbertragung:**\n"
        "$$\\left|\\underline{H}\\right| = \\frac{A_0}{\\sqrt{1 + \\left(\\frac{f}{f_g}\\right)^2}}$$\n"
        "<!-- Bild: OPV Tiefpass 1. Ordnung nicht-invertierend -->\n"
        "<!-- Dateiname: opv_tiefpass_1ordnung.png  \u2192 Google: \"active low pass filter op-amp non-inverting first order circuit\" -->"))

# ---- Rechner (rechts) ----
# ============================================================
# RECHNER 7.4: Aktiver OPV-Tiefpass 1. Ordnung
# ============================================================

atp_R  = make_input('R [Ω]   =', 'Filterwiederstand')
atp_C  = make_input('C [F]   =', 'z.B. 100n')
atp_Rf = make_input('R_f [Ω] =', 'Gegenkopplungswid.')
atp_R1 = make_input('R₁ [Ω]  =', 'leer → A₀ = 1 (Spannungsfolger)')
atp_f  = make_input('f [Hz]  =', 'optional: Verhalten bei f')
atp_U0 = make_input('U_ein [V]=', 'optional')
btn_atp = make_button()
out_atp = widgets.Output()

def calc_atp(_=None):
    out_atp.clear_output()
    with out_atp:
        try:
            R  = p(atp_R.value);  C  = p(atp_C.value)
            Rf = p(atp_Rf.value); R1 = p(atp_R1.value)
            f  = p(atp_f.value);  U0 = p(atp_U0.value)
            if any(x is None for x in [R, C]):
                print("R und C eingeben"); return
            fg = 1 / (2 * math.pi * R * C)
            if Rf and R1:   A0 = 1 + Rf / R1
            elif not Rf and not R1: A0 = 1.0
            else:
                print("Entweder beide (R_f und R₁) oder keinen eingeben"); return
            print(f"  f_g   = {fmt(fg, 'Hz')}")
            print(f"  A₀    = {A0:.5g}  ({fmt(A0, '')} = {20*math.log10(A0):.3g} dB)")
            if f:
                H     = A0 / math.sqrt(1 + (f/fg)**2)
                phi   = -math.degrees(math.atan(f/fg))
                dB_val = 20 * math.log10(H)
                print(f"\n  Bei f = {fmt(f, 'Hz')}:")
                print(f"  |H|   = {H:.5g}  ({dB_val:.3g} dB)")
                print(f"  φ     = {phi:.3g}°")
                if U0:
                    print(f"  U_aus = {fmt(H * U0, 'V')}")
        except Exception as e:
            print("Fehler:", e)

btn_atp.on_click(calc_atp)
for fw in [atp_R, atp_C, atp_Rf, atp_R1, atp_f, atp_U0]:
    fw.observe(calc_atp, names='value')
_calc_widget = (widgets.VBox([atp_R, atp_C, atp_Rf, atp_R1, atp_f, atp_U0, btn_atp, out_atp]))

# ---- HBox Layout ----
_left_box = widgets.Box([_theory_out], layout=widgets.Layout(**LEFT_STYLE))
_right_box = widgets.Box([_calc_widget], layout=widgets.Layout(**RIGHT_STYLE))

display(widgets.HBox(
    [_left_box, _right_box],
    layout=widgets.Layout(
        width='100%',
        align_items='flex-start',
        margin='8px 0 28px 0',
    )
))


In [ ]:
display(HTML('<a name="sec-7-5"></a>'))
# ================================================================
# LAYOUT: Theorie (links) | Rechner (rechts)
# ================================================================

# ---- Theorie (links) ----
_theory_out = widgets.Output(layout=THEORY_STYLE)

with _theory_out:
    display(Markdown(
        "---\n"
        "## 7.5 Aktiver Hochpass mit OPV (1. Ordnung)\n"
        "Gleiche Struktur wie aktiver Tiefpass, R und C getauscht.\n"
        "$$f_g = \\frac{1}{2\\pi \\cdot R \\cdot C} \\qquad A_0 = 1 + \\frac{R_f}{R_1}$$\n"
        "$$\\left|\\underline{H}\\right| = \\frac{A_0 \\cdot (f/f_g)}{\\sqrt{1 + \\left(\\frac{f}{f_g}\\right)^2}}$$\n"
        "<!-- Bild: OPV Hochpass 1. Ordnung -->\n"
        "<!-- Dateiname: opv_hochpass_1ordnung.png  \u2192 Google: \"active high pass filter op-amp first order circuit diagram\" -->"))

# ---- Rechner (rechts) ----
# ============================================================
# RECHNER 7.5: Aktiver OPV-Hochpass 1. Ordnung
# ============================================================

ahp_R  = make_input('R [Ω]   =')
ahp_C  = make_input('C [F]   =', 'z.B. 100n')
ahp_Rf = make_input('R_f [Ω] =', 'Gegenkopplungswid.')
ahp_R1 = make_input('R₁ [Ω]  =', 'leer → A₀ = 1')
ahp_f  = make_input('f [Hz]  =', 'optional: Verhalten bei f')
ahp_U0 = make_input('U_ein [V]=', 'optional')
btn_ahp = make_button()
out_ahp = widgets.Output()

def calc_ahp(_=None):
    out_ahp.clear_output()
    with out_ahp:
        try:
            R  = p(ahp_R.value);  C  = p(ahp_C.value)
            Rf = p(ahp_Rf.value); R1 = p(ahp_R1.value)
            f  = p(ahp_f.value);  U0 = p(ahp_U0.value)
            if any(x is None for x in [R, C]):
                print("R und C eingeben"); return
            fg = 1 / (2 * math.pi * R * C)
            if Rf and R1:           A0 = 1 + Rf / R1
            elif not Rf and not R1: A0 = 1.0
            else:
                print("Entweder beide (R_f und R₁) oder keinen eingeben"); return
            print(f"  f_g   = {fmt(fg, 'Hz')}")
            print(f"  A₀    = {A0:.5g}  ({20*math.log10(A0):.3g} dB)")
            if f:
                ratio  = f / fg
                H      = A0 * ratio / math.sqrt(1 + ratio**2)
                phi    = math.degrees(math.atan(fg / f))
                dB_val = 20 * math.log10(H)
                print(f"\n  Bei f = {fmt(f, 'Hz')}:")
                print(f"  |H|   = {H:.5g}  ({dB_val:.3g} dB)")
                print(f"  φ     = +{phi:.3g}°")
                if U0:
                    print(f"  U_aus = {fmt(H * U0, 'V')}")
        except Exception as e:
            print("Fehler:", e)

btn_ahp.on_click(calc_ahp)
for fw in [ahp_R, ahp_C, ahp_Rf, ahp_R1, ahp_f, ahp_U0]:
    fw.observe(calc_ahp, names='value')
_calc_widget = (widgets.VBox([ahp_R, ahp_C, ahp_Rf, ahp_R1, ahp_f, ahp_U0, btn_ahp, out_ahp]))

# ---- HBox Layout ----
_left_box = widgets.Box([_theory_out], layout=widgets.Layout(**LEFT_STYLE))
_right_box = widgets.Box([_calc_widget], layout=widgets.Layout(**RIGHT_STYLE))

display(widgets.HBox(
    [_left_box, _right_box],
    layout=widgets.Layout(
        width='100%',
        align_items='flex-start',
        margin='8px 0 28px 0',
    )
))


In [ ]:
display(HTML('<a name="sec-7-6"></a>'))
# ================================================================
# LAYOUT: Theorie (links) | Rechner (rechts)
# ================================================================

# ---- Theorie (links) ----
_theory_out = widgets.Output(layout=THEORY_STYLE)

with _theory_out:
    display(Markdown(
        "---\n"
        "## 7.6 Aktiver Bandpass mit OPV (2. Ordnung)\n"
        "L\u00e4sst einen Frequenzbereich um $f_0$ durch, d\u00e4mpft darunter und dar\u00fcber.\n"
        "$$f_0 = \\frac{1}{2\\pi \\cdot \\sqrt{R_1 \\cdot R_2 \\cdot C_1 \\cdot C_2}}$$\n"
        "Vereinfacht mit $R_1 = R_2 = R$ und $C_1 = C_2 = C$:\n"
        "$$f_0 = \\frac{1}{2\\pi \\cdot R \\cdot C}$$\n"
        "**G\u00fcte und Bandbreite:**\n"
        "$$Q = \\frac{f_0}{\\Delta f} \\qquad \\Delta f = \\frac{f_0}{Q}$$\n"
        "**Mittenverst\u00e4rkung** (bei $f_0$):\n"
        "$$A_0 = \\frac{R_2}{2 \\cdot R_1}$$\n"
        "**Flankensteilheit:** +20 dB/Dek. unterhalb, \u221220 dB/Dek. oberhalb $f_0$\n"
        "<!-- Bild: OPV Bandpass MFB (Multiple Feedback) -->\n"
        "<!-- Dateiname: opv_bandpass_mfb.png  \u2192 Google: \"active bandpass filter op-amp multiple feedback MFB circuit\" -->"))

# ---- Rechner (rechts) ----
# ============================================================
# RECHNER 7.6: Aktiver OPV-Bandpass (vereinfacht R1=R2, C1=C2)
# ============================================================

bp_R  = make_input('R [Ω]   =', 'R₁ = R₂')
bp_C  = make_input('C [F]   =', 'C₁ = C₂, z.B. 100n')
bp_Q  = make_input('Q       =', 'Güte, z.B. 5')
bp_f  = make_input('f [Hz]  =', 'optional: Verhalten bei f')
bp_U0 = make_input('U_ein [V]=', 'optional')
btn_bp = make_button()
out_bp = widgets.Output()

def calc_bp(_=None):
    out_bp.clear_output()
    with out_bp:
        try:
            R  = p(bp_R.value); C  = p(bp_C.value)
            Q  = p(bp_Q.value); f  = p(bp_f.value)
            U0 = p(bp_U0.value)
            if any(x is None for x in [R, C]):
                print("R und C eingeben"); return
            f0 = 1 / (2 * math.pi * R * C)
            Qv = Q if Q else 1/math.sqrt(2)
            df = f0 / Qv
            fu = f0 - df/2
            fo = f0 + df/2
            print(f"  f₀    = {fmt(f0, 'Hz')}")
            print(f"  Q     = {Qv:.4g}")
            print(f"  Δf    = {fmt(df, 'Hz')}")
            print(f"  f_u   = {fmt(fu, 'Hz')}")
            print(f"  f_o   = {fmt(fo, 'Hz')}")
            if f:
                # Bandpass Übertragungsfunktion 2. Ordnung
                norm  = f / f0
                H     = (1/Qv * norm) / math.sqrt((1 - norm**2)**2 + (norm/Qv)**2)
                dB_val = 20 * math.log10(H) if H > 0 else -99
                print(f"\n  Bei f = {fmt(f, 'Hz')}:")
                print(f"  |H|   = {H:.5g}  ({dB_val:.3g} dB)")
                if U0:
                    print(f"  U_aus = {fmt(H * U0, 'V')}")
                if abs(f - f0) < df/2:
                    print(f"  → Im Durchlassbereich")
                else:
                    print(f"  → Im Sperrbereich")
        except Exception as e:
            print("Fehler:", e)

btn_bp.on_click(calc_bp)
for fw in [bp_R, bp_C, bp_Q, bp_f, bp_U0]:
    fw.observe(calc_bp, names='value')
_calc_widget = (widgets.VBox([bp_R, bp_C, bp_Q, bp_f, bp_U0, btn_bp, out_bp]))

# ---- HBox Layout ----
_left_box = widgets.Box([_theory_out], layout=widgets.Layout(**LEFT_STYLE))
_right_box = widgets.Box([_calc_widget], layout=widgets.Layout(**RIGHT_STYLE))

display(widgets.HBox(
    [_left_box, _right_box],
    layout=widgets.Layout(
        width='100%',
        align_items='flex-start',
        margin='8px 0 28px 0',
    )
))


In [ ]:
display(HTML('<a name="sec-7-7"></a>'))
# ================================================================
# LAYOUT: Theorie (links) | Rechner (rechts)
# ================================================================

# ---- Theorie (links) ----
_theory_out = widgets.Output(layout=THEORY_STYLE)

with _theory_out:
    display(Markdown(
        "---\n"
        "## 7.7 Sallen-Key Filter (2. Ordnung)\n"
        "Der **Sallen-Key** ist die h\u00e4ufigste Topologie f\u00fcr aktive Filter 2. Ordnung.\n"
        "Er realisiert Tief- oder Hochpass mit w\u00e4hlbarer Charakteristik.\n"
        "### Tiefpass Sallen-Key (R\u2081=R\u2082=R, C\u2081=C\u2082=C)\n"
        "$$f_g = \\frac{1}{2\\pi \\cdot R \\cdot C}$$\n"
        "$$Q = \\frac{1}{3 - A_0} \\qquad A_0 = 1 + \\frac{R_f}{R_1}$$\n"
        "**Wichtige Charakteristiken:**\n"
        "| Charakteristik | Q | A\u2080 | Eigenschaft |\n"
        "|---|---|---|---|\n"
        "| Butterworth | 0.707 | 1.586 | Maximale Flachheit im Durchlass |\n"
        "| Bessel | 0.577 | 1.268 | Bestes Phasenverhalten |\n"
        "| Chebyshev 1 dB | 0.956 | 2.234 | Steilste Flanke |\n"
        "**Flankensteilheit:** \u221240 dB/Dekade (2. Ordnung = doppelt so steil wie 1. Ordnung!)\n"
        "<!-- Bild: Sallen-Key Tiefpass Schaltung -->\n"
        "<!-- Dateiname: sallen_key_tiefpass.png  \u2192 Google: \"Sallen-Key low pass filter circuit second order op-amp\" -->"))

# ---- Rechner (rechts) ----
# ============================================================
# RECHNER 7.7: Sallen-Key Tiefpass 2. Ordnung
# ============================================================

sk_char = widgets.Dropdown(
    options=[('Butterworth  (Q=0.707, A₀=1.586)', (0.7071, 1.5858)),
             ('Bessel       (Q=0.577, A₀=1.268)', (0.5774, 1.2679)),
             ('Chebyshev 1dB(Q=0.956, A₀=2.234)', (0.9565, 2.2346)),
             ('Manuell eingeben',                  None)],
    description='Charakteristik:', style={'description_width': '160px'},
    layout=widgets.Layout(width='380px')
)
sk_R   = make_input('R [Ω]   =', 'R₁ = R₂')
sk_C   = make_input('C [F]   =', 'C₁ = C₂, z.B. 100n')
sk_Q   = make_input('Q       =', 'nur bei Manuell')
sk_f   = make_input('f [Hz]  =', 'optional: Verhalten bei f')
sk_U0  = make_input('U_ein [V]=', 'optional')
btn_sk  = make_button()
out_sk  = widgets.Output()

def calc_sk(_=None):
    out_sk.clear_output()
    with out_sk:
        try:
            R  = p(sk_R.value); C = p(sk_C.value)
            f  = p(sk_f.value); U0 = p(sk_U0.value)
            char_val = sk_char.value
            if any(x is None for x in [R, C]):
                print("R und C eingeben"); return
            if char_val is not None:
                Qv, A0 = char_val
            else:
                Qv = p(sk_Q.value)
                if Qv is None:
                    print("Q eingeben oder Charakteristik wählen"); return
                A0 = 3 - 1/Qv
            fg = 1 / (2 * math.pi * R * C)
            Rf_R1 = A0 - 1
            print(f"  f_g   = {fmt(fg, 'Hz')}")
            print(f"  Q     = {Qv:.4g}")
            print(f"  A₀    = {A0:.4g}  (R_f/R₁ = {Rf_R1:.4g})")
            if A0 >= 3:
                print(f"  A₀ ≥ 3 → instabil! Q nicht realisierbar.")
                return
            if f:
                norm   = f / fg
                denom  = math.sqrt((1 - norm**2)**2 + (norm/Qv)**2)
                H      = A0 / denom
                phi    = -math.degrees(math.atan2(norm/Qv, 1 - norm**2))
                dB_val = 20 * math.log10(H)
                print(f"\n  Bei f = {fmt(f, 'Hz')}:")
                print(f"  |H|   = {H:.5g}  ({dB_val:.3g} dB)")
                print(f"  φ     = {phi:.3g}°")
                if U0:
                    print(f"  U_aus = {fmt(H * U0, 'V')}")
        except Exception as e:
            print("Fehler:", e)

btn_sk.on_click(calc_sk)
sk_char.observe(calc_sk, names='value')
for fw in [sk_R, sk_C, sk_Q, sk_f, sk_U0]:
    fw.observe(calc_sk, names='value')
_calc_widget = (widgets.VBox([sk_char, sk_R, sk_C, sk_Q, sk_f, sk_U0, btn_sk, out_sk]))

# ---- HBox Layout ----
_left_box = widgets.Box([_theory_out], layout=widgets.Layout(**LEFT_STYLE))
_right_box = widgets.Box([_calc_widget], layout=widgets.Layout(**RIGHT_STYLE))

display(widgets.HBox(
    [_left_box, _right_box],
    layout=widgets.Layout(
        width='100%',
        align_items='flex-start',
        margin='8px 0 28px 0',
    )
))


In [ ]:
display(HTML('<a name="sec-7-8"></a>'))
# ================================================================
# LAYOUT: Theorie (links) | Rechner (rechts)
# ================================================================

# ---- Theorie (links) ----
_theory_out = widgets.Output(layout=THEORY_STYLE)

with _theory_out:
    display(Markdown(
        "---\n"
        "## 7.8 Notch-Filter (Kerbfilter) mit OPV\n"
        "Ein **Notch-Filter** (Bandsperre) d\u00e4mpft eine einzelne Frequenz sehr stark\n"
        "und l\u00e4sst alle anderen durch. Typische Anwendung: 50 Hz Netzst\u00f6rungen unterdr\u00fccken.\n"
        "**Twin-T Notch:**\n"
        "$$f_0 = \\frac{1}{2\\pi \\cdot R \\cdot C}$$\n"
        "Mit OPV-Gegenkopplung ist die Kerbtiefe einstellbar.\n"
        "Ohne Gegenkopplung (passiv): Kerbtiefe theoretisch $-\\infty$ dB bei exakten Bauteilen.\n"
        "$$\\left|\\underline{H}\\right| = \\frac{\\left|1 - \\left(\\frac{f}{f_0}\\right)^2\\right|}{\\sqrt{\\left(1-\\left(\\frac{f}{f_0}\\right)^2\\right)^2 + \\left(\\frac{f}{f_0 \\cdot Q}\\right)^2}}$$\n"
        "<!-- Bild: Twin-T Notch Filter mit OPV -->\n"
        "<!-- Dateiname: twin_t_notch_filter_opv.png  \u2192 Google: \"Twin-T notch filter op-amp active circuit 50Hz\" -->"))

# ---- Rechner (rechts) ----
# ============================================================
# RECHNER 7.8: Notch-Filter (Twin-T)
# ============================================================

nt_R  = make_input('R [Ω]   =')
nt_C  = make_input('C [F]   =', 'z.B. 100n')
nt_Q  = make_input('Q       =', 'leer → passiv (Q≈0.25)')
nt_f  = make_input('f [Hz]  =', 'optional: Dämpfung bei f')
nt_U0 = make_input('U_ein [V]=', 'optional')
btn_nt = make_button()
out_nt = widgets.Output()

def calc_nt(_=None):
    out_nt.clear_output()
    with out_nt:
        try:
            R  = p(nt_R.value); C  = p(nt_C.value)
            Q  = p(nt_Q.value); fv = p(nt_f.value)
            U0 = p(nt_U0.value)
            if any(x is None for x in [R, C]):
                print("R und C eingeben"); return
            f0 = 1 / (2 * math.pi * R * C)
            Qv = Q if Q else 0.25
            df = f0 / Qv
            print(f"  f₀ (Kerbe) = {fmt(f0, 'Hz')}")
            print(f"  Q          = {Qv:.4g}")
            print(f"  Δf (−3dB)  = {fmt(df, 'Hz')}")
            if not Q:
                print(f"  (passiver Twin-T, Q ≈ 0.25)")
            if fv:
                norm   = fv / f0
                num    = abs(1 - norm**2)
                denom  = math.sqrt((1 - norm**2)**2 + (norm/Qv)**2)
                H      = num / denom if denom > 0 else 0
                dB_val = 20 * math.log10(H) if H > 1e-10 else -100
                print(f"\n  Bei f = {fmt(fv, 'Hz')}:")
                print(f"  |H|   = {H:.5g}  ({dB_val:.3g} dB)")
                if U0:
                    print(f"  U_aus = {fmt(H * U0, 'V')}")
                if abs(fv - f0) < 1:
                    print(f"  → Genau bei Kerbfrequenz – maximale Dämpfung")
        except Exception as e:
            print("Fehler:", e)

btn_nt.on_click(calc_nt)
for fw in [nt_R, nt_C, nt_Q, nt_f, nt_U0]:
    fw.observe(calc_nt, names='value')
_calc_widget = (widgets.VBox([nt_R, nt_C, nt_Q, nt_f, nt_U0, btn_nt, out_nt]))

# ---- HBox Layout ----
_left_box = widgets.Box([_theory_out], layout=widgets.Layout(**LEFT_STYLE))
_right_box = widgets.Box([_calc_widget], layout=widgets.Layout(**RIGHT_STYLE))

display(widgets.HBox(
    [_left_box, _right_box],
    layout=widgets.Layout(
        width='100%',
        align_items='flex-start',
        margin='8px 0 28px 0',
    )
))


In [ ]:
display(HTML('<a name="sec-8-1"></a>'))
# ================================================================
# LAYOUT: Theorie (links) | Rechner (rechts)
# ================================================================

# ---- Theorie (links) ----
_theory_out = widgets.Output(layout=THEORY_STYLE)

with _theory_out:
    display(Markdown(
        "---\n"
        "# Kapitel 8: Transformator\n"
        "---\n"
        "## 8.1 Grundlagen und \u00dcbersetzungsverh\u00e4ltnis\n"
        "Ein **Transformator** \u00fcbertr\u00e4gt elektrische Energie durch magnetische Kopplung\n"
        "zwischen zwei Wicklungen \u2013 galvanisch getrennt und verlustarm.\n"
        "| Gr\u00f6sse | Symbol | Einheit |\n"
        "|--------|--------|---------|\n"
        "| Prim\u00e4rspannung | $U_1$ | V |\n"
        "| Sekund\u00e4rspannung | $U_2$ | V |\n"
        "| Prim\u00e4rstrom | $I_1$ | A |\n"
        "| Sekund\u00e4rstrom | $I_2$ | A |\n"
        "| Prim\u00e4rwindungen | $N_1$ | \u2013 |\n"
        "| Sekund\u00e4rwindungen | $N_2$ | \u2013 |\n"
        "| \u00dcbersetzungsverh\u00e4ltnis | $\u00fc$ | \u2013 |\n"
        "### \u00dcbersetzungsverh\u00e4ltnis\n"
        "$$\u00fc = \\frac{N_1}{N_2} = \\frac{U_1}{U_2} = \\frac{I_2}{I_1}$$\n"
        "> Achtung: Beim Strom steht $I_2/I_1$ \u2013 also umgekehrt zu Spannung und Windungen!\n"
        "<!-- Bild: Transformator Schaltzeichen mit N1, N2, U1, U2, I1, I2 -->\n"
        "<!-- Dateiname: transformator_schaltzeichen.png  \u2192 Google: \"transformer schematic symbol primary secondary winding N1 N2\" -->"))

# ---- Rechner (rechts) ----
# ============================================================
# RECHNER 8.1: Übersetzungsverhältnis
# ============================================================

ue_N1 = make_input('N₁ (Primär)   =', 'Windungszahl')
ue_N2 = make_input('N₂ (Sekundär) =', 'Windungszahl')
ue_U1 = make_input('U₁ [V]        =')
ue_U2 = make_input('U₂ [V]        =')
ue_I1 = make_input('I₁ [A]        =')
ue_I2 = make_input('I₂ [A]        =')
btn_ue = make_button()
out_ue = widgets.Output()

def calc_ue(_=None):
    out_ue.clear_output()
    with out_ue:
        try:
            N1 = p(ue_N1.value); N2 = p(ue_N2.value)
            U1 = p(ue_U1.value); U2 = p(ue_U2.value)
            I1 = p(ue_I1.value); I2 = p(ue_I2.value)

            # ü aus vorhandenen Paaren bestimmen
            ue = None
            if N1 and N2: ue = N1 / N2; print(f"  ü = N₁/N₂ = {ue:.5g}")
            if U1 and U2:
                ue_u = U1 / U2
                print(f"  ü = U₁/U₂ = {ue_u:.5g}")
                if ue and abs(ue - ue_u) > 0.001:
                    print(f"  Widerspruch: N-Verhältnis ≠ U-Verhältnis!")
                ue = ue_u
            if I1 and I2:
                ue_i = I2 / I1
                print(f"  ü = I₂/I₁ = {ue_i:.5g}")
                if ue and abs(ue - ue_i) > 0.001:
                    print(f"  Widerspruch: Verhältnis passt nicht!")
                ue = ue_i

            if ue is None:
                print("Mindestens ein Paar (N₁+N₂, U₁+U₂ oder I₁+I₂) eingeben")
                return

            print(f"\n  ü     = {ue:.5g}")
            if   ue > 1: print(f"  → Abwärtstransformator (U₂ < U₁)")
            elif ue < 1: print(f"  → Aufwärtstransformator (U₂ > U₁)")
            else:        print(f"  → 1:1 Trenntransformator")

            # Fehlende Werte berechnen
            if ue:
                if U1 and not U2: print(f"  U₂ = U₁/ü = {fmt(U1/ue, 'V')}")
                if U2 and not U1: print(f"  U₁ = U₂·ü = {fmt(U2*ue, 'V')}")
                if I1 and not I2: print(f"  I₂ = I₁·ü = {fmt(I1*ue, 'A')}")
                if I2 and not I1: print(f"  I₁ = I₂/ü = {fmt(I2/ue, 'A')}")
                if N1 and not N2: print(f"  N₂ = N₁/ü = {N1/ue:.4g}")
                if N2 and not N1: print(f"  N₁ = N₂·ü = {N2*ue:.4g}")
        except Exception as e:
            print("Fehler:", e)

btn_ue.on_click(calc_ue)
for fw in [ue_N1, ue_N2, ue_U1, ue_U2, ue_I1, ue_I2]:
    fw.observe(calc_ue, names='value')
_calc_widget = (widgets.VBox([ue_N1, ue_N2, ue_U1, ue_U2, ue_I1, ue_I2, btn_ue, out_ue]))

# ---- HBox Layout ----
_left_box = widgets.Box([_theory_out], layout=widgets.Layout(**LEFT_STYLE))
_right_box = widgets.Box([_calc_widget], layout=widgets.Layout(**RIGHT_STYLE))

display(widgets.HBox(
    [_left_box, _right_box],
    layout=widgets.Layout(
        width='100%',
        align_items='flex-start',
        margin='8px 0 28px 0',
    )
))


In [ ]:
display(HTML('<a name="sec-8-2"></a>'))
# ================================================================
# LAYOUT: Theorie (links) | Rechner (rechts)
# ================================================================

# ---- Theorie (links) ----
_theory_out = widgets.Output(layout=THEORY_STYLE)

with _theory_out:
    display(Markdown(
        "---\n"
        "## 8.2 Leistung und Wirkungsgrad\n"
        "Beim idealen Transformator gilt:\n"
        "$$P_1 = P_2 \\qquad U_1 \\cdot I_1 = U_2 \\cdot I_2$$\n"
        "Beim realen Transformator entstehen Verluste:\n"
        "| Verlusttyp | Ursache | Wo |\n"
        "|------------|---------|-----|\n"
        "| Kupferverluste $P_{Cu}$ | Wicklungswiderstand $R_{Cu}$ | Wicklung |\n"
        "| Eisenverluste $P_{Fe}$ | Ummagnetisierung, Wirbelstr\u00f6me | Kern |\n"
        "### Wirkungsgrad\n"
        "$$\\eta = \\frac{P_2}{P_1} = \\frac{P_2}{P_2 + P_{Cu} + P_{Fe}}$$\n"
        "Gute Netztransformatoren erreichen $\\eta = 95\\ldots99\\,\\%$."))

# ---- Rechner (rechts) ----
# ============================================================
# RECHNER 8.2: Leistung und Wirkungsgrad
# ============================================================

eta_P1   = make_input('P₁ [W]    =', 'Primärleistung')
eta_P2   = make_input('P₂ [W]    =', 'Sekundärleistung')
eta_Pcu  = make_input('P_Cu [W]  =', 'Kupferverluste (optional)')
eta_Pfe  = make_input('P_Fe [W]  =', 'Eisenverluste (optional)')
eta_eta  = make_input('η [%]     =', 'Wirkungsgrad z.B. 97')
btn_eta  = make_button()
out_eta  = widgets.Output()

def calc_eta(_=None):
    # → Logik in calc.py: trafo_wirkungsgrad()
    out_eta.clear_output()
    with out_eta:
        try:
            r = trafo_wirkungsgrad(P1=p(eta_P1.value), P2=p(eta_P2.value), eta_pct=p(eta_eta.value))
            _u = {'P2': 'W', 'P1': 'W', 'eta_pct': '%', 'P_verl': 'W'}
            _skip = ('berechnet', 'ladetabelle', 'stromtabelle', 'widerspruch')
            for k, v in r.items():
                if k in _skip or v is None: continue
                if isinstance(v, bool):
                    if v: print(f'  {k}')
                    continue
                if isinstance(v, list): continue
                u = _u.get(k, '')
                print(f'  {k} = {fmt(v, u) if isinstance(v,(int,float)) else v}')
        except (ValueError, ZeroDivisionError) as e:
            print('i', e)
        except Exception as e:
            print('Fehler:', e)
btn_eta.on_click(calc_eta)
for fw in [eta_P1, eta_P2, eta_Pcu, eta_Pfe, eta_eta]:
    fw.observe(calc_eta, names='value')
_calc_widget = (widgets.VBox([eta_P1, eta_P2, eta_Pcu, eta_Pfe, eta_eta, btn_eta, out_eta]))

# ---- HBox Layout ----
_left_box = widgets.Box([_theory_out], layout=widgets.Layout(**LEFT_STYLE))
_right_box = widgets.Box([_calc_widget], layout=widgets.Layout(**RIGHT_STYLE))

display(widgets.HBox(
    [_left_box, _right_box],
    layout=widgets.Layout(
        width='100%',
        align_items='flex-start',
        margin='8px 0 28px 0',
    )
))


In [ ]:
display(HTML('<a name="sec-8-3"></a>'))
# ================================================================
# LAYOUT: Theorie (links) | Rechner (rechts)
# ================================================================

# ---- Theorie (links) ----
_theory_out = widgets.Output(layout=THEORY_STYLE)

with _theory_out:
    display(Markdown(
        "---\n"
        "## 8.3 Widerstandstransformation\n"
        "Ein Transformator transformiert nicht nur Spannung und Strom,\n"
        "sondern auch **Widerst\u00e4nde** \u2013 sehr wichtig f\u00fcr Impedanzanpassung!\n"
        "$$R_1' = \u00fc^2 \\cdot R_2 \\qquad \\text{(Lastwiderstand auf Prim\u00e4rseite umgerechnet)}$$\n"
        "$$R_2' = \\frac{R_1}{\u00fc^2} \\qquad \\text{(Prim\u00e4rwiderstand auf Sekund\u00e4rseite)}$$\n"
        "**Anwendung:** Lautsprecher-Impedanzanpassung, HF-Anpassung\n"
        "<!-- Bild: Transformator Widerstandstransformation -->\n"
        "<!-- Dateiname: transformator_impedanzanpassung.png  \u2192 Google: \"transformer impedance matching circuit diagram R primary secondary\" -->"))

# ---- Rechner (rechts) ----
# ============================================================
# RECHNER 8.3: Widerstandstransformation
# ============================================================

wt_ue  = make_input('ü = N₁/N₂  =', 'Übersetzungsverhältnis')
wt_R1  = make_input('R₁ [Ω]     =', 'Primärwiderstand')
wt_R2  = make_input('R₂ [Ω]     =', 'Sekundärlast')
btn_wt = make_button()
out_wt = widgets.Output()

def calc_wt(_=None):
    # → Logik in calc.py: widerstandstransformation()
    out_wt.clear_output()
    with out_wt:
        try:
            r = widerstandstransformation(ue=p(wt_ue.value), R1=p(wt_R1.value), R2=p(wt_R2.value))
            _u = {'R1_prim': 'Ω', 'R2_sek': 'Ω'}
            _skip = ('berechnet', 'ladetabelle', 'stromtabelle', 'widerspruch')
            for k, v in r.items():
                if k in _skip or v is None: continue
                if isinstance(v, bool):
                    if v: print(f'  {k}')
                    continue
                if isinstance(v, list): continue
                u = _u.get(k, '')
                print(f'  {k} = {fmt(v, u) if isinstance(v,(int,float)) else v}')
        except (ValueError, ZeroDivisionError) as e:
            print('i', e)
        except Exception as e:
            print('Fehler:', e)
btn_wt.on_click(calc_wt)
for fw in [wt_ue, wt_R1, wt_R2]:
    fw.observe(calc_wt, names='value')
_calc_widget = (widgets.VBox([wt_ue, wt_R1, wt_R2, btn_wt, out_wt]))

# ---- HBox Layout ----
_left_box = widgets.Box([_theory_out], layout=widgets.Layout(**LEFT_STYLE))
_right_box = widgets.Box([_calc_widget], layout=widgets.Layout(**RIGHT_STYLE))

display(widgets.HBox(
    [_left_box, _right_box],
    layout=widgets.Layout(
        width='100%',
        align_items='flex-start',
        margin='8px 0 28px 0',
    )
))


In [ ]:
display(HTML('<a name="sec-8-4"></a>'))
# ================================================================
# LAYOUT: Theorie (links) | Rechner (rechts)
# ================================================================

# ---- Theorie (links) ----
_theory_out = widgets.Output(layout=THEORY_STYLE)

with _theory_out:
    display(Markdown(
        "---\n"
        "## 8.4 Leerlauf und Kurzschluss\n"
        "### Leerlauf (Sekund\u00e4r offen, $I_2 = 0$)\n"
        "- Prim\u00e4rstrom = kleiner Magnetisierungsstrom $I_\\mu$\n"
        "- $U_2 = U_1 / \u00fc$ (Leerlaufspannung)\n"
        "- Keine Leistungs\u00fcbertragung (nur Eisenverluste)\n"
        "### Kurzschluss (Sekund\u00e4r kurzgeschlossen, $U_2 = 0$)\n"
        "$$I_{K2} = \\frac{U_1}{\u00fc \\cdot (R_{Cu1} + \u00fc^2 \\cdot R_{Cu2})}$$\n"
        "> Kurzschlussstrom kann sehr gross sein \u2013 Sicherung n\u00f6tig!\n"
        "### Spannungsabfall unter Last (N\u00e4herung)\n"
        "$$\\Delta U = I_2 \\cdot R_{Cu2} \\qquad \\text{(Sekund\u00e4rseitig)}$$\n"
        "$$U_{2,Last} \\approx U_{2,Leerlauf} - \\Delta U$$"))

# ---- Rechner (rechts) ----
# ============================================================
# RECHNER 8.4: Spannungsabfall unter Last
# ============================================================

last_U2ll  = make_input('U₂ Leerlauf [V] =')
last_Rcu2  = make_input('R_Cu2 [Ω]       =', 'Sek. Wicklungswid.')
last_I2    = make_input('I₂ [A]          =', 'Laststrom')
last_Rlast = make_input('R_Last [Ω]      =', 'oder Lastwid. statt I₂')
btn_last   = make_button()
out_last   = widgets.Output()

def calc_last(_=None):
    out_last.clear_output()
    with out_last:
        try:
            U2ll  = p(last_U2ll.value)
            Rcu2  = p(last_Rcu2.value)
            I2    = p(last_I2.value)
            Rlast = p(last_Rlast.value)

            if U2ll is None or Rcu2 is None:
                print("U₂ Leerlauf und R_Cu2 eingeben"); return

            if I2 is None and Rlast is not None:
                I2 = U2ll / (Rcu2 + Rlast)
                print(f"  I₂    = U₂ll/(R_Cu2+R_Last) = {fmt(I2, 'A')}")
            elif I2 is None:
                print("I₂ oder R_Last eingeben"); return

            dU   = I2 * Rcu2
            U2l  = U2ll - dU
            P2   = U2l * I2
            Pcu  = I2**2 * Rcu2

            print(f"  ΔU    = I₂·R_Cu2 = {fmt(dU, 'V')}")
            print(f"  U₂ Last = {fmt(U2l, 'V')}")
            print(f"  I₂      = {fmt(I2, 'A')}")
            print(f"  P₂      = {fmt(P2, 'W')}")
            print(f"  P_Cu2   = {fmt(Pcu, 'W')}")
            abw = dU / U2ll * 100
            print(f"  Spannungsabfall: {abw:.3g} %")
            if abw > 10:
                print(f"  Hoher Spannungsabfall – R_Cu2 zu gross?")
        except Exception as e:
            print("Fehler:", e)

btn_last.on_click(calc_last)
for fw in [last_U2ll, last_Rcu2, last_I2, last_Rlast]:
    fw.observe(calc_last, names='value')
_calc_widget = (widgets.VBox([last_U2ll, last_Rcu2, last_I2, last_Rlast, btn_last, out_last]))

# ---- HBox Layout ----
_left_box = widgets.Box([_theory_out], layout=widgets.Layout(**LEFT_STYLE))
_right_box = widgets.Box([_calc_widget], layout=widgets.Layout(**RIGHT_STYLE))

display(widgets.HBox(
    [_left_box, _right_box],
    layout=widgets.Layout(
        width='100%',
        align_items='flex-start',
        margin='8px 0 28px 0',
    )
))


In [ ]:
display(HTML('<a name="sec-8-5"></a>'))
# ================================================================
# LAYOUT: Theorie (links) | Rechner (rechts)
# ================================================================

# ---- Theorie (links) ----
_theory_out = widgets.Output(layout=THEORY_STYLE)

with _theory_out:
    display(Markdown(
        "---\n"
        "## 8.5 Spartrafo und Trenntransformator\n"
        "\n"
        "### Spartrafo\n"
        "Primär- und Sekundärwicklung sind **nicht galvanisch getrennt** – eine gemeinsame Wicklung.\n"
        "\n"
        "- Vorteil: Kleiner, leichter, günstiger\n"
        "- Nachteil: Keine galvanische Trennung → Gefahr bei Berührung!\n"
        "- Typisch: Stelltransformator (Variac)\n"
        "\n"
        "$$\\ddot{u} = \\frac{N_{ges}}{N_2} \\qquad \\text{oder} \\qquad \\ddot{u} = \\frac{N_1}{N_{ges}}$$\n"
        "\n"
        "| Grösse | Spartrafo | Normaler Trafo |\n"
        "|--------|-----------|----------------|\n"
        "| Galvanische Trennung | ✗ Nein | ✓ Ja |\n"
        "| Baugrösse | Klein | Grösser |\n"
        "| Wirkungsgrad | Sehr hoch | Hoch |\n"
        "| Sicherheit | Geringer | Höher |\n"
        "| Anwendung | Stelltrafo, Labor | Netzteil, Trennung |\n"
        "\n"
        "### Trenntransformator ($\\ddot{u} = 1$)\n"
        "- $U_2 = U_1$: keine Spannungsänderung\n"
        "- Zweck: **Galvanische Trennung** (Schutzklasse, Messtechnik)\n"
        "- Schutz vor Netzpotential – Berührung einer Seite ungefährlich\n"
        "\n"
        "> **Merkhilfe Spartrafo:** Der Strom im gemeinsamen Wicklungsabschnitt\n"
        "> ist nur die **Differenz** $I_1 - I_2$ → deswegen kleiner Kupferquerschnitt nötig.\n"
    ))

# ---- Rechner (rechts) ----
# ============================================================
# RECHNER 8.5: Spartrafo – Übersetzung, Ströme, Leistung
# ============================================================

sp_mode = widgets.ToggleButtons(
    options=['Spartrafo', 'Trenntransformator'],
    description='Typ:',
    style={'description_width': '120px', 'button_width': '160px'}
)
sp_U1    = make_input('U1 [V]       =', 'Eingangsspannung')
sp_U2    = make_input('U2 [V]       =', 'Ausgangsspannung')
sp_Nges  = make_input('N_ges        =', 'Gesamtwindungen')
sp_N2    = make_input('N2           =', 'Sekundärwindungen')
sp_I2    = make_input('I2 [A]       =', 'Sekundärstrom (Last)')
sp_P     = make_input('P [VA]       =', 'Leistung (optional)')
btn_sp   = make_button()
out_sp   = widgets.Output()

box_spar = widgets.VBox([sp_U1, sp_U2, sp_Nges, sp_N2, sp_I2, sp_P])
box_trenn = widgets.VBox([sp_U1, sp_I2, sp_P])
cont_sp  = widgets.VBox([])

def update_sp_view(_=None):
    if sp_mode.value == 'Spartrafo':
        cont_sp.children = [box_spar, btn_sp, out_sp]
    else:
        cont_sp.children = [box_trenn, btn_sp, out_sp]
    calc_sp()

def calc_sp(_=None):
    out_sp.clear_output()
    with out_sp:
        try:
            U1   = p(sp_U1.value)
            U2   = p(sp_U2.value)
            Nges = p(sp_Nges.value)
            N2   = p(sp_N2.value)
            I2   = p(sp_I2.value)
            Pv   = p(sp_P.value)

            if sp_mode.value == 'Trenntransformator':
                print(f"  --- Trenntransformator (ü = 1) ---")
                if U1:
                    print(f"  U1 = U2 = {fmt(U1, 'V')}")
                if I2:
                    print(f"  I1 = I2 = {fmt(I2, 'A')}  (ideal)")
                if U1 and I2:
                    S = U1 * I2
                    print(f"  S        = {fmt(S, 'VA')}")
                if Pv:
                    print(f"  S mind.  = {fmt(Pv, 'VA')}  (eingegebene Leistung)")
                print(f"  Galvanische Trennung: ✓ Ja")
                print(f"  Spannungsänderung:    Keine")
                return

            # Spartrafo
            # ü bestimmen
            ue = None
            if Nges and N2:
                ue = Nges / N2
                print(f"  ü = N_ges/N2 = {ue:.5g}")
            if U1 and U2:
                ue_u = U1 / U2
                print(f"  ü = U1/U2    = {ue_u:.5g}")
                if ue and abs(ue - ue_u) > 0.01:
                    print(f"  ⚠ Widerspruch zwischen N- und U-Verhältnis!")
                ue = ue_u

            if ue is None:
                print("N_ges + N2 oder U1 + U2 eingeben"); return

            # Fehlende Spannung
            if U1 and not U2:
                U2 = U1 / ue
                print(f"  U2       = {fmt(U2, 'V')}")
            if U2 and not U1:
                U1 = U2 * ue
                print(f"  U1       = {fmt(U1, 'V')}")

            if ue > 1:
                print(f"  → Abwärts-Spartrafo (U2 < U1)")
            elif ue < 1:
                print(f"  → Aufwärts-Spartrafo (U2 > U1)")
            else:
                print(f"  → ü = 1  (entspricht Trenntransformator)")

            if I2 and U1 and U2:
                I1      = I2 / ue          # Primärstrom (ideal)
                I_diff  = abs(I2 - I1)     # Strom im gemeinsamen Abschnitt
                S_ueber = U2 * I2          # Übertragene Leistung
                # Kupfereinsparung: geführte Leistung / übertragene Leistung
                S_induk = U1 * I1          # Induktiv übertragene Leistung
                S_leit  = S_ueber - S_induk if ue > 1 else S_induk - S_ueber
                einspar = abs(S_leit) / S_ueber * 100 if S_ueber > 0 else 0

                print(f"\n  I2       = {fmt(I2, 'A')}  (Sekundär / Last)")
                print(f"  I1       = {fmt(I1, 'A')}  (Primär, ideal)")
                print(f"  I_diff   = {fmt(I_diff, 'A')}  (Strom im gemeins. Abschnitt)")
                print(f"  S        = {fmt(S_ueber, 'VA')}")
                print(f"  Kupfereinsparung ≈ {einspar:.1f}% ggü. normalem Trafo")
        except Exception as e:
            print("Fehler:", e)

sp_mode.observe(update_sp_view, names='value')
btn_sp.on_click(calc_sp)
for fw in [sp_U1, sp_U2, sp_Nges, sp_N2, sp_I2, sp_P]:
    fw.observe(calc_sp, names='value')
update_sp_view()
_calc_widget = widgets.VBox([sp_mode, cont_sp])

# ---- HBox Layout ----
_left_box  = widgets.Box([_theory_out], layout=widgets.Layout(**LEFT_STYLE))
_right_box = widgets.Box([_calc_widget], layout=widgets.Layout(**RIGHT_STYLE))

display(widgets.HBox(
    [_left_box, _right_box],
    layout=widgets.Layout(
        width='100%',
        align_items='flex-start',
        margin='8px 0 28px 0',
    )
))


In [ ]:
display(HTML('<a name="sec-9-1"></a>'))
# ================================================================
# LAYOUT: Theorie (links) | Rechner (rechts)
# ================================================================

# ---- Theorie (links) ----
_theory_out = widgets.Output(layout=THEORY_STYLE)

with _theory_out:
    display(Markdown(
        "---\n"
        "# Kapitel 9: Gleichrichter und Siebung\n"
        "---\n"
        "## 9.1 Grundlagen der Gleichrichtung\n"
        "Ein **Gleichrichter** wandelt Wechselspannung (AC) in pulsierende Gleichspannung (DC) um.\n"
        "Die nachfolgende **Siebung** gl\u00e4ttet die Restwelligkeit.\n"
        "| Gr\u00f6sse | Symbol | Einheit |\n"
        "|--------|--------|---------|\n"
        "| Spitzenwert | $\\hat{U}$ | V |\n"
        "| Effektivwert | $U_{eff}$ | V |\n"
        "| Gleichspannung (Mittelwert) | $\\bar{U}$ | V |\n"
        "| Brummspannung | $U_{Br}$ | V |\n"
        "| Welligkeit | $w$ | % |\n"
        "### Wichtige Zusammenh\u00e4nge Sinusspannung\n"
        "$$\\hat{U} = U_{eff} \\cdot \\sqrt{2} \\qquad U_{eff} = \\frac{\\hat{U}}{\\sqrt{2}}$$\n"
        "<!-- Bild: Sinusspannung mit Spitzen- und Effektivwert -->\n"
        "<!-- Dateiname: sinusspannung_scheitel_effektiv.png  \u2192 Google: \"AC sine wave peak RMS voltage diagram labeled\" -->"))

# ---- Rechner (rechts) ----
# ============================================================
# RECHNER 9.1: Spitzenwert / Effektivwert Umrechnung
# ============================================================

eff_Up  = make_input('Û (Spitzenwert) [V] =')
eff_Ue  = make_input('U_eff [V]           =')
btn_eff = make_button()
out_eff = widgets.Output()

def calc_eff(_=None):
    # → Logik in calc.py: spitzenwert_effektivwert()
    out_eff.clear_output()
    with out_eff:
        try:
            r = spitzenwert_effektivwert(Up=p(eff_Up.value), Ue=p(eff_Ue.value))
            _u = {'Up': 'V', 'Ue': 'V'}
            _skip = ('berechnet', 'ladetabelle', 'stromtabelle', 'widerspruch')
            for k, v in r.items():
                if k in _skip or v is None: continue
                if isinstance(v, bool):
                    if v: print(f'  {k}')
                    continue
                if isinstance(v, list): continue
                u = _u.get(k, '')
                print(f'  {k} = {fmt(v, u) if isinstance(v,(int,float)) else v}')
        except (ValueError, ZeroDivisionError) as e:
            print('i', e)
        except Exception as e:
            print('Fehler:', e)
btn_eff.on_click(calc_eff)
for fw in [eff_Up, eff_Ue]:
    fw.observe(calc_eff, names='value')
_calc_widget = (widgets.VBox([eff_Up, eff_Ue, btn_eff, out_eff]))

# ---- HBox Layout ----
_left_box = widgets.Box([_theory_out], layout=widgets.Layout(**LEFT_STYLE))
_right_box = widgets.Box([_calc_widget], layout=widgets.Layout(**RIGHT_STYLE))

display(widgets.HBox(
    [_left_box, _right_box],
    layout=widgets.Layout(
        width='100%',
        align_items='flex-start',
        margin='8px 0 28px 0',
    )
))


In [ ]:
display(HTML('<a name="sec-9-2"></a>'))
# ================================================================
# LAYOUT: Theorie (links) | Rechner (rechts)
# ================================================================

# ---- Theorie (links) ----
_theory_out = widgets.Output(layout=THEORY_STYLE)

with _theory_out:
    display(Markdown(
        "---\n"
        "## 9.2 Einweggleichrichter (M1)\n"
        "Nur die **positive Halbwelle** wird durchgelassen. Einfachste Schaltung, aber schlechte Ausnutzung.\n"
        "$$\\bar{U} = \\frac{\\hat{U}}{\\pi} \\approx 0.318 \\cdot \\hat{U}$$\n"
        "$$\\bar{U} = 0.45 \\cdot U_{eff}$$\n"
        "**Eigenschaften:**\n"
        "- Nur 50% der Halbwellen genutzt\n"
        "- Brummfrequenz = Netzfrequenz ($f_{Br} = f_{Netz}$)\n"
        "- Grosse Restwelligkeit\n"
        "- Diodenspannung $U_F \\approx 0.7\\,V$ subtrahieren!\n"
        "$$\\bar{U}_{real} = \\bar{U} - U_F$$\n"
        "<!-- Bild: Einweggleichrichter Schaltung und Ausgangsspannung -->\n"
        "<!-- Dateiname: einweggleichrichter_schaltung.png  \u2192 Google: \"half wave rectifier circuit diagram output waveform\" -->"))

# ---- Rechner (rechts) ----
# ============================================================
# RECHNER 9.2: Einweggleichrichter (M1)
# ============================================================

m1_Ueff = make_input('U_eff (AC) [V] =', 'z.B. 230 für Netz')
m1_Up   = make_input('Û (AC) [V]    =', 'oder Spitzenwert')
m1_UF   = make_input('U_F Diode [V] =', 'Flussspannung, ca. 0.7')
btn_m1  = make_button()
out_m1  = widgets.Output()

def calc_m1(_=None):
    out_m1.clear_output()
    with out_m1:
        try:
            Ueff = p(m1_Ueff.value)
            Up   = p(m1_Up.value)
            UF   = p(m1_UF.value) if m1_UF.value.strip() else 0.7

            if Ueff is None and Up is None:
                print("U_eff oder Û eingeben"); return
            if Up is None:   Up   = Ueff * math.sqrt(2)
            if Ueff is None: Ueff = Up / math.sqrt(2)

            U_avg      = Up / math.pi
            U_avg_real = U_avg - UF
            f_Br       = 50  # Hz bei 50 Hz Netz

            print(f"  Û          = {fmt(Up, 'V')}")
            print(f"  U_eff      = {fmt(Ueff, 'V')}")
            print(f"  Ū (ideal)  = Û/π = {fmt(U_avg, 'V')}")
            print(f"  U_F        = {fmt(UF, 'V')}")
            print(f"  Ū (real)   = {fmt(U_avg_real, 'V')}")
            print(f"  f_Brumm    = {f_Br} Hz  (= Netzfrequenz)")
            print(f"  Ausnutzung = 50 %  (nur positive HW)")
            if U_avg_real < 0:
                print(f"  Ausgangsspannung negativ – Eingangsspannung zu klein!")
        except Exception as e:
            print("Fehler:", e)

btn_m1.on_click(calc_m1)
for fw in [m1_Ueff, m1_Up, m1_UF]:
    fw.observe(calc_m1, names='value')
_calc_widget = (widgets.VBox([m1_Ueff, m1_Up, m1_UF, btn_m1, out_m1]))

# ---- HBox Layout ----
_left_box = widgets.Box([_theory_out], layout=widgets.Layout(**LEFT_STYLE))
_right_box = widgets.Box([_calc_widget], layout=widgets.Layout(**RIGHT_STYLE))

display(widgets.HBox(
    [_left_box, _right_box],
    layout=widgets.Layout(
        width='100%',
        align_items='flex-start',
        margin='8px 0 28px 0',
    )
))


In [ ]:
display(HTML('<a name="sec-9-3"></a>'))
# ================================================================
# LAYOUT: Theorie (links) | Rechner (rechts)
# ================================================================

# ---- Theorie (links) ----
_theory_out = widgets.Output(layout=THEORY_STYLE)

with _theory_out:
    display(Markdown(
        "---\n"
        "## 9.3 Br\u00fcckengleichrichter / Graetz-Schaltung (B2)\n"
        "**Beide** Halbwellen werden gleichgerichtet \u2013 deutlich besser als M1.\n"
        "$$\\bar{U} = \\frac{2 \\cdot \\hat{U}}{\\pi} \\approx 0.637 \\cdot \\hat{U}$$\n"
        "$$\\bar{U} = 0.9 \\cdot U_{eff}$$\n"
        "**Eigenschaften:**\n"
        "- Beide Halbwellen genutzt \u2192 doppelte Ausnutzung\n"
        "- Brummfrequenz = **doppelte** Netzfrequenz ($f_{Br} = 2 \\cdot f_{Netz} = 100\\,Hz$)\n"
        "- Immer **2 Dioden** in Reihe leitend \u2192 $2 \\cdot U_F$ abziehen!\n"
        "$$\\bar{U}_{real} = \\frac{2 \\cdot \\hat{U}}{\\pi} - 2 \\cdot U_F$$\n"
        "<!-- Bild: Br\u00fcckengleichrichter Graetz Schaltung -->\n"
        "<!-- Dateiname: brueckengleichrichter_graetz.png  \u2192 Google: \"bridge rectifier Graetz circuit diagram B2 full wave\" -->"))

# ---- Rechner (rechts) ----
# ============================================================
# RECHNER 9.3: Brückengleichrichter B2 (Graetz)
# ============================================================

b2_Ueff = make_input('U_eff (AC) [V] =', 'z.B. 230 für Netz')
b2_Up   = make_input('Û (AC) [V]    =', 'oder Spitzenwert')
b2_UF   = make_input('U_F Diode [V] =', 'pro Diode, ca. 0.7')
btn_b2  = make_button()
out_b2  = widgets.Output()

def calc_b2(_=None):
    out_b2.clear_output()
    with out_b2:
        try:
            Ueff = p(b2_Ueff.value)
            Up   = p(b2_Up.value)
            UF   = p(b2_UF.value) if b2_UF.value.strip() else 0.7

            if Ueff is None and Up is None:
                print("U_eff oder Û eingeben"); return
            if Up is None:   Up   = Ueff * math.sqrt(2)
            if Ueff is None: Ueff = Up / math.sqrt(2)

            U_avg      = 2 * Up / math.pi
            U_avg_real = U_avg - 2 * UF
            U_sperr    = Up  # Sperrspannung je Diode

            print(f"  Û           = {fmt(Up, 'V')}")
            print(f"  U_eff       = {fmt(Ueff, 'V')}")
            print(f"  Ū (ideal)   = 2Û/π = {fmt(U_avg, 'V')}")
            print(f"  2·U_F       = {fmt(2*UF, 'V')}")
            print(f"  Ū (real)    = {fmt(U_avg_real, 'V')}")
            print(f"  f_Brumm     = 100 Hz  (= 2 · Netzfrequenz)")
            print(f"  U_Sperr     = {fmt(U_sperr, 'V')}  (je Diode mind.)")
            print(f"  Ausnutzung  = 100 %  (beide HW)")
            if U_avg_real < 0:
                print(f"  Ausgangsspannung negativ – Eingangsspannung zu klein!")
        except Exception as e:
            print("Fehler:", e)

btn_b2.on_click(calc_b2)
for fw in [b2_Ueff, b2_Up, b2_UF]:
    fw.observe(calc_b2, names='value')
_calc_widget = (widgets.VBox([b2_Ueff, b2_Up, b2_UF, btn_b2, out_b2]))

# ---- HBox Layout ----
_left_box = widgets.Box([_theory_out], layout=widgets.Layout(**LEFT_STYLE))
_right_box = widgets.Box([_calc_widget], layout=widgets.Layout(**RIGHT_STYLE))

display(widgets.HBox(
    [_left_box, _right_box],
    layout=widgets.Layout(
        width='100%',
        align_items='flex-start',
        margin='8px 0 28px 0',
    )
))


In [ ]:
display(HTML('<a name="sec-9-4"></a>'))
# ================================================================
# LAYOUT: Theorie (links) | Rechner (rechts)
# ================================================================

# ---- Theorie (links) ----
_theory_out = widgets.Output(layout=THEORY_STYLE)

with _theory_out:
    display(Markdown(
        "---\n"
        "## 9.4 Siebung mit Kondensator\n"
        "Der **Siebkondensator** gl\u00e4ttet die pulsierende Gleichspannung.\n"
        "Zwischen den Ladeimpulsen entl\u00e4dt er sich in die Last.\n"
        "### Brummspannung (N\u00e4herung)\n"
        "$$U_{Br} \\approx \\frac{I_{Last}}{f_{Br} \\cdot C} = \\frac{\\bar{U}}{R_{Last} \\cdot f_{Br} \\cdot C}$$\n"
        "### Mittlere Ausgangsspannung (mit Siebung)\n"
        "$$\\bar{U}_{DC} \\approx \\hat{U} - \\frac{U_{Br}}{2}$$\n"
        "### Welligkeit\n"
        "$$w = \\frac{U_{Br}}{\\bar{U}_{DC}} \\cdot 100\\,\\%$$\n"
        "**Faustregeln:**\n"
        "- Gute Siebung: $w < 1\\,\\%$\n"
        "- Ausreichend: $w < 5\\,\\%$\n"
        "- $C$ gross w\u00e4hlen \u2192 kleine Brummspannung\n"
        "- Br\u00fcckengleichrichter: $f_{Br} = 100\\,Hz$ \u2192 halb so grosser $C$ n\u00f6tig wie bei M1\n"
        "<!-- Bild: Siebkondensator Lade- und Entladekurve mit Brummspannung -->\n"
        "<!-- Dateiname: siebkondensator_brummspannung.png  \u2192 Google: \"capacitor filter ripple voltage DC power supply waveform labeled\" -->"))

# ---- Rechner (rechts) ----
# ============================================================
# RECHNER 9.4: Siebkondensator und Brummspannung
# ============================================================

sib_typ = widgets.ToggleButtons(
    options=['Brücke B2 (f=100Hz)', 'Einweg M1 (f=50Hz)'],
    description='Gleichrichter:', style={'description_width': '140px', 'button_width': '180px'}
)
sib_C    = make_input('C [F]        =', 'z.B. 1000u, 4700u')
sib_I    = make_input('I_Last [A]   =', 'Laststrom')
sib_R    = make_input('R_Last [Ω]   =', 'oder Lastwiderstand')
sib_UBr  = make_input('U_Br [V]     =', 'leer = berechnen')
sib_Up   = make_input('Û [V]        =', 'optional: U_DC berechnen')
btn_sib  = make_button()
out_sib  = widgets.Output()

def calc_sib(_=None):
    out_sib.clear_output()
    with out_sib:
        try:
            C    = p(sib_C.value)
            I    = p(sib_I.value)
            R    = p(sib_R.value)
            UBr  = p(sib_UBr.value)
            Up   = p(sib_Up.value)
            fBr  = 100 if 'B2' in sib_typ.value else 50

            print(f"  f_Brumm = {fBr} Hz")

            # I aus R berechnen wenn nötig
            if I is None and R is not None and Up is not None:
                I = Up / R
                print(f"  I_Last  = Û/R = {fmt(I, 'A')}")
            elif I is None and R is not None and UBr is None:
                print("Û oder I_Last zusätzlich eingeben"); return

            vals = {'C': C, 'U_Br': UBr}
            miss = [k for k, v in vals.items() if v is None]

            if C is not None and I is not None:
                UBr_calc = I / (fBr * C)
                print(f"  U_Br    = I/(f·C) = {fmt(UBr_calc, 'V')}")
                if Up:
                    UDC = Up - UBr_calc / 2
                    w   = UBr_calc / UDC * 100
                    print(f"  Ū_DC    ≈ Û - U_Br/2 = {fmt(UDC, 'V')}")
                    print(f"  Welligkeit w = {w:.3g} %")
                    if   w < 1:  print(f"  → Sehr gute Siebung")
                    elif w < 5:  print(f"  → Ausreichende Siebung")
                    else:        print(f"  Schlechte Siebung – C vergrössern!")

            elif UBr is not None and I is not None:
                C_calc = I / (fBr * UBr)
                print(f"  C = I/(f·U_Br) = {fmt(C_calc, 'F')}")
                print(f"  → Mindestkapazität für U_Br ≤ {fmt(UBr, 'V')}")

            elif C is not None and UBr is not None:
                I_max = UBr * fBr * C
                print(f"  I_max = U_Br·f·C = {fmt(I_max, 'A')}")
                print(f"  → Maximaler Laststrom bei U_Br ≤ {fmt(UBr, 'V')}")

            else:
                print("C + I_Last  oder  U_Br + I_Last  oder  C + U_Br eingeben")

        except Exception as e:
            print("Fehler:", e)

btn_sib.on_click(calc_sib)
sib_typ.observe(calc_sib, names='value')
for fw in [sib_C, sib_I, sib_R, sib_UBr, sib_Up]:
    fw.observe(calc_sib, names='value')
_calc_widget = (widgets.VBox([sib_typ, sib_C, sib_I, sib_R, sib_UBr, sib_Up, btn_sib, out_sib]))

# ---- HBox Layout ----
_left_box = widgets.Box([_theory_out], layout=widgets.Layout(**LEFT_STYLE))
_right_box = widgets.Box([_calc_widget], layout=widgets.Layout(**RIGHT_STYLE))

display(widgets.HBox(
    [_left_box, _right_box],
    layout=widgets.Layout(
        width='100%',
        align_items='flex-start',
        margin='8px 0 28px 0',
    )
))


In [ ]:
display(HTML('<a name="sec-9-5"></a>'))
# ================================================================
# LAYOUT: Theorie (links) | Rechner (rechts)
# ================================================================

# ---- Theorie (links) ----
_theory_out = widgets.Output(layout=THEORY_STYLE)

with _theory_out:
    display(Markdown(
        "---\n"
        "## 9.5 Netzteil-Dimensionierung (komplett)\n"
        "Typische Kette: **Trafo \u2192 Gleichrichter \u2192 Siebkondensator \u2192 Last**\n"
        "### Vorgehen\n"
        "1. Gew\u00fcnschte DC-Spannung $\\bar{U}_{DC}$ und Strom $I_{Last}$ festlegen\n"
        "2. Brummspannung $U_{Br}$ maximal festlegen (z.B. 1% von $\\bar{U}_{DC}$)\n"
        "3. Sekund\u00e4rspannung Trafo berechnen:\n"
        "$$U_{2,eff} = \\frac{\\bar{U}_{DC} + 2 \\cdot U_F}{\\sqrt{2} \\cdot 0.9} \\qquad \\text{(Br\u00fccke B2)}$$\n"
        "4. Siebkondensator berechnen:\n"
        "$$C = \\frac{I_{Last}}{f_{Br} \\cdot U_{Br}}$$\n"
        "5. Kondensator-Spannungsfestigkeit: mind. $\\hat{U} \\cdot 1.5$ als Reserve\n"
        "<!-- Bild: Komplettes Netzteil Blockschaltbild -->\n"
        "<!-- Dateiname: netzteil_blockschaltbild.png  \u2192 Google: \"DC power supply block diagram transformer rectifier filter regulator\" -->"))

# ---- Rechner (rechts) ----
# ============================================================
# RECHNER 9.5: Komplette Netzteil-Dimensionierung
# ============================================================

nt_UDC  = make_input('Ū_DC [V]      =', 'Gewünschte DC-Spannung')
nt_I    = make_input('I_Last [A]    =', 'Maximaler Laststrom')
nt_w    = make_input('w_max [%]     =', 'Max. Welligkeit, z.B. 1')
nt_UF   = make_input('U_F Diode [V] =', 'ca. 0.7 pro Diode')
btn_nt  = make_button()
out_nt  = widgets.Output()

def calc_nt(_=None):
    out_nt.clear_output()
    with out_nt:
        try:
            UDC = p(nt_UDC.value)
            I   = p(nt_I.value)
            w   = p(nt_w.value)
            UF  = p(nt_UF.value) if nt_UF.value.strip() else 0.7

            if any(x is None for x in [UDC, I]):
                print("Ū_DC und I_Last eingeben"); return

            w_fakt = (w / 100) if w else 0.01  # default 1%
            fBr    = 100  # Brücke B2

            # Brummspannung
            UBr    = w_fakt * UDC

            # Benötigter Spitzenwert
            Up_min = UDC + UBr / 2 + 2 * UF

            # Trafo Sekundärspannung
            U2eff  = Up_min / math.sqrt(2)

            # Siebkondensator
            C_min  = I / (fBr * UBr)

            # Spannungsfestigkeit
            U_fest = Up_min * 1.5

            # Trafo Scheinleistung
            S_trafo = U2eff * I * 1.5  # Faktor 1.5 wegen Stromform

            print(f"--- Ergebnis Netzteil-Dimensionierung (B2) ---")
            print(f"  Ū_DC gewünscht  = {fmt(UDC, 'V')}")
            print(f"  I_Last          = {fmt(I, 'A')}")
            print(f"  Welligkeit max  = {w_fakt*100:.4g} %")
            print(f"\n--- Brummspannung ---")
            print(f"  U_Br max        = {fmt(UBr, 'V')}")
            print(f"\n--- Transformator ---")
            print(f"  Û benötigt      = {fmt(Up_min, 'V')}")
            print(f"  U₂_eff (Trafo)  = {fmt(U2eff, 'V')}")
            print(f"  S_Trafo mind.   = {fmt(S_trafo, 'VA')}")
            print(f"\n--- Siebkondensator ---")
            print(f"  C_min           = {fmt(C_min, 'F')}")
            print(f"  Spg.-festigkeit = {fmt(U_fest, 'V')}  (mind.)")
            print(f"\n--- Dioden ---")
            print(f"  U_F je Diode    = {fmt(UF, 'V')}")
            print(f"  U_Sperr mind.   = {fmt(Up_min, 'V')} je Diode")
            print(f"  I_F mind.       = {fmt(I, 'A')} (Spitzenstrom ~3x höher!)")
        except Exception as e:
            print("Fehler:", e)

btn_nt.on_click(calc_nt)
for fw in [nt_UDC, nt_I, nt_w, nt_UF]:
    fw.observe(calc_nt, names='value')
_calc_widget = (widgets.VBox([nt_UDC, nt_I, nt_w, nt_UF, btn_nt, out_nt]))

# ---- HBox Layout ----
_left_box = widgets.Box([_theory_out], layout=widgets.Layout(**LEFT_STYLE))
_right_box = widgets.Box([_calc_widget], layout=widgets.Layout(**RIGHT_STYLE))

display(widgets.HBox(
    [_left_box, _right_box],
    layout=widgets.Layout(
        width='100%',
        align_items='flex-start',
        margin='8px 0 28px 0',
    )
))


In [ ]:
display(HTML('<a name="sec-9-6"></a>'))
# ================================================================
# LAYOUT: Theorie (links) | Rechner (rechts)
# ================================================================

# ---- Theorie (links) ----
_theory_out = widgets.Output(layout=THEORY_STYLE)

with _theory_out:
    display(Markdown(
        "---\n"
        "## 9.6 Vergleich Gleichrichtertypen\n"
        "\n"
        "| Eigenschaft | Einweg M1 | Brücke B2 |\n"
        "|-------------|-----------|-----------|\n"
        "| Anzahl Dioden | 1 | 4 |\n"
        "| Diodenabfall | $1 \\cdot U_F$ | $2 \\cdot U_F$ |\n"
        "| $\\bar{U} / \\hat{U}$ | 0.318 | 0.637 |\n"
        "| $\\bar{U} / U_{eff}$ | 0.45 | 0.90 |\n"
        "| Brummfrequenz | 50 Hz | 100 Hz |\n"
        "| Siebaufwand | Gross | Halb so gross |\n"
        "| Ausnutzung Trafo | Schlecht | Gut |\n"
        "| Anwendung | Kleine Signale | Netzteile |\n"
        "\n"
        "> In der Praxis wird fast immer der **Brückengleichrichter B2** verwendet,\n"
        "> da die doppelte Brummfrequenz den Siebkondensator halbiert.\n"
        "\n"
        "### Dioden-Sperrspannung\n"
        "| Typ | Sperrspannung je Diode |\n"
        "|-----|------------------------|\n"
        "| M1 | $\\hat{U}$ |\n"
        "| B2 | $\\hat{U}$ |\n"
        "\n"
        "### Mittlere Ausgangsspannung\n"
        "$$\\bar{U}_{M1} = 0.45 \\cdot U_{eff} - U_F$$\n"
        "$$\\bar{U}_{B2} = 0.90 \\cdot U_{eff} - 2 U_F$$\n"
    ))

# ---- Rechner (rechts) ----
# ============================================================
# RECHNER 9.6: Direktvergleich M1 vs B2 bei gleicher Eingangsspannung
# ============================================================

cmp_Ueff = make_input('U_eff (AC) [V]  =', 'z.B. 12 oder 230')
cmp_UF   = make_input('U_F Diode [V]   =', 'ca. 0.7 pro Diode')
cmp_I    = make_input('I_Last [A]      =', 'Laststrom')
cmp_C    = make_input('C [F]           =', 'Siebkond. z.B. 1000u')
btn_cmp  = make_button('Vergleich')
out_cmp  = widgets.Output()

def calc_cmp(_=None):
    out_cmp.clear_output()
    with out_cmp:
        try:
            Ueff = p(cmp_Ueff.value)
            UF   = p(cmp_UF.value) if cmp_UF.value.strip() else 0.7
            I    = p(cmp_I.value)
            C    = p(cmp_C.value)

            if Ueff is None:
                print("U_eff eingeben"); return

            Up = Ueff * math.sqrt(2)

            # M1
            U_m1  = 0.45 * Ueff - UF
            fBr_m1 = 50
            # B2
            U_b2  = 0.90 * Ueff - 2 * UF
            fBr_b2 = 100

            col = 20
            print(f"  {'':25} {'M1 Einweg':>12} {'B2 Brücke':>12}")
            print(f"  {'-'*50}")
            print(f"  {'Anz. Dioden':25} {'1':>12} {'4':>12}")
            print(f"  {'Diodenabfall':25} {fmt(UF,'V'):>12} {fmt(2*UF,'V'):>12}")
            print(f"  {'Û':25} {fmt(Up,'V'):>12} {fmt(Up,'V'):>12}")
            print(f"  {'Ū (Ausgang)':25} {fmt(U_m1,'V'):>12} {fmt(U_b2,'V'):>12}")
            print(f"  {'Brummfrequenz':25} {'50 Hz':>12} {'100 Hz':>12}")

            if I:
                C_m1 = I / (fBr_m1 * 1)  # für 1V Brumm als Referenz
                C_b2 = I / (fBr_b2 * 1)
                print(f"  {'C (für 1V Brumm)':25} {fmt(C_m1,'F'):>12} {fmt(C_b2,'F'):>12}")
                print(f"  {'→ B2 braucht':25} {'':>12} {'halb so viel C':>12}")

            if I and C:
                UBr_m1 = I / (fBr_m1 * C)
                UBr_b2 = I / (fBr_b2 * C)
                w_m1   = UBr_m1 / U_m1 * 100 if U_m1 > 0 else float('inf')
                w_b2   = UBr_b2 / U_b2 * 100 if U_b2 > 0 else float('inf')
                print(f"\n  --- Mit C = {fmt(C,'F')} und I = {fmt(I,'A')} ---")
                print(f"  {'U_Brumm':25} {fmt(UBr_m1,'V'):>12} {fmt(UBr_b2,'V'):>12}")
                print(f"  {'Welligkeit':25} {w_m1:.2f}%{'':{12-len(f'{w_m1:.2f}%')-0}} {w_b2:.2f}%")

            # Empfehlung
            print(f"\n  → Empfehlung: B2 (doppelte Brummfreq., halb so grosses C)")

        except Exception as e:
            print("Fehler:", e)

btn_cmp.on_click(calc_cmp)
for fw in [cmp_Ueff, cmp_UF, cmp_I, cmp_C]:
    fw.observe(calc_cmp, names='value')
_calc_widget = widgets.VBox([cmp_Ueff, cmp_UF, cmp_I, cmp_C, btn_cmp, out_cmp])

# ---- HBox Layout ----
_left_box  = widgets.Box([_theory_out], layout=widgets.Layout(**LEFT_STYLE))
_right_box = widgets.Box([_calc_widget], layout=widgets.Layout(**RIGHT_STYLE))

display(widgets.HBox(
    [_left_box, _right_box],
    layout=widgets.Layout(
        width='100%',
        align_items='flex-start',
        margin='8px 0 28px 0',
    )
))


In [ ]:
display(HTML('<a name="sec-10-1"></a>'))
# ================================================================
# LAYOUT: Theorie (links) | Rechner (rechts)
# ================================================================

# ---- Theorie (links) ----
_theory_out = widgets.Output(layout=THEORY_STYLE)

with _theory_out:
    display(Markdown(
        "---\n"
        "# Kapitel 10: Thermik\n"
        "---\n"
        "## 10.1 Grundlagen \u2013 W\u00e4rmewiderstand\n"
        "W\u00e4rme fliesst immer von warm nach kalt \u2013 analog zum elektrischen Strom!\n"
        "| Thermisch | Elektrisch | Einheit |\n"
        "|-----------|-----------|---------|\n"
        "| W\u00e4rmestrom $P_V$ | Strom $I$ | W |\n"
        "| Temperaturdifferenz $\\Delta T$ | Spannung $U$ | K |\n"
        "| W\u00e4rmewiderstand $R_{th}$ | Widerstand $R$ | K/W |\n"
        "$$\\Delta T = P_V \\cdot R_{th} \\qquad R_{th} = \\frac{\\Delta T}{P_V}$$\n"
        "### Thermische Kette (Reihenschaltung)\n"
        "$$R_{th,ges} = R_{th,jc} + R_{th,cs} + R_{th,sa}$$\n"
        "| W\u00e4rmewiderstand | Bedeutung |\n"
        "|-----------------|-----------|\n"
        "| $R_{th,jc}$ | Junction \u2192 Case (im Bauteil, aus Datenblatt) |\n"
        "| $R_{th,cs}$ | Case \u2192 Heatsink (W\u00e4rmeleitpaste/Pad) |\n"
        "| $R_{th,sa}$ | Heatsink \u2192 Ambient (K\u00fchlk\u00f6rper) |\n"
        "<!-- Bild: Thermische Kette Chip-Geh\u00e4use-K\u00fchlk\u00f6rper-Luft -->\n"
        "<!-- Dateiname: thermische_kette_rth.png  \u2192 Google: \"thermal resistance chain junction case heatsink ambient diagram\" -->"))

# ---- Rechner (rechts) ----
# ============================================================
# RECHNER 10.1: Wärmewiderstand und Temperatur
# ============================================================

th_PV   = make_input('P_V [W]      =', 'Verlustleistung')
th_Rth  = make_input('R_th [K/W]   =', 'Wärmewiderstand')
th_dT   = make_input('ΔT [K]       =', 'Temperaturdifferenz')
btn_th  = make_button()
out_th  = widgets.Output()

def calc_th(_=None):
    out_th.clear_output()
    with out_th:
        try:
            PV  = p(th_PV.value)
            Rth = p(th_Rth.value)
            dT  = p(th_dT.value)
            vals = {'P_V': PV, 'R_th': Rth, 'ΔT': dT}
            miss = [k for k, v in vals.items() if v is None]
            if len(miss) != 1:
                print("Genau ein Feld leer lassen!" if len(miss) > 1 else "Ein Feld leer lassen!")
                return
            m = miss[0]
            if   m == 'ΔT':   res = PV * Rth;  print(f"  ΔT   = P_V·R_th = {res:.4g} K")
            elif m == 'R_th': res = dT / PV;   print(f"  R_th = ΔT/P_V   = {res:.4g} K/W")
            else:             res = dT / Rth;  print(f"  P_V  = ΔT/R_th  = {fmt(res, 'W')}")
        except Exception as e:
            print("Fehler:", e)

btn_th.on_click(calc_th)
for fw in [th_PV, th_Rth, th_dT]:
    fw.observe(calc_th, names='value')
_calc_widget = (widgets.VBox([th_PV, th_Rth, th_dT, btn_th, out_th]))

# ---- HBox Layout ----
_left_box = widgets.Box([_theory_out], layout=widgets.Layout(**LEFT_STYLE))
_right_box = widgets.Box([_calc_widget], layout=widgets.Layout(**RIGHT_STYLE))

display(widgets.HBox(
    [_left_box, _right_box],
    layout=widgets.Layout(
        width='100%',
        align_items='flex-start',
        margin='8px 0 28px 0',
    )
))


In [ ]:
display(HTML('<a name="sec-10-2"></a>'))
# ================================================================
# LAYOUT: Theorie (links) | Rechner (rechts)
# ================================================================

# ---- Theorie (links) ----
_theory_out = widgets.Output(layout=THEORY_STYLE)

with _theory_out:
    display(Markdown(
        "---\n"
        "## 10.2 K\u00fchlk\u00f6rper-Dimensionierung\n"
        "### Maximale Sperrschichttemperatur\n"
        "$$T_j = T_{amb} + P_V \\cdot (R_{th,jc} + R_{th,cs} + R_{th,sa})$$\n"
        "Umgestellt nach ben\u00f6tigtem K\u00fchlk\u00f6rperwiderstand:\n"
        "$$R_{th,sa} = \\frac{T_{j,max} - T_{amb}}{P_V} - R_{th,jc} - R_{th,cs}$$\n"
        "**Typische Werte:**\n"
        "| | $R_{th,cs}$ |\n"
        "|--|------------|\n"
        "| Trocken (ohne Paste) | 0.5\u20131.0 K/W |\n"
        "| Mit W\u00e4rmeleitpaste | 0.1\u20130.3 K/W |\n"
        "| Mit W\u00e4rmeleitpad | 0.2\u20130.5 K/W |"))

# ---- Rechner (rechts) ----
# ============================================================
# RECHNER 10.2: Kühlkörper-Dimensionierung
# ============================================================

kk_Tj    = make_input('T_j,max [°C]  =', 'Max. Sperrschicht (Datenblatt)')
kk_Tamb  = make_input('T_amb [°C]    =', 'Umgebungstemperatur z.B. 40')
kk_PV    = make_input('P_V [W]       =', 'Verlustleistung')
kk_Rjc   = make_input('R_th,jc [K/W] =', 'Junction→Case (Datenblatt)')
kk_Rcs   = make_input('R_th,cs [K/W] =', 'Case→Sink (Paste ~0.2)')

kk_paste = widgets.Dropdown(
    options=[('-- Wärmeanbindung --', None),
             ('Trocken',             0.8),
             ('Wärmeleitpaste',      0.2),
             ('Wärmeleitpad',        0.4)],
    description='Anbindung:', style={'description_width': '140px'},
    layout=widgets.Layout(width='340px')
)

btn_kk  = make_button()
out_kk  = widgets.Output()

def set_rcs(change):
    if change['new'] is not None:
        kk_Rcs.value = str(change['new'])
kk_paste.observe(set_rcs, names='value')

def calc_kk(_=None):
    # → Logik in calc.py: kuehlkoerper_dimensionierung()
    out_kk.clear_output()
    with out_kk:
        try:
            r = kuehlkoerper_dimensionierung(Tj=p(kk_Tj.value), Tamb=p(kk_Tamb.value), PV=p(kk_PV.value), Rjc=p(kk_Rjc.value), Rcs=p(kk_Rcs.value))
            _u = {'Rth_ges_max': 'K/W', 'Rsa_max': 'K/W', 'empfehlung': ''}
            _skip = ('berechnet', 'ladetabelle', 'stromtabelle', 'widerspruch')
            for k, v in r.items():
                if k in _skip or v is None: continue
                if isinstance(v, bool):
                    if v: print(f'  {k}')
                    continue
                if isinstance(v, list): continue
                u = _u.get(k, '')
                print(f'  {k} = {fmt(v, u) if isinstance(v,(int,float)) else v}')
        except (ValueError, ZeroDivisionError) as e:
            print('i', e)
        except Exception as e:
            print('Fehler:', e)
btn_kk.on_click(calc_kk)
for fw in [kk_Tj, kk_Tamb, kk_PV, kk_Rjc, kk_Rcs]:
    fw.observe(calc_kk, names='value')
_calc_widget = (widgets.VBox([kk_paste, kk_Tj, kk_Tamb, kk_PV, kk_Rjc, kk_Rcs, btn_kk, out_kk]))

# ---- HBox Layout ----
_left_box = widgets.Box([_theory_out], layout=widgets.Layout(**LEFT_STYLE))
_right_box = widgets.Box([_calc_widget], layout=widgets.Layout(**RIGHT_STYLE))

display(widgets.HBox(
    [_left_box, _right_box],
    layout=widgets.Layout(
        width='100%',
        align_items='flex-start',
        margin='8px 0 28px 0',
    )
))


In [ ]:
display(HTML('<a name="sec-10-3"></a>'))
# ================================================================
# LAYOUT: Theorie (links) | Rechner (rechts)
# ================================================================

# ---- Theorie (links) ----
_theory_out = widgets.Output(layout=THEORY_STYLE)

with _theory_out:
    display(Markdown(
        "---\n"
        "## 10.3 Widerstands\u00e4nderung durch Temperatur (R\u00fcckbezug Kapitel 1.5)\n"
        "Die Verlustleistung im Widerstand erzeugt W\u00e4rme und \u00e4ndert seinen Wert:\n"
        "$$P_V = I^2 \\cdot R \\qquad \\Delta T = P_V \\cdot R_{th}$$\n"
        "$$R_T = R_{20} \\cdot [1 + \\alpha_{20} \\cdot (T - 20\u00b0C)]$$\n"
        "Dies kann zur **thermischen Instabilit\u00e4t** f\u00fchren wenn $\\alpha > 0$ (Metalle):\n"
        "Mehr Strom \u2192 mehr W\u00e4rme \u2192 h\u00f6herer R \u2192 weniger Strom (selbstlimitierend \u2713)\n"
        "Bei NTC ($\\alpha < 0$):\n"
        "Mehr Strom \u2192 mehr W\u00e4rme \u2192 kleinerer R \u2192 noch mehr Strom \u2192 **thermisches Durchgehen** \u26a0\ufe0f"))

# ---- Rechner (rechts) ----
# ============================================================
# RECHNER 10.3: Thermische Stabilität – Betriebstemperatur
# ============================================================

ts_R20  = make_input('R₂₀ [Ω]       =', 'Widerstand bei 20°C')
ts_I    = make_input('I [A]         =', 'Betriebsstrom')
ts_Rth  = make_input('R_th [K/W]    =', 'therm. Anbindung')
ts_Tamb = make_input('T_amb [°C]    =', 'Umgebung z.B. 25')
ts_Tmax = make_input('T_max [°C]    =', 'Max. zul. Temp.')
ts_alph = make_input('α₂₀ [1/K]     =', 'z.B. 0.00393 für Cu')
btn_ts  = make_button()
out_ts  = widgets.Output()

def calc_ts(_=None):
    # → Logik in calc.py: thermische_stabilitaet()
    out_ts.clear_output()
    with out_ts:
        try:
            r = thermische_stabilitaet(R20=p(ts_R20.value), I=p(ts_I.value), Rth=p(ts_Rth.value), Tamb=p(ts_Tamb.value), alpha=p(ts_alph.value), Tmax=p(ts_Tmax.value))
            _u = {'PV': 'W', 'dT': 'K', 'T_betr': '°C', 'R_betr': 'Ω', 'reserve_K': 'K'}
            _skip = ('berechnet', 'ladetabelle', 'stromtabelle', 'widerspruch')
            for k, v in r.items():
                if k in _skip or v is None: continue
                if isinstance(v, bool):
                    if v: print(f'  {k}')
                    continue
                if isinstance(v, list): continue
                u = _u.get(k, '')
                print(f'  {k} = {fmt(v, u) if isinstance(v,(int,float)) else v}')
        except (ValueError, ZeroDivisionError) as e:
            print('i', e)
        except Exception as e:
            print('Fehler:', e)
btn_ts.on_click(calc_ts)
for fw in [ts_R20, ts_I, ts_Rth, ts_Tamb, ts_Tmax, ts_alph]:
    fw.observe(calc_ts, names='value')
_calc_widget = (widgets.VBox([ts_R20, ts_I, ts_Rth, ts_Tamb, ts_Tmax, ts_alph, btn_ts, out_ts]))

# ---- HBox Layout ----
_left_box = widgets.Box([_theory_out], layout=widgets.Layout(**LEFT_STYLE))
_right_box = widgets.Box([_calc_widget], layout=widgets.Layout(**RIGHT_STYLE))

display(widgets.HBox(
    [_left_box, _right_box],
    layout=widgets.Layout(
        width='100%',
        align_items='flex-start',
        margin='8px 0 28px 0',
    )
))


---
# Anhang: Interaktive Diagramme

In [ ]:
%matplotlib inline
print("matplotlib bereit")

In [ ]:
display(HTML('<a name="anhang-a"></a>'))
# ================================================================
# LAYOUT: Theorie (links) | Rechner (rechts)
# ================================================================

# ---- Theorie (links) ----
_theory_out = widgets.Output(layout=THEORY_STYLE)

with _theory_out:
    display(Markdown("---\n## Diagramm A: RC-Lade- und Entladekurve"))

# ---- Rechner (rechts) ----
# ============================================================
# DIAGRAMM A: RC-Lade/Entladekurve – interaktiv
# ============================================================

dA_R    = make_input('R [Ω]  =', 'z.B. 1k')
dA_C    = make_input('C [F]  =', 'z.B. 100u')
dA_U0   = make_input('U₀ [V] =', 'Quellspannung z.B. 5')
dA_mode = widgets.ToggleButtons(
    options=['Laden', 'Entladen'],
    description='Vorgang:', style={'description_width': '100px', 'button_width': '100px'}
)
btn_dA  = make_button('📈 Zeichnen')
out_dA  = widgets.Output()

def plot_rc(_=None):
    out_dA.clear_output()
    with out_dA:
        try:
            R  = p(dA_R.value); C = p(dA_C.value); U0 = p(dA_U0.value)
            if any(x is None for x in [R, C, U0]):
                print("R, C und U₀ eingeben"); return
            tau = R * C
            t   = np.linspace(0, 5 * tau, 500)

            if dA_mode.value == 'Laden':
                U = U0 * (1 - np.exp(-t / tau))
                I = (U0 / R) * np.exp(-t / tau)
                titel = f'RC-Ladekurve  (τ = {fmt(tau, "s")})'
            else:
                U = U0 * np.exp(-t / tau)
                I = (U0 / R) * np.exp(-t / tau)
                titel = f'RC-Entladekurve  (τ = {fmt(tau, "s")})'

            fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
            fig.suptitle(titel, fontsize=13)

            # Spannung
            ax1.plot(t * 1000 if tau < 1 else t, U, 'b-', linewidth=2, label='U_C(t)')
            ax1.axhline(U0, color='gray', linestyle='--', alpha=0.5, label=f'U₀ = {U0} V')
            ax1.axhline(0.632 * U0 if dA_mode.value == 'Laden' else 0.368 * U0,
                        color='orange', linestyle=':', alpha=0.7, label='63.2% / 36.8%')
            tau_ms = tau * 1000 if tau < 1 else tau
            ax1.axvline(tau_ms, color='red', linestyle=':', alpha=0.6, label=f'τ = {fmt(tau, "s")}')
            ax1.set_ylabel('Spannung [V]')
            ax1.legend(fontsize=9)
            ax1.grid(True, alpha=0.3)
            ax1.set_ylim(-U0 * 0.05, U0 * 1.15)

            # Strom
            t_plot = t * 1000 if tau < 1 else t
            ax2.plot(t_plot, I * 1000 if max(I) < 0.1 else I,
                     'r-', linewidth=2, label='I(t)')
            ax2.axvline(tau_ms, color='red', linestyle=':', alpha=0.6)
            I_unit = 'mA' if max(I) < 0.1 else 'A'
            ax2.set_ylabel(f'Strom [{I_unit}]')
            t_unit = 'ms' if tau < 1 else 's'
            ax2.set_xlabel(f'Zeit [{t_unit}]')
            ax2.legend(fontsize=9)
            ax2.grid(True, alpha=0.3)

            # τ Markierungen
            for n in range(1, 6):
                ax1.annotate(f'{n}τ', xy=(n * tau_ms, 0),
                             xytext=(n * tau_ms, -U0 * 0.08),
                             ha='center', fontsize=8, color='red')

            plt.tight_layout()
            plt.show()
            print(f"  τ = {fmt(tau, 's')}  |  5τ = {fmt(5*tau, 's')}  |  I₀ = {fmt(U0/R, 'A')}")
        except Exception as e:
            print("Fehler:", e)

btn_dA.on_click(plot_rc)
_calc_widget = (widgets.VBox([dA_mode, dA_R, dA_C, dA_U0, btn_dA, out_dA]))

# ---- HBox Layout ----
_left_box = widgets.Box([_theory_out], layout=widgets.Layout(**LEFT_STYLE))
_right_box = widgets.Box([_calc_widget], layout=widgets.Layout(**RIGHT_STYLE))

display(widgets.HBox(
    [_left_box, _right_box],
    layout=widgets.Layout(
        width='100%',
        align_items='flex-start',
        margin='8px 0 28px 0',
    )
))


In [ ]:
display(HTML('<a name="anhang-b"></a>'))
# ================================================================
# LAYOUT: Theorie (links) | Rechner (rechts)
# ================================================================

# ---- Theorie (links) ----
_theory_out = widgets.Output(layout=THEORY_STYLE)

with _theory_out:
    display(Markdown("---\n## Diagramm B: Filterkurve (Bode-Diagramm)"))

# ---- Rechner (rechts) ----
# ============================================================
# DIAGRAMM B: Bode-Diagramm – KORRIGIERT
# ============================================================

dB_typ  = widgets.ToggleButtons(
    options=['RC-Tiefpass', 'RC-Hochpass', 'Sallen-Key TP 2.Ord'],
    description='Filtertyp:', style={'description_width': '120px', 'button_width': '160px'}
)
dB_R    = make_input('R [Ω]   =', 'z.B. 10k')
dB_C    = make_input('C [F]   =', 'z.B. 10n')
dB_Q    = make_input('Q       =', 'nur Sallen-Key, z.B. 0.707')
btn_dB2 = make_button('📈 Zeichnen')
out_dB2 = widgets.Output()

def plot_bode(_=None):
    out_dB2.clear_output()
    with out_dB2:
        try:
            R = p(dB_R.value); C = p(dB_C.value)
            Q = p(dB_Q.value) if dB_Q.value.strip() else 0.7071
            if any(x is None for x in [R, C]):
                print("R und C eingeben"); return

            fg    = 1 / (2 * math.pi * R * C)
            f     = np.logspace(np.log10(fg / 100), np.log10(fg * 100), 1000)
            ratio = f / fg
            typ   = dB_typ.value

            if typ == 'RC-Tiefpass':
                H     = 1 / np.sqrt(1 + ratio**2)
                phi   = -np.degrees(np.arctan(ratio))
                label = f'RC-Tiefpass  fg = {fmt(fg, "Hz")}'
                phi_fg = -45.0
                phi_lim = (-100, 10)

            elif typ == 'RC-Hochpass':
                H     = ratio / np.sqrt(1 + ratio**2)
                phi   = np.degrees(np.arctan(1 / ratio))
                label = f'RC-Hochpass  fg = {fmt(fg, "Hz")}'
                phi_fg = +45.0
                phi_lim = (-10, 100)

            else:  # Sallen-Key 2. Ordnung
                denom = np.sqrt((1 - ratio**2)**2 + (ratio / Q)**2)
                H     = 1 / denom
                # Korrekte Phase 2. Ordnung: geht von 0° bis −180°
                phi   = -np.degrees(np.arctan2(ratio / Q, 1 - ratio**2))
                label = f'Sallen-Key TP 2.Ord  fg={fmt(fg,"Hz")}  Q={Q:.3g}'
                phi_fg = -90.0   # bei fg ist Phase −90° (nicht −45°!)
                phi_lim = (-200, 10)

            dB_arr = 20 * np.log10(np.maximum(H, 1e-10))

            fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
            fig.suptitle(f'Bode-Diagramm – {label}', fontsize=12)

            # --- Amplitudengang ---
            ax1.semilogx(f, dB_arr, 'b-', linewidth=2, label='|H(f)|')
            ax1.axvline(fg, color='red', linestyle='--', alpha=0.7,
                        label=f'fg = {fmt(fg, "Hz")}')
            ax1.axhline(-3, color='orange', linestyle=':', alpha=0.7,
                        label='−3 dB')

            # Asymptoten
            if typ == 'RC-Tiefpass':
                ax1.semilogx([fg/100, fg], [0, 0], 'g--', alpha=0.4,
                             linewidth=1, label='Asymptote')
                ax1.semilogx([fg, fg*100], [0, -40], 'g--', alpha=0.4, linewidth=1)
            elif typ == 'RC-Hochpass':
                ax1.semilogx([fg/100, fg], [-40, 0], 'g--', alpha=0.4,
                             linewidth=1, label='Asymptote')
                ax1.semilogx([fg, fg*100], [0, 0], 'g--', alpha=0.4, linewidth=1)
            else:
                # Sallen-Key: −40 dB/Dekade Asymptote
                ax1.semilogx([fg/100, fg], [0, 0], 'g--', alpha=0.4,
                             linewidth=1, label='Asymptote (−40dB/Dek.)')
                ax1.semilogx([fg, fg*100], [0, -80], 'g--', alpha=0.4, linewidth=1)

            ax1.set_ylabel('Verstärkung [dB]')
            ax1.set_ylim(max(np.min(dB_arr) - 5, -90), 10)
            ax1.legend(fontsize=9)
            ax1.grid(True, which='both', alpha=0.3)

            # --- Phasengang ---
            ax2.semilogx(f, phi, 'r-', linewidth=2, label='φ(f)')
            ax2.axvline(fg, color='red', linestyle='--', alpha=0.7)
            ax2.axhline(phi_fg, color='orange', linestyle=':', alpha=0.8,
                        label=f'{phi_fg:.0f}° bei fg')

            ax2.set_ylabel('Phase [°]')
            ax2.set_xlabel('Frequenz [Hz]')
            ax2.set_ylim(phi_lim)
            ax2.yaxis.set_major_locator(ticker.MultipleLocator(45))
            ax2.legend(fontsize=9)
            ax2.grid(True, which='both', alpha=0.3)

            plt.tight_layout()
            plt.show()

            print(f"  fg   = {fmt(fg, 'Hz')}  |  ωg = {fmt(2*math.pi*fg, 'rad/s')}")
            if typ == 'Sallen-Key TP 2.Ord':
                print(f"  Q    = {Q:.4g}")
                print(f"  Phase bei fg: −90°  (2. Ordnung: 0° → −180°)")
                print(f"  Flankensteilheit: −40 dB/Dekade")
            else:
                print(f"  Phase bei fg: {phi_fg:+.0f}°  (1. Ordnung: ±90° gesamt)")
                print(f"  Flankensteilheit: ±20 dB/Dekade")
        except Exception as e:
            print("Fehler:", e)

btn_dB2.on_click(plot_bode)
dB_typ.observe(plot_bode, names='value')
_calc_widget = (widgets.VBox([dB_typ, dB_R, dB_C, dB_Q, btn_dB2, out_dB2]))

# ---- HBox Layout ----
_left_box = widgets.Box([_theory_out], layout=widgets.Layout(**LEFT_STYLE))
_right_box = widgets.Box([_calc_widget], layout=widgets.Layout(**RIGHT_STYLE))

display(widgets.HBox(
    [_left_box, _right_box],
    layout=widgets.Layout(
        width='100%',
        align_items='flex-start',
        margin='8px 0 28px 0',
    )
))


In [ ]:
display(HTML('<a name="anhang-c"></a>'))
# ================================================================
# LAYOUT: Theorie (links) | Rechner (rechts)
# ================================================================

# ---- Theorie (links) ----
_theory_out = widgets.Output(layout=THEORY_STYLE)

with _theory_out:
    display(Markdown("---\n## Diagramm C: RLC-Schwingkreis Frequenzgang"))

# ---- Rechner (rechts) ----
# ============================================================
# DIAGRAMM C: RLC Schwingkreis – KORRIGIERT
# ============================================================

dC_R   = make_input('R [Ω]  =', 'z.B. 50')
dC_L   = make_input('L [H]  =', 'z.B. 1m')
dC_C   = make_input('C [F]  =', 'z.B. 10u')
dC_typ = widgets.ToggleButtons(
    options=['Reihenschwingkreis', 'Parallelschwingkreis'],
    description='Typ:', style={'description_width': '100px', 'button_width': '180px'}
)
btn_dC = make_button('📈 Zeichnen')
out_dC = widgets.Output()

def plot_rlc(_=None):
    out_dC.clear_output()
    with out_dC:
        try:
            R = p(dC_R.value); L = p(dC_L.value); C = p(dC_C.value)
            if any(x is None for x in [R, L, C]):
                print("R, L und C eingeben"); return

            f0  = 1 / (2 * math.pi * math.sqrt(L * C))
            Q   = (1/R) * math.sqrt(L/C)
            df  = f0 / Q
            fu  = f0 - df/2
            fo  = f0 + df/2

            # Frequenzbereich: weit genug um beide Flanken zu zeigen
            # Mindestens Faktor 50 unter fu und über fo
            f_min = max(fu / 50, f0 / 200)
            f_max = fo * 50
            f     = np.logspace(np.log10(f_min), np.log10(f_max), 2000)
            w     = 2 * math.pi * f
            XL    = w * L
            XC    = 1 / (w * C)

            if dC_typ.value == 'Reihenschwingkreis':
                Z    = np.sqrt(R**2 + (XL - XC)**2)
                Z_dB = 20 * np.log10(Z)
                lbl  = f'Reihenschwingkreis  f₀={fmt(f0,"Hz")}  Q={Q:.3g}'
                # Reihe: Minimum bei f0
                Z_ref = 20 * np.log10(R)
                y_margin = max(np.max(Z_dB) - Z_ref + 5, 20)
                y_lim = (Z_ref - 5, Z_ref + y_margin)
            else:
                # Parallel: Z_parallel korrekt berechnet
                Z_L   = np.sqrt(R**2 + XL**2)       # Impedanz Spulenzweig
                Z_C   = XC                            # Impedanz Kondensator
                # Parallelschaltung beider Zweige
                Z_re  = (R * XC**2 + R * XL**2) / (R**2 + (XL - XC)**2 + 1e-30)
                Z_im  = (XL * XC**2 - XL**2 * XC - R**2 * XC) / (R**2 + (XL - XC)**2 + 1e-30)
                Z     = np.sqrt(Z_re**2 + Z_im**2)
                Z_dB  = 20 * np.log10(np.maximum(Z, 1e-6))
                lbl   = f'Parallelschwingkreis  f₀={fmt(f0,"Hz")}  Q={Q:.3g}'
                # Maximum bei f0 bestimmen – Skalierung darauf ausrichten
                Z_max_dB = np.max(Z_dB)
                Z_min_dB = np.min(Z_dB)
                y_lim = (Z_min_dB - 5, Z_max_dB + 10)

            fig, ax = plt.subplots(figsize=(11, 5))
            ax.semilogx(f, Z_dB, 'b-', linewidth=2, label='Z(f)')

            # f0 Linie
            ax.axvline(f0, color='red', linestyle='--', alpha=0.8,
                       label=f'f₀ = {fmt(f0, "Hz")}')

            # f_u und f_o nur wenn positiv
            if fu > 0:
                ax.axvline(fu, color='orange', linestyle=':', alpha=0.7,
                           label=f'f_u = {fmt(fu, "Hz")}')
            else:
                print(f"  f_u = {fu:.3g} Hz (negativ – Q zu niedrig für symmetrische Bandbreite)")

            ax.axvline(fo, color='orange', linestyle=':', alpha=0.7,
                       label=f'f_o = {fmt(fo, "Hz")}')

            ax.set_title(f'Impedanzkurve – {lbl}', fontsize=12)
            ax.set_xlabel('Frequenz [Hz]')
            ax.set_ylabel('Z [dBΩ]')
            ax.set_ylim(y_lim)
            ax.legend(fontsize=9)
            ax.grid(True, which='both', alpha=0.3)
            plt.tight_layout()
            plt.show()

            print(f"  f₀  = {fmt(f0, 'Hz')}")
            print(f"  Q   = {Q:.4g}")
            print(f"  Δf  = {fmt(df, 'Hz')}")
            if fu > 0:
                print(f"  f_u = {fmt(fu, 'Hz')}")
            else:
                print(f"  f_u = {fu:.3g} Hz  negativ (Q < 0.5 → Bandbreite > 2·f₀)")
            print(f"  f_o = {fmt(fo, 'Hz')}")
        except Exception as e:
            print("Fehler:", e)

btn_dC.on_click(plot_rlc)
dC_typ.observe(plot_rlc, names='value')
_calc_widget = (widgets.VBox([dC_typ, dC_R, dC_L, dC_C, btn_dC, out_dC]))

# ---- HBox Layout ----
_left_box = widgets.Box([_theory_out], layout=widgets.Layout(**LEFT_STYLE))
_right_box = widgets.Box([_calc_widget], layout=widgets.Layout(**RIGHT_STYLE))

display(widgets.HBox(
    [_left_box, _right_box],
    layout=widgets.Layout(
        width='100%',
        align_items='flex-start',
        margin='8px 0 28px 0',
    )
))


In [ ]:
display(HTML('<a name="anhang-d"></a>'))
# ================================================================
# LAYOUT: Theorie (links) | Rechner (rechts)
# ================================================================

# ---- Theorie (links) ----
_theory_out = widgets.Output(layout=THEORY_STYLE)

with _theory_out:
    display(Markdown("---\n## Diagramm D: RL-Einschaltstrom"))

# ---- Rechner (rechts) ----
# ============================================================
# DIAGRAMM D: RL Einschalt-/Ausschaltkurve
# ============================================================

dD_R    = make_input('R [Ω]  =', 'z.B. 100')
dD_L    = make_input('L [H]  =', 'z.B. 10m')
dD_U0   = make_input('U₀ [V] =', 'z.B. 12')
dD_mode = widgets.ToggleButtons(
    options=['Einschalten', 'Ausschalten'],
    description='Vorgang:', style={'description_width': '100px', 'button_width': '120px'}
)
btn_dD  = make_button('📈 Zeichnen')
out_dD  = widgets.Output()

def plot_rl(_=None):
    out_dD.clear_output()
    with out_dD:
        try:
            R = p(dD_R.value); L = p(dD_L.value); U0 = p(dD_U0.value)
            if any(x is None for x in [R, L, U0]):
                print("R, L und U₀ eingeben"); return

            tau   = L / R
            I_max = U0 / R
            t     = np.linspace(0, 5 * tau, 500)

            if dD_mode.value == 'Einschalten':
                I  = I_max * (1 - np.exp(-t / tau))
                UL = U0 * np.exp(-t / tau)
                UR = I * R
                titel = f'RL-Einschalten  (τ = {fmt(tau, "s")})'
            else:
                I  = I_max * np.exp(-t / tau)
                UL = -U0 * np.exp(-t / tau)
                UR = I * R
                titel = f'RL-Ausschalten  (τ = {fmt(tau, "s")})'

            t_ms = t * 1000 if tau < 0.1 else t
            t_unit = 'ms' if tau < 0.1 else 's'

            fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
            fig.suptitle(titel, fontsize=13)

            ax1.plot(t_ms, I * 1000 if I_max < 0.1 else I, 'b-', linewidth=2, label='I(t)')
            ax1.axhline(I_max * 1000 if I_max < 0.1 else I_max,
                        color='gray', linestyle='--', alpha=0.5, label=f'I_max = {fmt(I_max, "A")}')
            ax1.axvline(tau * 1000 if tau < 0.1 else tau,
                        color='red', linestyle=':', alpha=0.7, label=f'τ = {fmt(tau, "s")}')
            I_unit = 'mA' if I_max < 0.1 else 'A'
            ax1.set_ylabel(f'Strom [{I_unit}]')
            ax1.legend(fontsize=9)
            ax1.grid(True, alpha=0.3)

            ax2.plot(t_ms, UL, 'r-', linewidth=2, label='U_L(t)')
            ax2.plot(t_ms, UR, 'g-', linewidth=1.5, alpha=0.7, label='U_R(t)')
            ax2.axhline(0, color='black', linewidth=0.5)
            ax2.axvline(tau * 1000 if tau < 0.1 else tau,
                        color='red', linestyle=':', alpha=0.7)
            ax2.set_ylabel('Spannung [V]')
            ax2.set_xlabel(f'Zeit [{t_unit}]')
            ax2.legend(fontsize=9)
            ax2.grid(True, alpha=0.3)

            for n in range(1, 6):
                t_mark = n * tau * 1000 if tau < 0.1 else n * tau
                ax1.annotate(f'{n}τ', xy=(t_mark, 0),
                             xytext=(t_mark, -I_max * 0.08 * (1000 if I_max < 0.1 else 1)),
                             ha='center', fontsize=8, color='red')

            plt.tight_layout()
            plt.show()
            print(f"  τ = {fmt(tau, 's')}  |  I_max = {fmt(I_max, 'A')}  |  5τ = {fmt(5*tau, 's')}")
        except Exception as e:
            print("Fehler:", e)

btn_dD.on_click(plot_rl)
_calc_widget = (widgets.VBox([dD_mode, dD_R, dD_L, dD_U0, btn_dD, out_dD]))

# ---- HBox Layout ----
_left_box = widgets.Box([_theory_out], layout=widgets.Layout(**LEFT_STYLE))
_right_box = widgets.Box([_calc_widget], layout=widgets.Layout(**RIGHT_STYLE))

display(widgets.HBox(
    [_left_box, _right_box],
    layout=widgets.Layout(
        width='100%',
        align_items='flex-start',
        margin='8px 0 28px 0',
    )
))


In [ ]:
display(HTML('<a name="anhang-e"></a>'))
# ================================================================
# LAYOUT: Theorie (links) | Rechner (rechts)
# ================================================================

# ---- Theorie (links) ----
_theory_out = widgets.Output(layout=THEORY_STYLE)

with _theory_out:
    display(Markdown("---\n## Diagramm E: Gleichrichter Wellenformen"))

# ---- Rechner (rechts) ----
# ============================================================
# DIAGRAMM E: Gleichrichter Wellenformen – KORRIGIERT
# ============================================================

dE_Ueff = make_input('U_eff [V]  =', 'z.B. 12')
dE_UF   = make_input('U_F [V]    =', 'Flussspannung ~0.7')
dE_C    = make_input('C [F]      =', 'Siebkond. z.B. 1000u')
dE_I    = make_input('I_Last [A] =', 'z.B. 0.5')
dE_typ  = widgets.ToggleButtons(
    options=['Einweg M1', 'Brücke B2'],
    description='Typ:', style={'description_width': '100px', 'button_width': '120px'}
)
btn_dE  = make_button('📈 Zeichnen')
out_dE  = widgets.Output()

def plot_gleichrichter(_=None):
    out_dE.clear_output()
    with out_dE:
        try:
            Ueff = p(dE_Ueff.value)
            UF   = p(dE_UF.value) if dE_UF.value.strip() else 0.7
            C    = p(dE_C.value)
            I    = p(dE_I.value)
            if Ueff is None:
                print("U_eff eingeben"); return

            Up   = Ueff * math.sqrt(2)
            f    = 50
            t    = np.linspace(0, 2/f, 2000)
            U_ac = Up * np.sin(2 * math.pi * f * t)

            if dE_typ.value == 'Einweg M1':
                U_rect = np.where(U_ac > UF, U_ac - UF, 0.0)
                fBr    = f
                UF_tot = UF
                titel  = f'Einweggleichrichter M1  (Û = {fmt(Up, "V")})'
            else:
                U_rect = np.abs(U_ac) - 2 * UF
                U_rect = np.where(U_rect > 0, U_rect, 0.0)
                fBr    = 2 * f
                UF_tot = 2 * UF
                titel  = f'Brückengleichrichter B2  (Û = {fmt(Up, "V")})'

            U_avg = np.mean(U_rect)

            fig, ax = plt.subplots(figsize=(10, 5))
            ax.plot(t * 1000, U_ac, 'b-', alpha=0.35, linewidth=1.5,
                    label='U_AC (Eingang)')
            ax.plot(t * 1000, U_rect, 'r-', linewidth=2,
                    label='U_rect (nach Gleichrichter)')

            warnings = []

            if C and I:
                UBr  = I / (fBr * C)
                UDC  = Up - UF_tot - UBr / 2

                if UBr >= Up - UF_tot:
                    # Welligkeit grösser als gesamte Spannung – C viel zu klein
                    warnings.append(
                        f"C zu klein / I zu gross: Welligkeit ({fmt(UBr,'V')}) ≥ "
                        f"Spitzenwert ({fmt(Up-UF_tot,'V')}) → Siebung unwirksam!"
                    )
                    # DC-Linie trotzdem zeigen, aber auf 0 begrenzen
                    UDC_plot  = max(UDC, 0)
                    UBr_plot  = min(UBr, Up - UF_tot)
                    U_low     = max(UDC - UBr / 2, 0)
                else:
                    UDC_plot = UDC
                    UBr_plot = UBr
                    U_low    = UDC - UBr / 2

                    if U_low < 0:
                        warnings.append(
                            f"Untere Welligkeit ({fmt(U_low,'V')}) unter 0 V – "
                            f"C vergrössern oder I reduzieren!"
                        )
                        U_low = 0  # Auf 0 begrenzen für Darstellung

                ax.axhline(UDC_plot, color='green', linestyle='--', linewidth=1.8,
                           label=f'Ū_DC ≈ {UDC_plot:.3g} V')
                ax.axhline(U_low, color='purple', linestyle=':',
                           linewidth=1.2, label=f'U_min ≈ {U_low:.3g} V')
                ax.fill_between(t * 1000,
                                U_low,
                                min(UDC_plot + UBr_plot / 2, Up - UF_tot),
                                alpha=0.15, color='green', label='Welligkeit')

                w_pct = (UBr / UDC * 100) if UDC > 0 else float('inf')
                print(f"  Û           = {fmt(Up, 'V')}")
                print(f"  Ū_DC        = {fmt(UDC, 'V')}")
                print(f"  U_Br        = {fmt(UBr, 'V')}")
                if math.isfinite(w_pct):
                    print(f"  Welligkeit  = {w_pct:.3g} %")
                else:
                    print(f"  Welligkeit  = nicht definiert (U_DC ≤ 0)")
            else:
                U_avg_th = Up / math.pi - UF_tot/2 if dE_typ.value == 'Einweg M1' \
                           else 2*Up/math.pi - UF_tot
                ax.axhline(U_avg_th, color='orange', linestyle='--', alpha=0.8,
                           label=f'Ū (ohne Siebung) = {U_avg_th:.3g} V')
                print(f"  Û      = {fmt(Up, 'V')}")
                print(f"  Ū      = {fmt(U_avg_th, 'V')}")

            ax.axhline(0, color='black', linewidth=0.8, alpha=0.5)
            ax.set_title(titel, fontsize=12)
            ax.set_xlabel('Zeit [ms]')
            ax.set_ylabel('Spannung [V]')
            ax.legend(fontsize=9, loc='upper right')
            ax.grid(True, alpha=0.3)

            # Y-Achse immer vernünftig skalieren – nie unter −Up*0.15
            ax.set_ylim(-Up * 0.15, Up * 1.2)

            plt.tight_layout()
            plt.show()

            for w in warnings:
                print(w)

        except Exception as e:
            print("Fehler:", e)

btn_dE.on_click(plot_gleichrichter)
dE_typ.observe(plot_gleichrichter, names='value')
_calc_widget = (widgets.VBox([dE_typ, dE_Ueff, dE_UF, dE_C, dE_I, btn_dE, out_dE]))

# ---- HBox Layout ----
_left_box = widgets.Box([_theory_out], layout=widgets.Layout(**LEFT_STYLE))
_right_box = widgets.Box([_calc_widget], layout=widgets.Layout(**RIGHT_STYLE))

display(widgets.HBox(
    [_left_box, _right_box],
    layout=widgets.Layout(
        width='100%',
        align_items='flex-start',
        margin='8px 0 28px 0',
    )
))


In [ ]:
display(HTML('<a name="anhang-f"></a>'))
# ================================================================
# LAYOUT: Theorie (links) | Rechner (rechts)
# ================================================================

# ---- Theorie (links) ----
_theory_out = widgets.Output(layout=THEORY_STYLE)

with _theory_out:
    display(Markdown("---\n## Diagramm F: Zeigerdiagramm (Phasordiagramm)"))

# ---- Rechner (rechts) ----
# ============================================================
# DIAGRAMM F: Zeigerdiagramm für RL, RC, RLC-Reihenschaltung
# ============================================================

zd_typ  = make_toggle('Schaltung:', ['RL-Reihe', 'RC-Reihe', 'RLC-Reihe'], '110px')
zd_R    = make_input('R [Ω]  =', 'z.B. 100')
zd_XL   = make_input('X_L [Ω]=', 'induktiver Widerstand')
zd_XC   = make_input('X_C [Ω]=', 'kapazitiver Widerstand')
zd_U    = make_input('U_ges [V]=', 'optional, z.B. 230')
btn_zd  = make_button('📈 Zeichnen')
out_zd  = widgets.Output()

def update_zd_fields(_=None):
    typ = zd_typ.value
    if typ == 'RL-Reihe':
        zd_XC.layout.visibility = 'hidden'
    elif typ == 'RC-Reihe':
        zd_XL.layout.visibility = 'hidden'
        zd_XC.layout.visibility = 'visible'
    else:
        zd_XL.layout.visibility = 'visible'
        zd_XC.layout.visibility = 'visible'
    plot_zd()

def plot_zd(_=None):
    out_zd.clear_output()
    with out_zd:
        try:
            R  = p(zd_R.value)
            XL = p(zd_XL.value)
            XC = p(zd_XC.value)
            U0 = p(zd_U.value)
            typ = zd_typ.value

            if R is None: print("R eingeben"); return

            if typ == 'RL-Reihe':
                if XL is None: print("X_L eingeben"); return
                XC = 0
            elif typ == 'RC-Reihe':
                if XC is None: print("X_C eingeben"); return
                XL = 0
            else:
                if XL is None or XC is None: print("X_L und X_C eingeben"); return

            X_net = XL - XC
            Z     = math.sqrt(R**2 + X_net**2)
            phi   = math.degrees(math.atan2(X_net, R))

            scale = U0 if U0 else Z  # Use U as scale if given, else Z

            # Spannungen
            UR = R   / Z * scale
            UX = X_net / Z * scale
            UL = XL  / Z * scale if XL else 0
            UC = XC  / Z * scale if XC else 0

            fig, ax = plt.subplots(figsize=(7, 7))
            ax.set_aspect('equal')

            # Achsen
            ax.axhline(0, color='gray', linewidth=0.8, alpha=0.5)
            ax.axvline(0, color='gray', linewidth=0.8, alpha=0.5)

            # Strom-Referenzzeiger (horizontal)
            I_len = scale * 0.15
            ax.annotate('', xy=(I_len, 0), xytext=(0, 0),
                        arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))
            ax.text(I_len * 1.05, 0.02 * scale, 'I (Referenz)', color='gray', fontsize=9)

            # UR (ohmsch, in Phase mit I)
            ax.annotate('', xy=(UR, 0), xytext=(0, 0),
                        arrowprops=dict(arrowstyle='->', color='blue', lw=2.5))
            ax.text(UR/2, -0.06*scale, f'U_R = {fmt(UR, "V")}', color='blue',
                    ha='center', fontsize=10)

            # UL (induktiv, +90°)
            if UL > 0:
                ax.annotate('', xy=(UR, UL), xytext=(UR, 0),
                            arrowprops=dict(arrowstyle='->', color='red', lw=2.5))
                ax.text(UR + 0.04*scale, UL/2, f'U_L = {fmt(UL,"V")}',
                        color='red', fontsize=10)

            # UC (kapazitiv, −90°)
            if UC > 0:
                ax.annotate('', xy=(UR, -UC), xytext=(UR, 0),
                            arrowprops=dict(arrowstyle='->', color='green', lw=2.5))
                ax.text(UR + 0.04*scale, -UC/2, f'U_C = {fmt(UC,"V")}',
                        color='green', fontsize=10)

            # Gesamt U_ges
            Uy = UL - UC
            ax.annotate('', xy=(UR, Uy), xytext=(0, 0),
                        arrowprops=dict(arrowstyle='->', color='black', lw=3))
            ax.text(UR/2 + 0.03*scale, Uy/2, f'U_ges = {fmt(scale,"V")}',
                    color='black', fontsize=11, fontweight='bold')

            # Phasenwinkel Bogen
            theta = np.linspace(0, math.radians(phi), 50)
            arc_r = scale * 0.18
            ax.plot(arc_r * np.cos(theta), arc_r * np.sin(theta), 'k--', lw=1)
            ax.text(arc_r * 1.1 * math.cos(math.radians(phi/2)),
                    arc_r * 1.1 * math.sin(math.radians(phi/2)),
                    f'φ={phi:.1f}°', fontsize=10)

            margin = scale * 0.25
            ax.set_xlim(-margin, scale + margin)
            ax.set_ylim(-scale * 0.6, scale * 0.6)
            ax.set_xlabel('Wirkanteil [V]')
            ax.set_ylabel('Blindanteil [V]')
            ax.set_title(f'Zeigerdiagramm {typ}   Z = {fmt(Z,"Ω")}   φ = {phi:.2f}°', fontsize=12)
            ax.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.show()

            print(f"  R    = {fmt(R,'Ω')}")
            if XL: print(f"  X_L  = {fmt(XL,'Ω')}")
            if XC: print(f"  X_C  = {fmt(XC,'Ω')}")
            print(f"  Z    = {fmt(Z,'Ω')}")
            print(f"  φ    = {phi:.3g}°  ({'induktiv' if phi > 0 else 'kapazitiv' if phi < 0 else 'ohmsch'})")
        except Exception as e:
            print("Fehler:", e)

zd_typ.observe(update_zd_fields, names='value')
btn_zd.on_click(plot_zd)
for fw in [zd_R, zd_XL, zd_XC, zd_U]:
    fw.observe(plot_zd, names='value')
update_zd_fields()
_calc_widget = (widgets.VBox([zd_typ, zd_R, zd_XL, zd_XC, zd_U, btn_zd, out_zd]))

# ---- HBox Layout ----
_left_box = widgets.Box([_theory_out], layout=widgets.Layout(**LEFT_STYLE))
_right_box = widgets.Box([_calc_widget], layout=widgets.Layout(**RIGHT_STYLE))

display(widgets.HBox(
    [_left_box, _right_box],
    layout=widgets.Layout(
        width='100%',
        align_items='flex-start',
        margin='8px 0 28px 0',
    )
))


In [ ]:
display(HTML('<a name="anhang-g"></a>'))
# ================================================================
# LAYOUT: Theorie (links) | Rechner (rechts)
# ================================================================

# ---- Theorie (links) ----
_theory_out = widgets.Output(layout=THEORY_STYLE)

with _theory_out:
    display(Markdown("---\n## Diagramm G: NTC / PTC Kennlinie"))

# ---- Rechner (rechts) ----
# ============================================================
# DIAGRAMM G: NTC/PTC Widerstand-Temperatur-Kennlinie
# ============================================================

ntc_typ  = make_toggle('Typ:', ['NTC (B-Parameter)', 'PTC (linear, Metall)'], '200px')
ntc_R25  = make_input('R₂₅ [Ω]  =', 'Widerstand bei 25°C')
ntc_B    = make_input('B [K]     =', 'NTC B-Parameter z.B. 3950')
ntc_R20  = make_input('R₂₀ [Ω]  =', 'PTC: Widerstand bei 20°C')
ntc_alph = make_input('α₂₀ [1/K] =', 'PTC: Temp.koeff. z.B. 0.00393')
ntc_T    = make_input('T [°C]    =', 'optional: R bei T berechnen')
btn_ntc  = make_button('📈 Zeichnen')
out_ntc  = widgets.Output()
box_ntc  = widgets.VBox([ntc_R25, ntc_B])
box_ptc  = widgets.VBox([ntc_R20, ntc_alph])
cont_ntc = widgets.VBox([])

def update_ntc(_=None):
    if ntc_typ.value == 'NTC (B-Parameter)':
        cont_ntc.children = [box_ntc, ntc_T, btn_ntc, out_ntc]
    else:
        cont_ntc.children = [box_ptc, ntc_T, btn_ntc, out_ntc]
    plot_ntc()

def plot_ntc(_=None):
    out_ntc.clear_output()
    with out_ntc:
        try:
            T_arr = np.linspace(-40, 150, 500)
            T_K   = T_arr + 273.15
            Ttest = p(ntc_T.value)

            if ntc_typ.value == 'NTC (B-Parameter)':
                R25 = p(ntc_R25.value)
                B   = p(ntc_B.value)
                if R25 is None or B is None:
                    print("R₂₅ und B eingeben"); return
                T0  = 298.15  # 25°C in K
                R_arr = R25 * np.exp(B * (1/T_K - 1/T0))
                title = f'NTC Kennlinie  R₂₅={fmt(R25,"Ω")}  B={B:.0f} K'
                color = 'red'
                if Ttest is not None:
                    R_T = R25 * math.exp(B * (1/(Ttest+273.15) - 1/T0))
                    print(f"  Bei T = {Ttest} °C:")
                    print(f"  R_NTC = {fmt(R_T, 'Ω')}")
            else:
                R20  = p(ntc_R20.value)
                alph = p(ntc_alph.value)
                if R20 is None or alph is None:
                    print("R₂₀ und α₂₀ eingeben"); return
                R_arr = R20 * (1 + alph * (T_arr - 20))
                title = f'PTC Kennlinie  R₂₀={fmt(R20,"Ω")}  α={alph:.5g} 1/K'
                color = 'blue'
                if Ttest is not None:
                    R_T = R20 * (1 + alph * (Ttest - 20))
                    print(f"  Bei T = {Ttest} °C:")
                    print(f"  R_PTC = {fmt(R_T, 'Ω')}")

            fig, ax = plt.subplots(figsize=(9, 5))
            ax.semilogy(T_arr, R_arr, color=color, linewidth=2)
            ax.axvline(25, color='gray', linestyle='--', alpha=0.5, label='25°C')
            if Ttest is not None:
                ax.axvline(Ttest, color='orange', linestyle=':', label=f'{Ttest}°C')
            ax.set_xlabel('Temperatur [°C]')
            ax.set_ylabel('Widerstand [Ω] (log)')
            ax.set_title(title)
            ax.legend()
            ax.grid(True, which='both', alpha=0.3)
            plt.tight_layout()
            plt.show()
        except Exception as e:
            print("Fehler:", e)

ntc_typ.observe(update_ntc, names='value')
btn_ntc.on_click(plot_ntc)
for fw in [ntc_R25, ntc_B, ntc_R20, ntc_alph, ntc_T]:
    fw.observe(plot_ntc, names='value')
update_ntc()
_calc_widget = (widgets.VBox([ntc_typ, cont_ntc]))

# ---- HBox Layout ----
_left_box = widgets.Box([_theory_out], layout=widgets.Layout(**LEFT_STYLE))
_right_box = widgets.Box([_calc_widget], layout=widgets.Layout(**RIGHT_STYLE))

display(widgets.HBox(
    [_left_box, _right_box],
    layout=widgets.Layout(
        width='100%',
        align_items='flex-start',
        margin='8px 0 28px 0',
    )
))
